# Guía PC2 · Machine learning paso a paso en Google Colab

Esta guía está pensada para una prueba: carga **tu archivo**, cambia los parámetros señalados y ejecuta las celdas en orden. No descarga bases demostrativas ni elige limpiezas por ti.

**Recorrido:** 1. Entorno → 2. Carga y comprensión → **3. Calidad y limpieza** → **4. EDA de la base limpia** → 5. Preparación y división → 6. Modelos manuales → 7. AutoML y red → 8. Elección y prueba → 9. Interpretación → 10. Base nueva.

Puedes predecir una categoría (clasificación), una cantidad (regresión) o valores de una serie temporal. Los bloques que no corresponden al problema se omiten. Las opciones costosas empiezan desactivadas.

Cada transformación conserva su entrada: **df_original → df_reglas → df_tipos_corregidos → df_sin_columnas → df_sin_filas → df_sin_duplicados → df_limpio**. Nunca necesitas ejecutar una asignación que transforme una variable usando esa misma variable. Los bucles solo acumulan resultados locales en listas; no modifican las tablas de entrada.

Las decisiones del EDA se toman sobre la base disponible antes de dividir, como en las referencias. La prueba posterior es una evaluación interna; una segunda base independiente permite examinar generalización externa. No se aprenden imputación ni escalado antes de dividir.

Si cambias datos, reglas o columnas, reinicia el entorno y ejecuta desde el principio. Cambia hiperparámetros antes de evaluar prueba; volver a mirar prueba para elegir convertiría ese conjunto en otra validación.

## Antes de empezar · Uso rápido en tu PC2

**Trabaja en este notebook dentro de Google Colab.** Las rutas `/content/...`, la celda `%pip` y los selectores de archivos ya están preparados para ese uso. Para resolver la práctica no necesitas instalar Python en tu PC ni ejecutar los verificadores locales del proyecto.

### Qué hacer en orden

1. Sube este `.ipynb` a Colab. Ejecuta la sección **1. Preparación del entorno**; deja las herramientas opcionales en `False` salvo que las necesites. Si la instalación pide reiniciar la sesión, reinicia y vuelve a ejecutar las importaciones.
2. En **2. Carga**, usa `SUBIR_ARCHIVO=True` para elegir tu CSV/Excel, o súbelo al panel Archivos y escribe su ruta en `RUTA_ARCHIVO`. Ajusta `FORMATO_ARCHIVO` y, si es CSV, su separador. Ejecuta la carga y revisa los nombres reales de las columnas.
3. **Edita la celda de configuración existente** con tu tipo de problema, target y columnas. Las tablas siguientes explican cada parámetro. No ejecutes todo el notebook antes de completar esa configuración.
4. Revisa calidad, activa solo las correcciones justificadas y realiza el EDA. Continúa con preparación y modelos en orden: los bloques de otros tipos de problema se omiten automáticamente.
5. Compara los resultados de **validación**, escribe el nombre elegido en la celda de elección correspondiente y luego evalúa **prueba**. Un nombre escrito por defecto es un ejemplo; revisa tus resultados antes de conservarlo.
6. Si te piden predicciones futuras o para otra base, completa la sección **10**. Descarga los CSV generados y guarda tu notebook antes de terminar la sesión.

### Qué configuración corresponde a tu pregunta

| Si te piden… | Configura… | Revisa especialmente… |
|---|---|---|
| Predecir una categoría | `TIPO_PROBLEMA="clasificacion"`, `TARGET`, listas numéricas/categóricas; clase positiva si es binaria | Métricas, matriz de confusión y umbral si corresponde. |
| Predecir un monto o cantidad por registro | `TIPO_PROBLEMA="regresion"`, `TARGET` y listas de predictores | Línea base, MAE/RMSE, real frente a predicción y residuos. |
| Pronosticar una serie por fechas | `TIPO_PROBLEMA="serie_tiempo"`, `TARGET`, `FECHA`, formato, frecuencia y horizonte; listas de predictores vacías | Serie ordenada, periodos sin huecos, validación recursiva y backtesting. |

### Ejemplo para adaptar una serie diaria

Copia estos valores **dentro de la celda de configuración del problema** y cambia los nombres y el horizonte según tu enunciado. Es un ejemplo, no una configuración elegida a partir de tus datos:

```python
TIPO_PROBLEMA = "serie_tiempo"
TARGET = "cant_trx"  # Sustituye por tu columna de valores
FECHA = "fecha"     # Sustituye por tu columna de fechas
FORMATO_FECHA_SERIE = "%Y-%m-%d"  # %d/%m/%Y si tu archivo usa día/mes/año
FRECUENCIA = "D"
COLUMNAS_NUMERICAS = []
COLUMNAS_CATEGORICAS = []
REZAGOS = [1, 7, 14]
VENTANAS = [7]
HORIZONTE = 21  # Ejemplo: 21 días; usa el plazo solicitado
```

Si hay historia suficiente, `PERIODOS_VALIDACION_SERIE=21` y `PERIODOS_PRUEBA_SERIE=21` permiten evaluar ventanas de esa duración. Con `None` se mantienen las fracciones. Para un patrón semanal puedes activar `PERIODO_ESTACIONAL=7`; para mostrar la tendencia, `VENTANA_MEDIA_EDA=7`. Si necesitas reportes mensuales, usa `AGREGACION_MENSUAL="sum"` para cantidades acumulables o `"mean"` para niveles.

Para fechas futuras, no necesitas otro CSV: deja `USAR_ARCHIVO_FUTURO=False`. Si el enunciado da una fecha final, escríbela en `FECHA_FIN_PRONOSTICO`; para entregar solo un intervalo, completa las fechas del reporte. Si eliges calendario ampliado, declara también los feriados del horizonte futuro.

**Para redactar tu respuesta:** indica qué limpiaste y por qué, dos hallazgos con evidencia, qué modelo elegiste y con qué métrica de validación, su resultado en prueba y el pronóstico solicitado. Usa los resultados de tu ejecución; las cifras de la solución de referencia pertenecen a sus casos.


## 1. Preparación del entorno
**Objetivo:** disponer de las herramientas que usa el recorrido básico. **Entrada:** una sesión nueva de Colab. **Operación:** instalar versiones compatibles e importar librerías. **Resultado:** versiones registradas y nombres disponibles. **Interpretación:** una instalación sin errores permite continuar; si Colab pide reiniciar, reinicia y vuelve a importar.

Las dependencias de LazyPredict, FLAML, TensorFlow, ProfileReport y PyCaret se instalan solo al activar sus bloques. No actives todo por obligación durante la prueba: cada bloque explica su costo. Las instalaciones deben ejecutarse antes de importar la herramienta correspondiente.

### Opciones costosas: decide antes de instalar
Cada bandera empieza en False: la guía básica puede completarse sin estas herramientas.
- EJECUTAR_PROFILE_REPORT: activa el informe automático del EDA; sus alertas no limpian datos.
- EJECUTAR_LAZY: activa la comparación amplia de modelos por defecto.
- EJECUTAR_FLAML: activa una búsqueda con presupuesto de tiempo y CV.
- EJECUTAR_RED: activa una red neuronal tabular con TensorFlow.
- INSTALAR_OPCIONALES: True instala las dependencias de los bloques que activaste. False supone que ya las instalaste en esta sesión.

Cambia las banderas aquí y ejecuta instalación antes de las importaciones. Cambiarlas después de importar librerías puede requerir reiniciar Colab si pip modifica versiones. PyCaret se trata en un anexo separado porque sus dependencias pueden requerir otro entorno.

In [ ]:
EJECUTAR_PROFILE_REPORT = False
EJECUTAR_LAZY = False
EJECUTAR_FLAML = False
EJECUTAR_RED = False
INSTALAR_OPCIONALES = True

In [ ]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<2.3" "scikit-learn>=1.5,<1.7" "matplotlib>=3.8,<4" "seaborn>=0.13,<1" "xgboost>=2.1,<3" "lightgbm>=4.5,<5" "shap>=0.46,<0.50" openpyxl joblib

In [ ]:
import sys
import subprocess

paquetes_opcionales = (
    (["ydata-profiling[notebook]>=4.18,<5", "setuptools<81"] if EJECUTAR_PROFILE_REPORT else [])
    + (["lazypredict==0.4.0"] if EJECUTAR_LAZY else [])
    + (["flaml[automl]==2.7.0"] if EJECUTAR_FLAML else [])
    + (["tensorflow>=2.18,<2.21"] if EJECUTAR_RED else [])
)
if INSTALAR_OPCIONALES and paquetes_opcionales:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "pandas>=2.2,<3", "numpy>=1.26,<2.3", "scikit-learn>=1.5,<1.7", "xgboost>=2.1,<3",
        *paquetes_opcionales])
print("Python:", sys.version.split()[0])
print("Opcionales solicitadas:", paquetes_opcionales)

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import subprocess
import json
import copy
import time
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, mean_absolute_error, mean_squared_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
versiones_entorno = {p: version(p) for p in ["pandas", "numpy", "scikit-learn", "xgboost", "lightgbm", "shap", "joblib"]}
display(pd.Series(versiones_entorno, name="Versión"))

## 2. Carga y comprensión inicial
**Objetivo:** leer tu base y reconocer qué representa cada fila. **Entrada:** un CSV o Excel que subes a Colab. **Operación:** leerlo con los parámetros indicados. **Resultado:** df_original, primeras filas, dimensiones y tipos. **Interpretación:** confirma nombres, unidades y separadores antes de configurar el target.

| Parámetro | Significado, ejemplo y efecto |
|---|---|
| SUBIR_ARCHIVO | False usa la ruta escrita; True abre el selector de archivos de Colab. Sube un solo archivo. |
| RUTA_ARCHIVO | Ubicación del archivo, por ejemplo /content/clientes.csv. Debe coincidir con Archivos de Colab. |
| FORMATO_ARCHIVO | csv para texto separado; excel para .xlsx. No determina el tipo del problema. |
| SEPARADOR | Carácter entre campos de un CSV: "," o ";". Si toda la fila aparece en una columna, revísalo. |
| CODIFICACION | Cómo se representan letras y tildes; utf-8 es el inicio. latin1 puede corresponder a archivos antiguos. Solo CSV. |
| DECIMAL | "." o "," según cómo se escriben números, por ejemplo 3.5 o 3,5. No elimina separadores de miles. |
| HOJA_EXCEL | 0 lee la primera hoja; "Ventas" lee una hoja por nombre. Solo Excel. |

Si cambias cualquiera de estos valores, vuelve a ejecutar desde la carga. Las candidatas numéricas/categóricas que se muestran orientan; tú decides cuáles serán predictores. Un código postal puede leerse como número y representar una categoría.

In [ ]:
SUBIR_ARCHIVO = False
RUTA_ARCHIVO = "/content/mi_base.csv"
FORMATO_ARCHIVO = "csv"  # "csv" o "excel"
SEPARADOR = ","
CODIFICACION = "utf-8"
DECIMAL = "."
HOJA_EXCEL = 0

In [ ]:
if SUBIR_ARCHIVO:
    from google.colab import files
    archivos_subidos = files.upload()
    if len(archivos_subidos) != 1:
        raise ValueError("Sube exactamente un archivo y vuelve a ejecutar esta celda.")
    ruta_lectura = Path(next(iter(archivos_subidos)))
else:
    ruta_lectura = Path(RUTA_ARCHIVO)

if not ruta_lectura.is_file():
    raise FileNotFoundError(f"No existe {ruta_lectura}. Sube tu base y ajusta RUTA_ARCHIVO.")
if FORMATO_ARCHIVO == "csv":
    df_original = pd.read_csv(ruta_lectura, sep=SEPARADOR, encoding=CODIFICACION, decimal=DECIMAL)
elif FORMATO_ARCHIVO == "excel":
    df_original = pd.read_excel(ruta_lectura, sheet_name=HOJA_EXCEL, decimal=DECIMAL)
else:
    raise ValueError("FORMATO_ARCHIVO debe ser csv o excel.")

if df_original.empty or not df_original.columns.is_unique:
    raise ValueError("La base está vacía o tiene nombres de columnas duplicados.")
print("Filas, columnas:", df_original.shape)
display(df_original.head())
display(df_original.dtypes.to_frame("Tipo leído"))
print("Candidatas numéricas:", df_original.select_dtypes(include="number").columns.tolist())
print("Candidatas categóricas:", df_original.select_dtypes(exclude="number").columns.tolist())

### 2.1 Define qué quieres predecir
**Objetivo:** traducir la pregunta de la prueba a columnas y reglas. **Entrada:** nombres de df_original. **Operación:** declarar opciones, sin elegir automáticamente variables. **Resultado:** configuración que usarán las secciones siguientes. **Interpretación:** los predictores deben estar disponibles en el momento de hacer una predicción real.

| Parámetro | Qué controla y cómo usarlo |
|---|---|
| TIPO_PROBLEMA | clasificacion predice etiquetas como "compra/no compra"; regresion una cantidad como minutos; serie_tiempo valores futuros ordenados por fecha. Cambiarlo cambia el recorrido ejecutado. |
| TARGET | Nombre exacto de la respuesta histórica que se desea predecir, por ejemplo "compra". Nunca lo incluyas como predictor. |
| COLUMNAS_NUMERICAS | Lista de cantidades usadas para predecir, por ejemplo ["edad", "saldo"]. [] no selecciona ninguna. |
| COLUMNAS_CATEGORICAS | Lista de grupos, por ejemplo ["pais", "canal"]. Sus valores se codificarán después. Un número que identifica una categoría también puede ir aquí. |
| COLUMNAS_ID | Identificadores conservados para reconocer filas, por ejemplo ["cliente_id"]. No se entrenan. |
| COLUMNAS_EXCLUIR | Columnas que no quieres modelar, especialmente información que se conoce después del resultado. Por ejemplo "fecha_cancelacion" al predecir fuga. |
| SEMILLA | Entero que permite repetir particiones y componentes aleatorios. 42 es un identificador arbitrario, no un valor óptimo. Cambiarlo puede cambiar resultados. |
| CLASE_POSITIVA | Etiqueta original del evento que interesa detectar, como "Abandona". Solo binaria; no significa necesariamente el número 1. |
| UMBRAL | Probabilidad mínima para anunciar la clase positiva. 0.50 significa 50%. Bajarlo suele detectar más eventos y generar más falsas alarmas; revisa validación. Solo binaria y modelos con probabilidades. |

**Valores de Python:** [] es una lista vacía; {} un diccionario vacío de reglas; None indica ausencia de una opción; True activa y False desactiva. Las etiquetas entre comillas son texto.

En series deja vacías las listas de predictores: el recorrido construye calendario y memoria del target. En tabular necesitas al menos uno. Si hay varias filas de la misma persona o tienda, diseña una partición por entidad antes de usar una separación aleatoria.

Al cambiar target, columnas, problema o semilla, reinicia y ejecuta desde configuración.

### Qué significan las fracciones de validación y prueba
Estos parámetros se declaran ahora, pero **la división se ejecuta después de calidad, limpieza y EDA**, en la sección 5.

**Entrenamiento:** observaciones con las que el modelo aprende relaciones entre predictores y respuesta.

**FRACCION_VALIDACION:** parte de la base que reservas para comparar modelos, ajustar hiperparámetros o elegir un umbral. Con 0.20 reservas el 20% del total. El modelo comparado se entrena sin esas filas; tus decisiones sí usan sus resultados.

**FRACCION_PRUEBA:** parte de la base guardada para evaluar después de elegir el modelo. Con 0.20 reservas otro 20%. No se utiliza para buscar candidatos.

| Ejemplo con 1.000 filas | Cantidad aproximada | Para qué sirve |
|---|---:|---|
| Entrenamiento | 600 | Aprender |
| Validación | 200 | Comparar y ajustar decisiones |
| Prueba | 200 | Evaluar la elección final |

Si aumentas validación a 0.30 manteniendo prueba en 0.20, quedarían 500/300/200. Evaluarás con más casos, pero habrá menos para aprender. Ambos valores deben ser mayores que cero y sumar menos de uno. Los tamaños reales dependen del redondeo y las clases.

Después de elegir, el modelo final aprende con entrenamiento + validación; prueba queda fuera. En series se conservan los últimos periodos como validación y prueba. Cambiar fracciones exige repetir desde la división y volver a entrenar todo.

### Opciones para tratar nulos y preparar predictores
Un nulo representa un dato ausente, no necesariamente cero. Estas opciones **no ejecutan una limpieza al declararlas**; se usan en la preparación posterior a la división.

| Parámetro | Significado, valor inicial y alternativas |
|---|---|
| IMPUTAR_NUMERICAS | False detiene el modelado si hay nulos numéricos. True autoriza sustituirlos con la estrategia elegida, aprendida solo en entrenamiento. |
| ESTRATEGIA_NUMERICA | median usa la mediana; mean el promedio; most_frequent el más repetido; constant el valor indicado. Ejemplo: [10,20,nulo] con median se completa con 15. Una mediana resiste mejor valores extremos que el promedio, sin garantizar que sea adecuada. |
| VALOR_NUMERICO_CONSTANTE | 0 es la sustitución si elegiste constant. Se ignora con otras estrategias; usar cero puede confundir ausencia con una cantidad real. |
| IMPUTAR_CATEGORICAS | False exige categorías sin nulos. True activa la estrategia categórica dentro del Pipeline. |
| ESTRATEGIA_CATEGORICA | most_frequent usa la categoría más común de entrenamiento; constant crea la categoría indicada. |
| VALOR_CATEGORICO_CONSTANTE | "Sin dato" es el texto usado solo con constant. Permite distinguir ausencia de una categoría observada. |
| ESCALAR | True centra las cantidades y las expresa en unidades comparables. Es útil para logística, modelos de distancia y redes; los árboles normalmente no lo necesitan. False conserva la escala original. |
| CATEGORIA_DESCONOCIDA | ignore representa una categoría nueva con ceros en sus columnas one-hot; error detiene la predicción. No aprende categorías nuevas ni garantiza buen desempeño en ellas. |

Las reglas explícitas pueden dejar nulos en df_limpio: el EDA los mostrará. Imputar no es obligatorio ni se realiza antes de dividir. Si cambias estas opciones, vuelve a ejecutar preparación y todo el modelado.

### Opciones de series de tiempo
Se ignoran en clasificación/regresión tabular. Un periodo depende de FRECUENCIA, no siempre significa un día.

| Parámetro | Qué determina, ejemplo y efecto |
|---|---|
| FECHA | Nombre de la columna temporal, por ejemplo "fecha". Debe representar el orden de observación. |
| FORMATO_FECHA_SERIE | Formato explícito al leer fechas, por ejemplo %Y-%m-%d para 2026-12-31. Evita intercambiar día y mes. |
| FRECUENCIA | D diaria, W-MON semanal con lunes como etiqueta, MS mensual con inicio de mes. Debe coincidir con tu serie. |
| REZAGOS | [1,7,14] incorpora el valor de hace 1, 7 y 14 periodos. Más rezagos exigen más historia y agregan predictores; no garantizan mejorar. |
| VENTANAS | [7] agrega el promedio de los siete periodos anteriores. Nunca incluye el target de la misma fila. |
| HORIZONTE | 14 solicita 14 periodos futuros y fija la longitud de cada ventana de backtesting. No cambia la longitud de las reservas de validación/prueba. |
| VENTANAS_BACKTEST | 3 simula tres momentos históricos de pronóstico. Aumentarlo repite entrenamientos y exige más datos. |
| PASO_BACKTEST | 7 separa los orígenes históricos por siete periodos. Si es menor que HORIZONTE, las ventanas se solapan. |
| AGREGACION_SERIE | None conserva una fila por fecha; sum suma los valores del periodo; mean calcula su promedio. Elige según la unidad del target, no por cuál ajuste mejor. |
| PERMITIR_ORDENAR_FECHAS | False exige fechas ordenadas. True autoriza crear otra tabla ordenada; no cambia df_original. |

Ejemplo diario: con historia hasta el 30 de noviembre y horizonte 3, se predicen 1, 2 y 3 de diciembre. lag_7 del 1 de diciembre usa el 24 de noviembre. Reinicia desde la preparación temporal si cambias estas opciones.

### Opciones adicionales de la práctica de regresión y series
Todas las opciones siguientes empiezan desactivadas. No cambies la frecuencia para conseguir artificialmente más observaciones.

| Parámetro | Uso y efecto |
|---|---|
| PERIODOS_VALIDACION_SERIE / PERIODOS_PRUEBA_SERIE | None conserva cada fracción; un entero positivo reserva exactamente esa cantidad de periodos al final. Por ejemplo, 21/21 evalúa dos horizontes de tres semanas si FRECUENCIA="D". |
| CALENDARIO_AMPLIADO | True agrega semana ISO, fin de semana y feriado. Las mismas columnas y el mismo calendario se reutilizan en todas las etapas. |
| FERIADOS | Lista explícita como ["2025-01-01", "2026-01-01"]. Requiere CALENDARIO_AMPLIADO=True. Incluye el horizonte futuro: una fecha omitida se considera no feriado. La lista no es un calendario oficial. |
| PERIODO_ESTACIONAL | None omite la base estacional; 7 en frecuencia diaria repite los siete valores anteriores y los recicla recursivamente. En frecuencia mensual, 12 representa un año. Requiere historia suficiente en cada origen. |
| VENTANA_MEDIA_EDA | None omite la curva suavizada; 7 utiliza los últimos siete periodos, incluido el actual, solo para describir. El predictor media_7 mantiene shift(1). |
| AGREGACION_MENSUAL | None omite tablas mensuales; sum suma flujos como transacciones, mean promedia niveles como un precio. No sumes precios ni interpretes un mes parcial como completo. |

La tabla de particiones muestra inicio, fin y cantidad de periodos. Compara esas cantidades con HORIZONTE: un error recursivo a 60 pasos no equivale a uno a 14 pasos. La evaluación de prueba permanece posterior a la elección.

In [ ]:
TIPO_PROBLEMA = "clasificacion"
TARGET = "CAMBIA_POR_TU_TARGET"
COLUMNAS_NUMERICAS = []  # Ejemplo: ["edad", "saldo"]
COLUMNAS_CATEGORICAS = []  # Ejemplo: ["pais", "canal"]
COLUMNAS_ID = []  # Ejemplo: ["cliente_id"]
COLUMNAS_EXCLUIR = []  # Ejemplo: ["resultado_posterior"] (fuga de información)
SEMILLA = 42
FRACCION_VALIDACION = 0.20
FRACCION_PRUEBA = 0.20
CLASE_POSITIVA = 1  # Puede ser "Sí", "Abandona", etc.
UMBRAL = 0.50

IMPUTAR_NUMERICAS = False
ESTRATEGIA_NUMERICA = "median"  # "mean", "median", "most_frequent" o "constant"
VALOR_NUMERICO_CONSTANTE = 0
IMPUTAR_CATEGORICAS = False
ESTRATEGIA_CATEGORICA = "most_frequent"  # O "constant"
VALOR_CATEGORICO_CONSTANTE = "Sin dato"
ESCALAR = True
CATEGORIA_DESCONOCIDA = "ignore"  # "ignore": todo cero; "error": detenerse

FECHA = "fecha"
FORMATO_FECHA_SERIE = "%Y-%m-%d"
FRECUENCIA = "D"  # "D": diaria, "W-MON": semanal, "MS": inicio de mes
REZAGOS = [1, 7, 14]  # En periodos de FRECUENCIA
VENTANAS = [7]
HORIZONTE = 14
VENTANAS_BACKTEST = 3
PASO_BACKTEST = 7
AGREGACION_SERIE = None  # None, "sum" o "mean": tú decides si agregas periodos
PERMITIR_ORDENAR_FECHAS = False

# Opciones temporales: None conserva las fracciones/horizonte actuales.
PERIODOS_VALIDACION_SERIE = None  # Ejemplo: 21 periodos
PERIODOS_PRUEBA_SERIE = None  # Ejemplo: 21 periodos
CALENDARIO_AMPLIADO = False  # Semana ISO, fin de semana y feriado
FERIADOS = []  # Fechas YYYY-MM-DD conocidas, históricas y futuras; sin país automático
PERIODO_ESTACIONAL = None  # 7 repite la semana anterior en una serie diaria
VENTANA_MEDIA_EDA = None  # 7: media descriptiva de los últimos 7 periodos
AGREGACION_MENSUAL = None  # "sum" para cantidades; "mean" para niveles; None omite

In [ ]:
if TIPO_PROBLEMA not in {"clasificacion","regresion","serie_tiempo"}:
    raise ValueError("Elige clasificacion, regresion o serie_tiempo.")
if TARGET not in df_original:
    raise ValueError(f"TARGET={TARGET!r} no existe. Edita la configuración antes de continuar.")
columnas_configuradas_iniciales = COLUMNAS_NUMERICAS + COLUMNAS_CATEGORICAS + COLUMNAS_ID + COLUMNAS_EXCLUIR
if set(columnas_configuradas_iniciales) - set(df_original.columns):
    raise ValueError(f"Columnas declaradas inexistentes: {set(columnas_configuradas_iniciales) - set(df_original.columns)}")
if TIPO_PROBLEMA == "serie_tiempo" and FECHA not in df_original:
    raise ValueError("La columna FECHA no existe.")

## 3. Control de calidad y limpieza
**Objetivo:** saber si la base puede analizarse sin confundir errores con patrones reales. **Entrada:** df_original. **Operación:** revisar tipos, ausencias, duplicados, constantes e infinitos; después ejecutar solo tus reglas. **Resultado:** diagnóstico inicial y nuevas tablas de limpieza. **Interpretación:** revisa el significado de cada hallazgo antes de corregirlo.

Un ID repetido puede indicar registros duplicados o compras diferentes de una persona. Un campo constante quizá no distingue casos, pero el código no lo elimina. Un infinito no es un valor numérico utilizable por estos modelos. Los nulos se cuentan por columna; su porcentaje ayuda a valorar cuánto falta.

No se elimina ni se imputa nada en el diagnóstico. Primero corrige reglas y tipos. Los nulos del target deben resolverse explícitamente antes del modelado; no se inventa la respuesta histórica.

In [ ]:
perfil_calidad_antes = pd.DataFrame({
    "tipo": df_original.dtypes.astype(str),
    "nulos": df_original.isna().sum(),
    "nulos_pct": df_original.isna().mean().mul(100),
    "distintos": df_original.nunique(dropna=True),
    "constante": df_original.nunique(dropna=True).eq(1),
    "infinitos": {c: int(np.isinf(df_original[c].to_numpy(dtype=float, na_value=np.nan)).sum())
        if pd.api.types.is_numeric_dtype(df_original[c]) else 0 for c in df_original.columns}
})
display(perfil_calidad_antes)
print("Filas duplicadas completas:", df_original.duplicated().sum())
if COLUMNAS_ID:
    print("Identificadores repetidos:", df_original.duplicated(subset=COLUMNAS_ID).sum())

### 3.1 Reemplazos y normalización declarados
**Objetivo:** corregir representaciones conocidas, no aprender decisiones estadísticas. **Entrada:** df_original y tus diccionarios. **Operación:** reemplazar valores y, si lo indicas, normalizar textos o convertir columnas. **Resultado:** df_reglas; el original permanece intacto. **Interpretación:** compara filas y tipos antes de continuar.

**aplicar_reglas(datos, numericas, textos, fechas, reemplazos)** devuelve otro DataFrame.
- datos: tabla de entrada.
- numericas: columnas que se convierten con pd.to_numeric; [] no convierte ninguna.
- textos: columnas a texto, sin espacios extremos y en minúsculas; no se aplica al target salvo que lo listes.
- fechas: diccionario {columna: formato}; {} no convierte fechas.
- reemplazos: diccionario {columna: {valor_actual: nuevo_valor}}.

Ejemplo ejecutable: aplicar_reglas(pd.DataFrame({"edad":["20"],"pais":[" Perú "]}), ["edad"], ["pais"], {}, {}).
Salida esperada: edad=20 y pais="perú", con la entrada sin modificar.

REGLAS_NUMERICAS, REGLAS_TEXTOS, REGLAS_FECHAS y REGLAS_REEMPLAZOS alimentan esos argumentos. Por ejemplo {"pais":{"PE":"Perú"}} cambia solo ese valor. Usa la siguiente función si quieres especificar tipos exactos, como enteros o categorías. Ante cambios, repite desde esta celda y revisa el diagnóstico final.

In [ ]:
def aplicar_reglas(datos, numericas, textos, fechas, reemplazos):
    """Aplica solo reglas declaradas; devuelve otra tabla sin modificar datos."""
    columnas_reglas = set(numericas) | set(textos) | set(fechas) | set(reemplazos)
    columnas_ausentes = columnas_reglas - set(datos.columns)
    if columnas_ausentes:
        raise ValueError(f"Faltan columnas para las reglas: {sorted(columnas_ausentes)}")
    etapa_reemplazos = datos.replace(reemplazos)
    etapa_numerica = etapa_reemplazos.assign(**{
        c: pd.to_numeric(etapa_reemplazos[c], errors="raise") for c in numericas
    })
    etapa_texto = etapa_numerica.assign(**{
        c: etapa_numerica[c].astype("string").str.strip().str.lower().astype(object)
            .where(etapa_numerica[c].notna(), np.nan) for c in textos
    })
    etapa_fechas = etapa_texto.assign(**{
        c: pd.to_datetime(etapa_texto[c], format=f, errors="raise") for c, f in fechas.items()
    })
    return etapa_fechas

REGLAS_NUMERICAS = []
REGLAS_TEXTOS = []
REGLAS_FECHAS = {}  # Ejemplo: {"fecha_alta": "%d/%m/%Y"}
REGLAS_REEMPLAZOS = {}  # Ejemplo: {"pais": {"PE": "Perú"}}
df_reglas = aplicar_reglas(df_original, REGLAS_NUMERICAS, REGLAS_TEXTOS, REGLAS_FECHAS, REGLAS_REEMPLAZOS)
display(df_reglas.head())

### 3.2 Cambiar explícitamente el tipo de los datos
**Objetivo:** evitar que "25" se trate como una categoría o que una fecha se interprete como texto. **Entrada:** df_reglas. **Operación:** convertir solo las columnas declaradas. **Resultado:** df_tipos_corregidos con las mismas filas. **Interpretación:** verifica los tipos y valores; una conversión incorrecta se detiene, no se transforma silenciosamente en un nulo.

**cambiar_tipos(datos, tipos, formatos_fecha=None)**
- datos: DataFrame que no se modifica.
- tipos: diccionario {columna: tipo}; admite numero, entero, decimal, texto, categoria y fecha. {} conserva todos los tipos.
- formatos_fecha: diccionario de formatos, por ejemplo {"fecha":"%d/%m/%Y"}. Se requiere un formato para cada conversión de fecha.
- Devuelve una tabla nueva. entero usa Int64, que permite nulos; numero deja que pandas determine entero o decimal; decimal fuerza float64.

Ejemplo ejecutable: cambiar_tipos(pd.DataFrame({"edad":["25"],"fecha":["31/12/2026"]}), {"edad":"entero","fecha":"fecha"}, {"fecha":"%d/%m/%Y"}).
Salida esperada: edad vale 25 con tipo Int64 y fecha vale 2026-12-31 con tipo datetime64. La entrada conserva sus textos.

TIPOS_COLUMNAS y FORMATOS_FECHA son las reglas de esta celda y se guardarán para la base nueva. Un ID "0012" debe seguir como texto si sus ceros forman parte de la identificación. No uses conversión booleana directa para "Sí"/"No": declara un reemplazo explícito en la celda anterior.

In [ ]:
def cambiar_tipos(datos, tipos, formatos_fecha=None):
    """Devuelve otra tabla con tipos declarados; errores de conversión se propagan."""
    formatos_declarados = {} if formatos_fecha is None else formatos_fecha
    tipos_admitidos = {"numero", "entero", "decimal", "texto", "categoria", "fecha"}
    if set(tipos) - set(datos.columns):
        raise ValueError(f"Columnas para conversión no presentes: {set(tipos) - set(datos.columns)}")
    if set(tipos.values()) - tipos_admitidos:
        raise ValueError(f"Tipos admitidos: {sorted(tipos_admitidos)}")
    columnas_fecha_sin_formato = {c for c, t in tipos.items() if t == "fecha"} - set(formatos_declarados)
    if columnas_fecha_sin_formato:
        raise ValueError(f"Declara formato para: {columnas_fecha_sin_formato}")
    tabla_numeros = datos.assign(**{
        c: pd.to_numeric(datos[c], errors="raise") for c, t in tipos.items() if t in {"numero", "entero", "decimal"}})
    tabla_precision = tabla_numeros.assign(**{
        c: tabla_numeros[c].astype("Int64" if t == "entero" else "float64")
        for c, t in tipos.items() if t in {"entero", "decimal"}})
    tabla_textos = tabla_precision.assign(**{
        c: tabla_precision[c].astype("string").astype(object).where(tabla_precision[c].notna(), np.nan)
        for c, t in tipos.items() if t == "texto"})
    tabla_categorias = tabla_textos.assign(**{
        c: tabla_textos[c].astype("category") for c, t in tipos.items() if t == "categoria"})
    tabla_fechas = tabla_categorias.assign(**{
        c: pd.to_datetime(tabla_categorias[c], format=formatos_declarados[c], errors="raise")
        for c, t in tipos.items() if t == "fecha"})
    return tabla_fechas

TIPOS_COLUMNAS = {}  # Ejemplo: {"edad": "entero", "pais": "categoria"}
FORMATOS_FECHA = {}  # Ejemplo: {"fecha": "%d/%m/%Y"}
df_tipos_corregidos = cambiar_tipos(df_reglas, TIPOS_COLUMNAS, FORMATOS_FECHA)
display(df_tipos_corregidos.dtypes.to_frame("Tipo después de convertir"))

### 3.3 Posibles atípicos: observar antes de eliminar
**Objetivo:** localizar cantidades inusuales para revisarlas. **Entrada:** columnas numéricas de df_tipos_corregidos. **Operación:** calcular cuartiles y límites IQR. **Resultado:** límites e índices candidatos. **Interpretación:** estar fuera de un límite no significa que el dato sea incorrecto.

COLUMNAS_CALIDAD_NUMERICAS indica qué columnas revisar; la lista inicial muestra las numéricas leídas, sin seleccionarlas para el modelo. FACTOR_IQR=1.5 coloca límites a 1.5 rangos intercuartiles por debajo de Q1 y por encima de Q3. Aumentarlo marca menos candidatos; reducirlo marca más. IQR=Q3-Q1 representa la amplitud del 50% central.

Ejemplo: Q1=10, Q3=20 e IQR=10 producen límites -5 y 35 con factor 1.5. Un valor 100 merece revisión, pero puede ser una venta legítima. indices_atipicos solo lista filas; la eliminación requiere tu llamada explícita posterior. Cambiar columnas o factor exige repetir este diagnóstico, no necesariamente entrenar de nuevo si no limpias nada.

In [ ]:
COLUMNAS_CALIDAD_NUMERICAS = df_tipos_corregidos.select_dtypes(include="number").columns.tolist()
FACTOR_IQR = 1.5
cuartil_inferior = df_tipos_corregidos[COLUMNAS_CALIDAD_NUMERICAS].quantile(0.25)
cuartil_superior = df_tipos_corregidos[COLUMNAS_CALIDAD_NUMERICAS].quantile(0.75)
rango_intercuartil = cuartil_superior - cuartil_inferior
limites_iqr = pd.DataFrame({"inferior": cuartil_inferior - FACTOR_IQR * rango_intercuartil,
    "superior": cuartil_superior + FACTOR_IQR * rango_intercuartil})
mascara_atipicos = df_tipos_corregidos[COLUMNAS_CALIDAD_NUMERICAS].lt(limites_iqr["inferior"]) | df_tipos_corregidos[COLUMNAS_CALIDAD_NUMERICAS].gt(limites_iqr["superior"])
indices_atipicos = df_tipos_corregidos.index[mascara_atipicos.any(axis=1)].tolist()
display(limites_iqr.assign(candidatos=mascara_atipicos.sum()))
display(df_tipos_corregidos.loc[indices_atipicos].head(20))
print("Filas candidatas, sin eliminar:", len(indices_atipicos))

### 3.4 Ejecutar solo las limpiezas que decides
**Objetivo:** crear la base que analizarás. **Entrada:** df_tipos_corregidos. **Operación:** eliminar únicamente columnas, índices o duplicados indicados. **Resultado:** df_limpio y tablas intermedias. **Interpretación:** compara tamaños y calidad; no elimines filas por mejorar una métrica.

**limpiar_explicitamente(datos, columnas=None, indices=None, quitar_duplicados=False, subset=None)**
- datos: entrada que permanece intacta.
- columnas: nombres a retirar; None no retira columnas.
- indices: lista de etiquetas de filas a retirar; None conserva todas.
- quitar_duplicados: solo True autoriza deduplicar.
- subset: columnas que definen un duplicado; None considera toda la fila.
- Devuelve otro DataFrame; nombres o índices inexistentes provocan error.

Ejemplo ejecutable: limpiar_explicitamente(pd.DataFrame({"a":[1,1,2]}), quitar_duplicados=True).
Salida esperada: dos filas, índices [0,2], valores [1,2].

Cada etapa tiene una línea activa que continúa sin limpiar y una alternativa comentada. Para activar la alternativa, comenta primero la línea de continuidad; **ejecuta solo una asignación para ese nombre de salida**. Si retiras un predictor, actualiza tus listas. No apliques filtro de nulos a toda la base sin revisar qué filas perderías.

En series, eliminar fechas interiores puede crear huecos: el control temporal posterior los detectará y te pedirá decidir cómo resolverlos.

In [ ]:
def limpiar_explicitamente(datos, columnas=None, indices=None, quitar_duplicados=False, subset=None):
    """Retorna otra tabla al aplicar exclusivamente las eliminaciones solicitadas."""
    etapa_columnas = datos.drop(columns=[] if columnas is None else columnas)
    etapa_filas = etapa_columnas.drop(index=[] if indices is None else indices)
    etapa_duplicados = etapa_filas.drop_duplicates(subset=subset) if quitar_duplicados else etapa_filas.copy()
    return etapa_duplicados

df_sin_columnas = df_tipos_corregidos.copy()
# df_sin_columnas = limpiar_explicitamente(df_tipos_corregidos, columnas=["columna_revisada"])
df_sin_filas = df_sin_columnas.copy()
# df_sin_filas = limpiar_explicitamente(df_sin_columnas, indices=[10, 25])
# Para targets nulos, después de revisar: indices=df_sin_columnas.index[df_sin_columnas[TARGET].isna()].tolist()
df_sin_duplicados = df_sin_filas.copy()
# df_sin_duplicados = limpiar_explicitamente(df_sin_filas, quitar_duplicados=True, subset=COLUMNAS_ID or None)
df_limpio = df_sin_duplicados.copy()
if df_limpio.empty:
    raise ValueError("La limpieza dejó la base vacía.")

### 3.5 Confirmar la calidad de la salida
**Objetivo:** verificar qué cambió y decidir si puedes pasar al análisis. **Entrada:** df_original y df_limpio. **Operación:** comparar dimensiones, tipos y faltantes. **Resultado:** historial de etapas y diagnóstico final. **Interpretación:** los problemas restantes requieren tu decisión; que la tabla se llame limpia no significa que el programa haya resuelto todo.

Los nulos de predictores pueden conservarse para imputarlos solo dentro del entrenamiento posterior. No se rellenan aquí con estadísticas de toda la base. El EDA informará de los faltantes que hayas decidido conservar.

In [ ]:
comparacion_etapas = pd.DataFrame({
    "Filas": [len(t) for t in [df_original, df_reglas, df_tipos_corregidos, df_sin_columnas, df_sin_filas, df_sin_duplicados, df_limpio]],
    "Columnas": [t.shape[1] for t in [df_original, df_reglas, df_tipos_corregidos, df_sin_columnas, df_sin_filas, df_sin_duplicados, df_limpio]]
}, index=["Original", "Reglas", "Tipos", "Columnas", "Filas", "Duplicados", "Limpio"])
perfil_calidad_despues = pd.DataFrame({
    "tipo": df_limpio.dtypes.astype(str), "nulos": df_limpio.isna().sum(),
    "nulos_pct": df_limpio.isna().mean().mul(100), "distintos": df_limpio.nunique(dropna=True),
    "constante": df_limpio.nunique(dropna=True).eq(1),
    "infinitos": {c: int(np.isinf(df_limpio[c].to_numpy(dtype=float, na_value=np.nan)).sum())
        if pd.api.types.is_numeric_dtype(df_limpio[c]) else 0 for c in df_limpio.columns}
})
display(comparacion_etapas)
display(perfil_calidad_despues)
print("Duplicados completos después:", df_limpio.duplicated().sum())

## 4. Análisis exploratorio de la base limpia
**Objetivo:** entender distribuciones y relaciones después de las correcciones elegidas. **Entrada:** df_limpio. **Operación:** resumir variables, graficar y comparar grupos. **Resultado:** tablas, figuras y hallazgos escritos por ti. **Interpretación:** busca explicaciones plausibles y posibles predictores; una asociación no demuestra causalidad.

Este bloque sigue las referencias: categorías → estadísticos y distribuciones → análisis bivariado → correlaciones → perfiles agrupados → hallazgos. La división para modelar se ejecutará después.

| Parámetro | Qué modifica |
|---|---|
| COLUMNAS_EDA_NUMERICAS | Cantidades a explorar; inicialmente tus predictores numéricos, o el target para series. Agregar una columna solo la grafica, no la incorpora al modelo. |
| COLUMNAS_EDA_CATEGORICAS | Grupos a explorar; inicialmente tus predictores categóricos. |
| MAX_GRAFICOS | 6 limita cuántas columnas de cada lista se grafican para controlar tiempo y tamaño. |
| TOP_CATEGORIAS | 15 limita categorías mostradas en tablas y barras; las restantes no se borran de df_limpio. |
| BINS | 30 intervalos de histogramas. Más intervalos revelan detalle y ruido; menos agrupan valores. |

Para cambiar solo las figuras, repite desde el EDA. Los valores ausentes que conservaste se informan; no se transforman para poder dibujar.

In [ ]:
COLUMNAS_EDA_NUMERICAS = COLUMNAS_NUMERICAS if TIPO_PROBLEMA != "serie_tiempo" else [TARGET]
COLUMNAS_EDA_CATEGORICAS = COLUMNAS_CATEGORICAS
MAX_GRAFICOS = 6
TOP_CATEGORIAS = 15
BINS = 30
plt.figure(figsize=(10, 3))
sns.heatmap(df_limpio.head(200).isna(), cbar=False)
plt.title("Faltantes conservados: primeras 200 filas de df_limpio")
plt.show()

### 4.0 ProfileReport: panorama de la base limpia
**Objetivo:** obtener un informe navegable que complemente el EDA. **Entrada:** df_limpio, o las columnas que indiques. **Operación:** ProfileReport calcula resúmenes y alertas sin cambiar la tabla. **Resultado:** informe integrado en Colab y, opcionalmente, un HTML descargable. **Interpretación:** revisa ausencias, tipos y distribuciones; una alerta no justifica por sí sola eliminar datos.

| Parámetro | Uso y efecto |
|---|---|
| EJECUTAR_PROFILE_REPORT | Bandera de la sección 1. Debe ser True para instalar y ejecutar esta parte. |
| PERFIL_MINIMO | True evita cálculos costosos como múltiples correlaciones e interacciones. False solicita un informe más completo y puede tardar mucho más. |
| COLUMNAS_PERFIL | [] incluye todas las columnas de df_limpio; una lista limita el informe a esas columnas, sin quitarlas de la base. |
| TITULO_PERFIL | Texto que identifica la base del informe; no afecta las estadísticas. |
| EXPORTAR_PERFIL | True escribe el HTML; False solo lo muestra. |
| RUTA_PERFIL | /content/reporte_eda.html inicialmente. Cambia el nombre si quieres conservar varios reportes. |

Ejemplo de uso: ProfileReport(df_limpio, title="EDA", minimal=True).
Salida esperada: un objeto de informe. to_notebook_iframe() lo muestra; to_file(...) crea el HTML. No es una función propia de limpieza: es la clase de ydata_profiling.
Cambiar estas opciones solo requiere repetir el reporte. El EDA manual sigue disponible si no lo activas.

In [ ]:
PERFIL_MINIMO = True
COLUMNAS_PERFIL = []
TITULO_PERFIL = "EDA de la base limpia"
EXPORTAR_PERFIL = True
RUTA_PERFIL = "/content/reporte_eda.html"

if EJECUTAR_PROFILE_REPORT:
    from ydata_profiling import ProfileReport
    df_para_perfil = df_limpio.loc[:, COLUMNAS_PERFIL].copy() if COLUMNAS_PERFIL else df_limpio.copy()
    informe_perfil = ProfileReport(df_para_perfil, title=TITULO_PERFIL, minimal=PERFIL_MINIMO)
    informe_perfil.to_notebook_iframe()
    if EXPORTAR_PERFIL:
        Path(RUTA_PERFIL).parent.mkdir(parents=True, exist_ok=True)
        informe_perfil.to_file(RUTA_PERFIL)
        print("Reporte HTML:", RUTA_PERFIL)
        # from google.colab import files
        # files.download(RUTA_PERFIL)
    # Reactiva el backend inline si el reporte modifica el renderizado de Colab.
    if "google.colab" in sys.modules:
        get_ipython().run_line_magic("matplotlib", "inline")

### 4.1 Variables categóricas
**Entrada:** df_limpio y COLUMNAS_EDA_CATEGORICAS. **Operación:** contar categorías y sus porcentajes. **Resultado:** tablas y barras para las categorías más frecuentes. **Interpretación:** revisa grupos desbalanceados, etiquetas inconsistentes y cardinalidad elevada; no agrupes ni borres categorías solo porque sean poco frecuentes.

Una columna con un valor distinto por fila podría ser un ID. La barra muestra frecuencia, no importancia predictiva. TOP_CATEGORIAS limita solo la visualización.

In [ ]:
for columna_frecuencia in COLUMNAS_EDA_CATEGORICAS[:MAX_GRAFICOS]:
    tabla_frecuencias = pd.concat([
        df_limpio[columna_frecuencia].value_counts(dropna=False).rename("Frecuencia"),
        df_limpio[columna_frecuencia].value_counts(dropna=False, normalize=True).mul(100).rename("Porcentaje")
    ], axis=1)
    display(tabla_frecuencias.head(TOP_CATEGORIAS))
    tabla_frecuencias["Frecuencia"].head(TOP_CATEGORIAS).plot.bar(title=columna_frecuencia, figsize=(8, 3))
    plt.show()

### 4.2 Estadísticos y distribuciones
**Entrada:** cantidades de df_limpio. **Operación:** describir y dibujar histogramas y boxplots. **Resultado:** posición, dispersión y forma de las variables. **Interpretación:** media y mediana diferentes pueden acompañar asimetría; el histograma permite ver concentraciones y colas. Un boxplot localiza valores extremos, pero no decide eliminarlos.

count cuenta valores no nulos; mean es promedio; std desviación estándar; min/max son extremos; 25%,50%,75% son cuartiles. La mediana es 50%. La desviación está en las unidades de la variable.

En clasificación también se muestra el tamaño de cada clase; en regresión/series, la distribución del target. Si aparecen errores de tipo, vuelve a calidad y declara la conversión correcta.

In [ ]:
display(df_limpio.describe(include="all").T)

In [ ]:
for columna_eda in COLUMNAS_EDA_NUMERICAS[:MAX_GRAFICOS]:
    if not pd.api.types.is_numeric_dtype(df_limpio[columna_eda]):
        raise TypeError(f"{columna_eda} no es numérica; configura una conversión explícita.")
    fig_distribucion, ejes_distribucion = plt.subplots(1, 2, figsize=(10, 3))
    sns.histplot(df_limpio[columna_eda], bins=BINS, ax=ejes_distribucion[0])
    sns.boxplot(x=df_limpio[columna_eda], ax=ejes_distribucion[1])
    fig_distribucion.suptitle(columna_eda)
    plt.tight_layout()
    plt.show()

if TIPO_PROBLEMA == "clasificacion":
    display(df_limpio[TARGET].value_counts(dropna=False).to_frame("Distribución del target"))
    df_limpio[TARGET].value_counts().plot.bar(title="Clases en la base limpia")
    plt.show()
else:
    sns.histplot(df_limpio[TARGET], bins=BINS)
    plt.title("Distribución del target")
    plt.show()

### 4.3 Análisis bivariado
**Entrada:** predictores y target de df_limpio. **Operación:** comparar dos variables a la vez. **Resultado:** cajas por clase y proporciones categóricas en clasificación, dispersiones y promedios por grupo en regresión, calendario en series. **Interpretación:** busca asociaciones, tamaños de grupo y posibles errores; no atribuyas causalidad.

Ejemplo: distintas distribuciones de edad entre compradores y no compradores sugieren revisar edad como predictor, pero no prueban que cambiar edad cause la compra. Este EDA sigue describiendo toda la base disponible antes de dividir.

### Utilidades de calendario y resumen temporal
Se definen aquí para usarlas en el EDA y reutilizarlas durante el modelado; definirlas no entrena modelos ni cambia tablas.

- `variables_calendario(fechas, feriados=(), calendario_ampliado=False)` devuelve una tabla nueva. Con True añade semana ISO, fin de semana (sábado/domingo) y feriados declarados. Semana ISO y año calendario pueden pertenecer a años diferentes cerca del 1 de enero.
- `construir_rezagos(historia, rezagos, ventanas, feriados=(), calendario_ampliado=False)` y `pronostico_recursivo(modelo, historia, fechas, rezagos, ventanas, feriados=(), calendario_ampliado=False, periodo_estacional=None)` se explican con ejemplos en 6.4. Los argumentos nuevos conservan los valores anteriores por defecto.
- `resumir_mensual(serie, frecuencia, agregacion)` devuelve valor, periodos observados/esperados, cobertura porcentual y mes_completo. Cuenta periodos de la frecuencia declarada, no presupone días. Los huecos no se convierten en ceros.
- `errores_temporales(reales, predicciones)` requiere dos Series con las mismas fechas y devuelve real, predicción, paso, día de semana, residuo y error absoluto.

Estos bloques se inspiran en **PD_Solucion_Regresion_Series.ipynb**. Conservamos la media de entrenamiento desplazada un periodo para evitar la fuga del target que aparece en la referencia.

In [ ]:
def variables_calendario(fechas, feriados=(), calendario_ampliado=False):
    """Devuelve calendario conocido de antemano, sin modificar fechas ni feriados."""
    indice = pd.DatetimeIndex(fechas)
    fechas_feriado = pd.DatetimeIndex(pd.to_datetime(list(feriados), format="%Y-%m-%d", errors="raise"))
    if indice.hasnans or fechas_feriado.hasnans:
        raise ValueError("El calendario no admite fechas vacías.")
    basicas = pd.DataFrame({"anio": indice.year, "mes": indice.month,
        "dia": indice.day, "dia_semana": indice.dayofweek}, index=indice)
    return basicas.assign(semana_iso=indice.isocalendar().week.to_numpy(dtype=int),
        fin_de_semana=(indice.dayofweek >= 5).astype(int),
        es_feriado=indice.normalize().isin(fechas_feriado.normalize()).astype(int)) if calendario_ampliado else basicas


def construir_rezagos(historia, rezagos, ventanas, feriados=(), calendario_ampliado=False):
    """Construye otra tabla; cada predictor histórico usa solo periodos anteriores."""
    tabla_calendario = variables_calendario(historia.index, feriados, calendario_ampliado)
    tabla_memoria = tabla_calendario.assign(**{
        **{f"lag_{k}": historia.shift(k) for k in rezagos},
        **{f"media_{k}": historia.shift(1).rolling(k, min_periods=k).mean() for k in ventanas}})
    return tabla_memoria.assign(objetivo=historia.to_numpy())


def pronostico_recursivo(modelo, historia, fechas, rezagos, ventanas, feriados=(),
                        calendario_ampliado=False, periodo_estacional=None):
    """Pronostica sin targets futuros; None usa persistencia o la base estacional."""
    if periodo_estacional is not None and (isinstance(periodo_estacional, bool)
            or not isinstance(periodo_estacional, int) or periodo_estacional < 1):
        raise ValueError("El periodo estacional debe ser un entero positivo.")
    if periodo_estacional is not None and modelo is not None:
        raise ValueError("El periodo estacional se usa únicamente con modelo=None.")
    memoria = max(rezagos + ventanas + ([periodo_estacional] if periodo_estacional else []))
    if len(historia) < memoria:
        raise ValueError("Historia insuficiente para construir la memoria temporal.")
    if len(fechas) == 0 or fechas.freq is None:
        raise ValueError("fechas debe ser un DatetimeIndex no vacío con frecuencia explícita.")
    esperadas_recursion = pd.date_range(historia.index[-1] + fechas.freq, periods=len(fechas), freq=fechas.freq)
    historia_esperada = pd.date_range(historia.index[0], periods=len(historia), freq=fechas.freq)
    if not historia.index.equals(historia_esperada) or not np.isfinite(historia.to_numpy(dtype=float)).all():
        raise ValueError("La historia debe ser regular, ordenada y finita.")
    if not fechas.equals(esperadas_recursion):
        raise ValueError("El pronóstico debe comenzar en el siguiente periodo y ser consecutivo.")
    calendario_futuro = variables_calendario(fechas, feriados, calendario_ampliado)
    valores_conocidos = historia.to_numpy(dtype=float, na_value=np.nan).tolist()
    predicciones_recursivas = []
    for numero_paso, fecha_paso in enumerate(fechas):
        fila_paso = calendario_futuro.iloc[[numero_paso]].assign(**{
            **{f"lag_{k}": [valores_conocidos[-k]] for k in rezagos},
            **{f"media_{k}": [np.mean(valores_conocidos[-k:])] for k in ventanas}})
        prediccion_paso = float(valores_conocidos[-(periodo_estacional or 1)]) if modelo is None else float(modelo.predict(fila_paso)[0])
        if not np.isfinite(prediccion_paso):
            raise ValueError("El modelo produjo un pronóstico no finito.")
        predicciones_recursivas.append(prediccion_paso)
        valores_conocidos.append(prediccion_paso)
    return pd.Series(predicciones_recursivas, index=fechas, name="prediccion")


def resumir_mensual(serie, frecuencia, agregacion):
    """Agrega sin rellenar huecos; marca cobertura según la frecuencia de la serie."""
    if agregacion not in {"sum", "mean"}:
        raise ValueError("La agregación mensual debe ser sum o mean.")
    if serie.empty or not isinstance(serie.index, pd.DatetimeIndex) or not serie.index.is_unique or serie.index.hasnans:
        raise ValueError("El resumen mensual requiere fechas únicas y no vacías.")
    ordenada = serie.sort_index()
    inicio_mes = ordenada.index[0].to_period("M").start_time
    fin_mes = ordenada.index[-1].to_period("M").end_time
    # Extiende la misma cuadrícula en ambas direcciones para medir meses parciales.
    desplazamiento = pd.tseries.frequencies.to_offset(frecuencia)
    if isinstance(desplazamiento, pd.tseries.offsets.Tick):
        inicio_alineado = ordenada.index[0] - ((ordenada.index[0] - inicio_mes) // pd.Timedelta(desplazamiento)) * desplazamiento
        anteriores = pd.date_range(start=inicio_alineado, end=ordenada.index[0], freq=frecuencia)
    else:
        anteriores_amplias = pd.date_range(end=ordenada.index[0], periods=32, freq=frecuencia)
        anteriores = anteriores_amplias[anteriores_amplias >= inicio_mes]
    posteriores = pd.date_range(start=ordenada.index[0], end=fin_mes, freq=frecuencia)
    esperadas = anteriores.union(posteriores)
    if len(ordenada.index.difference(esperadas)):
        raise ValueError("Las fechas del resumen no coinciden con la frecuencia declarada.")
    meses = pd.period_range(inicio_mes, fin_mes, freq="M")
    grupos = ordenada.groupby(ordenada.index.to_period("M"))
    valores = grupos.sum(min_count=1) if agregacion == "sum" else grupos.mean()
    conteo_esperado = pd.Series(1, index=esperadas).groupby(esperadas.to_period("M")).sum()
    resumen = pd.DataFrame({"valor": valores, "periodos_observados": grupos.count(),
        "periodos_esperados": conteo_esperado}).reindex(meses)
    conteos = resumen.assign(periodos_observados=resumen["periodos_observados"].fillna(0).astype(int),
        periodos_esperados=resumen["periodos_esperados"].fillna(0).astype(int))
    return conteos.assign(cobertura_pct=conteos["periodos_observados"].div(conteos["periodos_esperados"].replace(0, np.nan)).mul(100),
        mes_completo=conteos["periodos_observados"].eq(conteos["periodos_esperados"]) & conteos["periodos_esperados"].gt(0)).rename_axis("mes")


def errores_temporales(reales, predicciones):
    """Devuelve errores alineados por fecha y paso desde el origen, sin mutar series."""
    if not reales.index.equals(predicciones.index):
        raise ValueError("Reales y predicciones deben tener las mismas fechas y orden.")
    tabla = pd.DataFrame({"Real": reales.to_numpy(), "Predicción": predicciones.to_numpy(),
        "paso": np.arange(1, len(reales) + 1), "dia_semana": reales.index.dayofweek}, index=reales.index)
    return tabla.assign(residuo=tabla["Real"] - tabla["Predicción"],
        error_absoluto=np.abs(tabla["Real"] - tabla["Predicción"]))

In [ ]:
if TIPO_PROBLEMA == "clasificacion":
    for columna_bivariada in COLUMNAS_EDA_NUMERICAS[:MAX_GRAFICOS]:
        sns.boxplot(data=df_limpio, x=TARGET, y=columna_bivariada)
        plt.title(f"{columna_bivariada} por clase")
        plt.show()
    for columna_cruce in COLUMNAS_EDA_CATEGORICAS[:MAX_GRAFICOS]:
        tabla_cruzada = pd.crosstab(df_limpio[columna_cruce], df_limpio[TARGET], normalize="index")
        display(tabla_cruzada.head(TOP_CATEGORIAS))
elif TIPO_PROBLEMA == "regresion":
    for columna_dispersion in COLUMNAS_EDA_NUMERICAS[:MAX_GRAFICOS]:
        sns.scatterplot(data=df_limpio, x=columna_dispersion, y=TARGET, alpha=0.5)
        plt.show()
    for columna_segmento_eda in COLUMNAS_EDA_CATEGORICAS[:MAX_GRAFICOS]:
        display(df_limpio.groupby(columna_segmento_eda, dropna=False)[TARGET].agg(["count", "mean", "median"]).head(TOP_CATEGORIAS))
else:
    df_limpio.plot(x=FECHA, y=TARGET, figsize=(12, 4), title="Serie de base limpia")
    plt.show()
    resumen_calendario = df_limpio.assign(dia_semana=pd.to_datetime(df_limpio[FECHA], format=FORMATO_FECHA_SERIE).dt.dayofweek,
        mes=pd.to_datetime(df_limpio[FECHA], format=FORMATO_FECHA_SERIE).dt.month)
    display(resumen_calendario.groupby("dia_semana")[TARGET].agg(["count", "mean", "median"]))
    display(resumen_calendario.groupby("mes")[TARGET].agg(["count", "mean", "median"]))

if TIPO_PROBLEMA == "serie_tiempo":
    serie_eda_fechas = df_limpio.assign(**{FECHA: pd.to_datetime(df_limpio[FECHA], format=FORMATO_FECHA_SERIE, errors="raise")}).set_index(FECHA)[TARGET].sort_index()
    if AGREGACION_SERIE in {"sum", "mean"}:
        grupos_eda_periodos = serie_eda_fechas.resample(FRECUENCIA)
        serie_eda_periodos = grupos_eda_periodos.sum(min_count=1) if AGREGACION_SERIE == "sum" else grupos_eda_periodos.mean()
    else:
        serie_eda_periodos = serie_eda_fechas.copy()
    if VENTANA_MEDIA_EDA is not None:
        if isinstance(VENTANA_MEDIA_EDA, bool) or not isinstance(VENTANA_MEDIA_EDA, int) or VENTANA_MEDIA_EDA < 1:
            raise ValueError("VENTANA_MEDIA_EDA debe ser None o un entero positivo.")
        if not serie_eda_periodos.index.is_unique:
            raise ValueError("Decide una agregación para las fechas repetidas antes de suavizar.")
        curva_descriptiva = pd.DataFrame({"Serie": serie_eda_periodos,
            "Media descriptiva": serie_eda_periodos.rolling(VENTANA_MEDIA_EDA, min_periods=VENTANA_MEDIA_EDA).mean()})
        curva_descriptiva.plot(figsize=(12, 4), title="EDA: media descriptiva, no predictor")
        plt.show()
    if AGREGACION_MENSUAL is not None:
        resumen_mensual_eda = resumir_mensual(serie_eda_periodos, FRECUENCIA, AGREGACION_MENSUAL)
        display(resumen_mensual_eda)
        resumen_mensual_eda["valor"].plot.bar(figsize=(12, 4), title=f"Histórico mensual ({AGREGACION_MENSUAL}); revisar cobertura")
        plt.show()

### 4.4 Correlaciones numéricas
**Entrada:** columnas numéricas de df_limpio; en regresión/series se incluye el target numérico. **Operación:** calcular correlación de Pearson. **Resultado:** matriz de -1 a 1. **Interpretación:** cerca de 1 indica asociación lineal positiva; cerca de -1 negativa; cerca de 0 puede ocultar relaciones no lineales.

No se asignan números arbitrarios a categorías para interpretar su correlación. Variables muy correlacionadas pueden ser redundantes, pero aquí no se elimina ninguna. Una columna constante puede producir NaN porque su correlación no está definida.

In [ ]:
columnas_correlacion = list(dict.fromkeys(COLUMNAS_EDA_NUMERICAS + ([TARGET] if TIPO_PROBLEMA != "clasificacion" else [])))
if len(columnas_correlacion) >= 2:
    matriz_correlacion = df_limpio[columnas_correlacion].corr()
    plt.figure(figsize=(9, 7))
    sns.heatmap(matriz_correlacion, cmap="coolwarm", center=0, vmin=-1, vmax=1)
    plt.title("Correlación entre variables numéricas · base limpia")
    plt.show()

### 4.5 Perfiles agrupados y hallazgos
**Objetivo:** comparar segmentos útiles para la pregunta. **Entrada:** df_limpio y una lista de columnas. **Operación:** resumir el target por grupo. **Resultado:** conteos y promedios o proporciones por clase. **Interpretación:** no atribuyas diferencias a grupos con pocas observaciones sin revisar el tamaño.

COLUMNAS_PERFILES empieza como []; escribe por ejemplo ["canal"] para agrupar por canal. Para clasificación se muestran proporciones por clase; para regresión/series, count/mean/median del target. Si el target debería ser numérico pero sigue como texto, corrige tipos antes.

Ejemplo: un grupo con 2 casos y tasa 100% aporta menos evidencia que uno con 500 casos y tasa 60%. Escribe después qué patrones observaste y qué variables estarán realmente disponibles al predecir. Cambiar agrupaciones solo requiere repetir esta celda.

In [ ]:
COLUMNAS_PERFILES = []
for columna_perfil in COLUMNAS_PERFILES:
    if TIPO_PROBLEMA == "clasificacion":
        display(pd.crosstab(df_limpio[columna_perfil], df_limpio[TARGET], margins=True))
        display(pd.crosstab(df_limpio[columna_perfil], df_limpio[TARGET], normalize="index"))
    else:
        display(df_limpio.groupby(columna_perfil, dropna=False)[TARGET].agg(["count", "mean", "median"]))
# Escribe tus hallazgos en una celda de texto: patrón, evidencia, decisión y limitación.

## 5. Preparación para modelado
### 5.1 Comprobar columnas y objetivo
**Objetivo:** asegurarnos de que el problema está definido y puede entrenarse. **Entrada:** df_limpio y configuración. **Operación:** comprobar columnas, duplicidad de listas, fracciones y target. **Resultado:** predictores_declarados y errores claros si falta resolver algo. **Interpretación:** corrige el dato o la configuración; el código no elige una solución alternativa.

Los nulos de predictores pueden pasar al Pipeline si autorizaste imputar. Los del target no se imputan porque no sabemos la respuesta. Para regresión/series el target debe ser numérico y finito. En clasificación debe haber al menos dos clases con suficientes ejemplos. Si la limpieza cambió nombres o eliminó predictores, actualiza tus listas.

In [ ]:
if TIPO_PROBLEMA not in {"clasificacion", "regresion", "serie_tiempo"}:
    raise ValueError("TIPO_PROBLEMA no reconocido.")
if TARGET not in df_limpio:
    raise ValueError(f"TARGET={TARGET!r} no existe. Edita la configuración.")
if df_limpio[TARGET].isna().any():
    raise ValueError(f"Hay {df_limpio[TARGET].isna().sum()} targets nulos. Decide cómo resolverlos.")
if not (0 < FRACCION_VALIDACION < 1 and 0 < FRACCION_PRUEBA < 1 and FRACCION_VALIDACION + FRACCION_PRUEBA < 1):
    raise ValueError("Las fracciones deben ser positivas y sumar menos de 1.")
if not 0 <= UMBRAL <= 1:
    raise ValueError("UMBRAL debe estar entre 0 y 1.")
if CATEGORIA_DESCONOCIDA not in {"ignore", "error"}:
    raise ValueError("CATEGORIA_DESCONOCIDA debe ser ignore o error.")
predictores_declarados = COLUMNAS_NUMERICAS + COLUMNAS_CATEGORICAS
columnas_declaradas = predictores_declarados + COLUMNAS_ID + COLUMNAS_EXCLUIR
if set(columnas_declaradas) - set(df_limpio.columns):
    raise ValueError(f"Columnas inexistentes: {set(columnas_declaradas) - set(df_limpio.columns)}")
if len(predictores_declarados) != len(set(predictores_declarados)):
    raise ValueError("Un predictor está repetido o declarado como numérico y categórico.")
if set(predictores_declarados) & ({TARGET} | set(COLUMNAS_ID) | set(COLUMNAS_EXCLUIR)):
    raise ValueError("El target, ID y exclusiones no pueden estar entre los predictores.")
if TIPO_PROBLEMA != "serie_tiempo" and not predictores_declarados:
    raise ValueError("Confirma al menos un predictor en las listas de columnas.")
if TIPO_PROBLEMA == "serie_tiempo" and predictores_declarados:
    raise ValueError("Este recorrido temporal usa calendario y rezagos. Deja vacías las listas de predictores.")
if TIPO_PROBLEMA != "clasificacion":
    if not pd.api.types.is_numeric_dtype(df_limpio[TARGET]) or not np.isfinite(df_limpio[TARGET].to_numpy(dtype=float, na_value=np.nan)).all():
        raise ValueError("El target de regresión/serie debe ser numérico y finito. Configura las conversiones.")
if len(df_limpio) < 15:
    raise ValueError("Se requieren al menos 15 filas; según clases y rezagos puedes necesitar muchas más.")

### 5.2 Dividir después del EDA
**Objetivo:** separar aprender, comparar y evaluar. **Entrada:** df_limpio. **Operación:** reservar prueba y luego validación, usando las fracciones de la sección 2. **Resultado:** df_reserva_train, df_reserva_validacion y df_reserva_prueba. **Interpretación:** verifica sus cantidades; no uses prueba para seleccionar candidatos.

Clasificación conserva aproximadamente la proporción de clases mediante estratificación. Regresión usa una separación aleatoria. Series utiliza bloques cronológicos: pasado para entrenamiento y periodos posteriores para validar y probar.

Con FRACCION_PRUEBA=0.20 se apartan 200 de 1.000. Quedan 800; reservar 200 para validación requiere 200/800=0.25 del remanente. Por eso el código usa FRACCION_VALIDACION/(1-FRACCION_PRUEBA). La fracción declarada por ti sigue siendo 0.20 del total, no necesitas escribir 0.25.

La separación aleatoria supone observaciones independientes. El EDA previo ya describió la base; una nueva base externa da una comprobación adicional. Cambiar fracciones/semilla obliga a repetir todas las etapas posteriores.

In [ ]:
if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
    if TIPO_PROBLEMA == "clasificacion":
        conteo_clases_inicial = df_limpio[TARGET].value_counts()
        if len(conteo_clases_inicial) < 2 or conteo_clases_inicial.min() < 5:
            raise ValueError("Clasificación requiere al menos dos clases y cinco filas por clase para este esquema.")
    try:
        df_desarrollo, df_reserva_prueba = train_test_split(
            df_limpio, test_size=FRACCION_PRUEBA, random_state=SEMILLA,
            stratify=df_limpio[TARGET] if TIPO_PROBLEMA == "clasificacion" else None)
        df_reserva_train, df_reserva_validacion = train_test_split(
            df_desarrollo, test_size=FRACCION_VALIDACION / (1 - FRACCION_PRUEBA), random_state=SEMILLA,
            stratify=df_desarrollo[TARGET] if TIPO_PROBLEMA == "clasificacion" else None)
    except ValueError as error_particion:
        raise ValueError("No se pudo dividir. Revisa tamaño, clases y fracciones sin desactivar silenciosamente la estratificación.") from error_particion

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    if FECHA not in df_limpio or FECHA == TARGET:
        raise ValueError("Declara una columna FECHA existente y distinta del target.")
    if not REZAGOS or any(not isinstance(k, int) or k < 1 for k in REZAGOS + VENTANAS):
        raise ValueError("REZAGOS no puede estar vacío y los rezagos/ventanas deben ser enteros positivos.")
    for nombre_entero, valor_entero in {"HORIZONTE": HORIZONTE, "VENTANAS_BACKTEST": VENTANAS_BACKTEST,
            "PASO_BACKTEST": PASO_BACKTEST, "PERIODOS_VALIDACION_SERIE": PERIODOS_VALIDACION_SERIE,
            "PERIODOS_PRUEBA_SERIE": PERIODOS_PRUEBA_SERIE, "PERIODO_ESTACIONAL": PERIODO_ESTACIONAL}.items():
        if valor_entero is not None and (isinstance(valor_entero, bool) or not isinstance(valor_entero, int) or valor_entero < 1):
            raise ValueError(f"{nombre_entero} debe ser un entero positivo.")
        if valor_entero is None and nombre_entero in {"HORIZONTE", "VENTANAS_BACKTEST", "PASO_BACKTEST"}:
            raise ValueError(f"{nombre_entero} debe ser un entero positivo.")
    if FERIADOS and not CALENDARIO_AMPLIADO:
        raise ValueError("Activa CALENDARIO_AMPLIADO para utilizar FERIADOS.")
    calendario_validado = variables_calendario(pd.DatetimeIndex([]), FERIADOS, CALENDARIO_AMPLIADO)
    df_serie_fechas = df_limpio.assign(**{FECHA: pd.to_datetime(df_limpio[FECHA], format=FORMATO_FECHA_SERIE, errors="raise")})
    if df_serie_fechas[FECHA].isna().any():
        raise ValueError("Hay fechas vacías.")
    print("Fechas repetidas:", df_serie_fechas[FECHA].duplicated().sum())
    print("Orden cronológico:", df_serie_fechas[FECHA].is_monotonic_increasing)
    if not df_serie_fechas[FECHA].is_monotonic_increasing and not PERMITIR_ORDENAR_FECHAS:
        raise ValueError("Fechas desordenadas. Revisa y activa PERMITIR_ORDENAR_FECHAS si corresponde.")
    df_serie_ordenada = df_serie_fechas.sort_values(FECHA) if PERMITIR_ORDENAR_FECHAS else df_serie_fechas.copy()
    if AGREGACION_SERIE is None:
        if df_serie_ordenada[FECHA].duplicated().any():
            raise ValueError("Hay varias filas por fecha. Decide una agregación explícita.")
        serie_periodos = df_serie_ordenada.set_index(FECHA)[TARGET].copy()
    elif AGREGACION_SERIE in {"sum", "mean"}:
        agrupacion_periodos = df_serie_ordenada.set_index(FECHA)[TARGET].resample(FRECUENCIA)
        serie_periodos = agrupacion_periodos.sum(min_count=1) if AGREGACION_SERIE == "sum" else agrupacion_periodos.mean()
    else:
        raise ValueError("AGREGACION_SERIE debe ser None, sum o mean.")
    fechas_esperadas = pd.date_range(serie_periodos.index.min(), serie_periodos.index.max(), freq=FRECUENCIA)
    fechas_faltantes = fechas_esperadas.difference(serie_periodos.index)
    fechas_fuera_frecuencia = serie_periodos.index.difference(fechas_esperadas)
    print("Periodos faltantes:", fechas_faltantes.tolist()[:20])
    print("Fechas fuera de frecuencia:", fechas_fuera_frecuencia.tolist()[:20])
    if len(fechas_faltantes) or len(fechas_fuera_frecuencia) or serie_periodos.isna().any():
        raise ValueError("La serie debe tener un valor por periodo sin huecos. Decide cómo corregirla; no se rellenará automáticamente.")
    df_serie_regular = serie_periodos.rename(TARGET).rename_axis(FECHA).reset_index()
    n_prueba_temporal = PERIODOS_PRUEBA_SERIE if PERIODOS_PRUEBA_SERIE is not None else max(1, int(np.ceil(len(df_serie_regular) * FRACCION_PRUEBA)))
    n_validacion_temporal = PERIODOS_VALIDACION_SERIE if PERIODOS_VALIDACION_SERIE is not None else max(1, int(np.ceil(len(df_serie_regular) * FRACCION_VALIDACION)))
    corte_train_temporal = len(df_serie_regular) - n_prueba_temporal - n_validacion_temporal
    corte_validacion_temporal = len(df_serie_regular) - n_prueba_temporal
    memoria_temporal = max(REZAGOS + VENTANAS + ([PERIODO_ESTACIONAL] if PERIODO_ESTACIONAL else []))
    if corte_train_temporal <= memoria_temporal + 5:
        raise ValueError("Historia insuficiente para los rezagos y las particiones elegidas.")
    df_reserva_train = df_serie_regular.iloc[:corte_train_temporal].copy()
    df_reserva_validacion = df_serie_regular.iloc[corte_train_temporal:corte_validacion_temporal].copy()
    df_reserva_prueba = df_serie_regular.iloc[corte_validacion_temporal:].copy()

display(pd.DataFrame({"Filas": [len(df_reserva_train), len(df_reserva_validacion), len(df_reserva_prueba)]},
                     index=["Entrenamiento", "Validación", "Prueba reservada"]))

if TIPO_PROBLEMA == "serie_tiempo":
    duraciones_temporales = pd.DataFrame([{"Partición": etiqueta, "Periodos": len(tabla),
        "Inicio": tabla[FECHA].iloc[0], "Fin": tabla[FECHA].iloc[-1]}
        for etiqueta, tabla in [("Entrenamiento", df_reserva_train), ("Validación", df_reserva_validacion), ("Prueba", df_reserva_prueba)]])
    display(duraciones_temporales)
    print("Horizonte de backtesting/futuro:", HORIZONTE, "periodos; frecuencia:", FRECUENCIA)
    if n_validacion_temporal != HORIZONTE or n_prueba_temporal != HORIZONTE:
        print("Atención: validación, prueba y backtesting no tienen el mismo horizonte; compara sus errores con esa diferencia presente.")

In [ ]:
df_train_limpio = df_reserva_train.copy()
df_validacion_limpia = df_reserva_validacion.copy()
df_prueba_limpia = df_reserva_prueba.copy()

### 5.3 Preparar las entradas con un Pipeline
**Objetivo:** usar exactamente las mismas transformaciones al entrenar y predecir. **Entrada:** particiones ya separadas. **Operación:** seleccionar tus columnas y construir transformaciones aún sin ajustar. **Resultado:** X_train/X_validacion/X_prueba y un preprocesador sin ajustar. **Interpretación:** el Pipeline aprende estadísticas solo cuando se entrena con train.

**Imputar** sustituye faltantes: una mediana aprendida de [10,20,30] vale 20 y se reutiliza para completar ausencias nuevas. No se calcula una mediana diferente sobre la base nueva.

**Escalar** centra cantidades y divide por su desviación. Si media=100 y desviación=20, el valor 140 pasa a 2. No altera la unidad del target.

**One-hot encoding** crea indicadores: pais=Perú puede convertirse en pais_Perú=1 y pais_Chile=0. El codificador conserva las categorías aprendidas. No se vuelve a inventar columnas para cada archivo.

**Pipeline** reúne estos pasos con el modelo. ColumnTransformer aplica operaciones distintas a columnas numéricas y categóricas. Al ajustar dentro de un fold, también se aprenden sus medianas y categorías solo con la parte de entrenamiento de ese fold.

**Fuga de información** ocurre si un paso aprende de datos que debe evaluar. Por eso no se ajustan imputación ni escalado usando toda df_limpio antes de dividir.

La matriz one-hot es densa para facilitar lectura; muchas categorías pueden consumir memoria. Las estrategias son las que elegiste en la sección 2. Si cambias preparación, vuelve a ejecutar desde aquí y todos los modelos.

In [ ]:
if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
    X_train = df_train_limpio.loc[:, predictores_declarados].copy()
    X_validacion = df_validacion_limpia.loc[:, predictores_declarados].copy()
    X_prueba = df_prueba_limpia.loc[:, predictores_declarados].copy()
    for nombre_revision, tabla_revision in [("train", X_train), ("validación", X_validacion), ("prueba", X_prueba)]:
        for columna_numerica in COLUMNAS_NUMERICAS:
            if not pd.api.types.is_numeric_dtype(tabla_revision[columna_numerica]):
                raise TypeError(f"{nombre_revision}: {columna_numerica} debe ser numérica.")
        if np.isinf(tabla_revision[COLUMNAS_NUMERICAS].to_numpy(dtype=float, na_value=np.nan)).any():
            raise ValueError(f"{nombre_revision}: hay infinitos. Define un reemplazo explícito.")
        if not IMPUTAR_NUMERICAS and tabla_revision[COLUMNAS_NUMERICAS].isna().any().any():
            raise ValueError(f"{nombre_revision}: hay nulos numéricos. Decide si habilitas imputación.")
        if not IMPUTAR_CATEGORICAS and tabla_revision[COLUMNAS_CATEGORICAS].isna().any().any():
            raise ValueError(f"{nombre_revision}: hay nulos categóricos. Decide si habilitas imputación.")
        for columna_categorica in COLUMNAS_CATEGORICAS:
            tipos_categoria = {"texto" if isinstance(v, str) else "numero" if isinstance(v, (int, float, np.number, bool)) else "otro"
                for v in tabla_revision[columna_categorica].dropna().unique()}
            if "otro" in tipos_categoria or len(tipos_categoria) > 1:
                raise TypeError(f"{nombre_revision}: {columna_categorica} mezcla tipos incompatibles; declara una regla de texto.")
            tipos_train_categoria = {"texto" if isinstance(v, str) else "numero" if isinstance(v, (int,float,np.number,bool)) else "otro"
                for v in X_train[columna_categorica].dropna().unique()}
            categoria_train_numerica = tipos_train_categoria == {"numero"}
            if tipos_categoria and tipos_train_categoria and categoria_train_numerica != (tipos_categoria == {"numero"}):
                raise TypeError(f"{nombre_revision}: {columna_categorica} cambió entre códigos numéricos y texto.")
    pipeline_numerico = Pipeline([
        ("imputacion", SimpleImputer(strategy=ESTRATEGIA_NUMERICA, fill_value=VALOR_NUMERICO_CONSTANTE, keep_empty_features=True) if IMPUTAR_NUMERICAS else "passthrough"),
        ("escalado", StandardScaler() if ESCALAR else "passthrough")])
    pipeline_categorico = Pipeline([
        ("imputacion", SimpleImputer(strategy=ESTRATEGIA_CATEGORICA, fill_value=VALOR_CATEGORICO_CONSTANTE, keep_empty_features=True) if IMPUTAR_CATEGORICAS else "passthrough"),
        ("codificacion", OneHotEncoder(handle_unknown=CATEGORIA_DESCONOCIDA, sparse_output=False))])
    preprocesador_sin_ajustar = ColumnTransformer([
        ("numericas", pipeline_numerico, COLUMNAS_NUMERICAS),
        ("categoricas", pipeline_categorico, COLUMNAS_CATEGORICAS)], remainder="drop")

    if TIPO_PROBLEMA == "clasificacion":
        codificador_target_sin_ajustar = LabelEncoder()
        codificador_target_ajustado = copy.deepcopy(codificador_target_sin_ajustar).fit(df_train_limpio[TARGET])
        clases_originales = codificador_target_ajustado.classes_
        clases_codificadas = np.arange(len(clases_originales))
        if len(clases_originales) < 2:
            raise ValueError("La limpieza dejó menos de dos clases.")
        if len(clases_originales) == 2 and CLASE_POSITIVA not in clases_originales:
            raise ValueError("CLASE_POSITIVA no aparece en entrenamiento.")
        clases_fuera_train = (set(df_validacion_limpia[TARGET]) | set(df_prueba_limpia[TARGET])) - set(clases_originales)
        if clases_fuera_train:
            raise ValueError(f"La limpieza dejó clases fuera de entrenamiento: {clases_fuera_train}. Revisa la limpieza y partición.")
        indice_positivo = int(codificador_target_ajustado.transform([CLASE_POSITIVA])[0]) if len(clases_originales) == 2 else None
        y_train = pd.Series(codificador_target_ajustado.transform(df_train_limpio[TARGET]), index=X_train.index)
        y_validacion = pd.Series(codificador_target_ajustado.transform(df_validacion_limpia[TARGET]), index=X_validacion.index)
        display(pd.DataFrame({"Etiqueta original": clases_originales, "Código": clases_codificadas}))
    else:
        y_train = df_train_limpio[TARGET].copy()
        y_validacion = df_validacion_limpia[TARGET].copy()

### 5.4 Cómo medir el resultado
**Objetivo:** comparar errores relevantes para tu pregunta. **Entrada:** respuestas reales y predicciones en el mismo orden. **Operación:** calcular indicadores. **Resultado:** diccionarios que luego se convierten en tablas. **Interpretación:** compara validación con la línea base; ningún número aislado demuestra utilidad.

#### Clasificación: etiquetas, probabilidades y matriz de confusión
Una **etiqueta** es la decisión final ("Abandona"). Una **probabilidad** es un valor como 0.70, no una garantía. El umbral convierte la probabilidad en etiqueta para clasificación binaria.

Para el evento positivo: TP son eventos detectados; FN eventos que se escaparon; FP falsas alarmas; TN negativos correctamente descartados. La matriz muestra filas reales y columnas predichas.
- Accuracy: fracción total de aciertos. 90% puede ser poco útil si 90% de la base ya es la clase mayoritaria.
- Precision: entre los anunciados positivos, qué proporción era realmente positiva. 8 TP y 2 FP dan 0.80.
- Recall: entre los positivos reales, cuántos se detectaron. 8 TP y 12 FN dan 0.40.
- F1: combina precision y recall; valores mayores son mejores. No incorpora costos monetarios.
- ROC-AUC: capacidad de ordenar positivos por encima de negativos a través de umbrales. 0.5 equivale a ordenamiento aleatorio; 1 es separación perfecta. No es porcentaje de aciertos ni calibración de probabilidades.
- En multiclase se usa promedio macro: cada clase pesa igual. LazyPredict también muestra promedios ponderados: las clases grandes pesan más.

#### Regresión: cantidades y errores
- MAE: promedio de errores absolutos en la unidad del target. MAE=5 minutos significa error absoluto promedio de 5 minutos.
- RMSE: raíz del promedio de errores cuadrados, también en la unidad del target; penaliza más los errores grandes.
- R²: mejora respecto a predecir el promedio de los reales del conjunto evaluado; 1 es perfecto, 0 equivale a esa referencia y puede ser negativo.
- MAPE_pct: error absoluto relativo en porcentaje; aquí se excluyen reales iguales a cero y se informa cuántos. Cerca de cero puede exagerar el error.

**metricas_regresion(reales, predicciones)** recibe vectores alineados y retorna MAE, RMSE, R2, MAPE_pct y ceros_excluidos_MAPE.
Ejemplo: metricas_regresion([0,2,4], [0,2,4]) → MAE=0, RMSE=0, R2=1, MAPE_pct=0 y un cero excluido. Si todos son cero, MAPE es NaN (no calculable).

**metricas_clasificacion(reales, predicciones, probabilidades, clases, indice_positivo, puntuaciones=None)** recibe códigos de clases y probabilidades ordenadas por clases. indice_positivo identifica el evento binario; puntuaciones permite AUC binaria con decision_function cuando no hay probabilidades. Retorna Accuracy, Precision, Recall, F1 y AUC; AUC es NaN si falta alguna clase o no hay puntuación adecuada.
Ejemplo: metricas_clasificacion([0,1], [0,1], np.array([[.9,.1],[.1,.9]]), [0,1], 1) → todos los indicadores valen 1.

In [ ]:
def metricas_regresion(reales, predicciones):
    """Calcula errores; MAPE usa únicamente targets distintos de cero."""
    vector_real = np.asarray(reales, dtype=float)
    vector_predicho = np.asarray(predicciones, dtype=float)
    casos_no_cero = vector_real != 0
    return {
        "MAE": mean_absolute_error(vector_real, vector_predicho),
        "RMSE": np.sqrt(mean_squared_error(vector_real, vector_predicho)),
        "R2": r2_score(vector_real, vector_predicho) if len(vector_real) > 1 else np.nan,
        "MAPE_pct": np.mean(np.abs((vector_real[casos_no_cero] - vector_predicho[casos_no_cero]) / vector_real[casos_no_cero])) * 100 if casos_no_cero.any() else np.nan,
        "ceros_excluidos_MAPE": int((~casos_no_cero).sum())
    }


def metricas_clasificacion(reales, predicciones, probabilidades, clases, indice_positivo, puntuaciones=None):
    """Métricas de etiquetas; AUC solo cuando hay todas las clases y puntuación válida."""
    caso_binario = len(clases) == 2
    parametros_promedio = {"average": "binary", "pos_label": indice_positivo} if caso_binario else {"average": "macro", "labels": clases}
    auc_calculable = set(np.unique(reales)) == set(clases)
    if auc_calculable and probabilidades is not None:
        valor_auc = roc_auc_score(np.asarray(reales) == indice_positivo, probabilidades[:, list(clases).index(indice_positivo)]) if caso_binario else roc_auc_score(reales, probabilidades, labels=clases, multi_class="ovr", average="macro")
    elif auc_calculable and caso_binario and puntuaciones is not None:
        valor_auc = roc_auc_score(np.asarray(reales) == indice_positivo, puntuaciones)
    else:
        valor_auc = np.nan
    return {"Accuracy": accuracy_score(reales, predicciones),
        "Precision": precision_score(reales, predicciones, zero_division=0, **parametros_promedio),
        "Recall": recall_score(reales, predicciones, zero_division=0, **parametros_promedio),
        "F1": f1_score(reales, predicciones, zero_division=0, **parametros_promedio), "AUC": valor_auc}

### Una función para leer salidas sin cambiar el modelo
**predecir_tabular(modelo, datos, problema, clases=None, umbral=0.5, indice_positivo=None, preparacion=None, es_red=False)** evita repetir conversiones de salida en comparación, prueba y base nueva.
- modelo: Pipeline/estimador ya entrenado, o red Keras entrenada.
- datos: predictores crudos con el mismo esquema de entrenamiento.
- problema: clasificacion o regresion.
- clases: códigos ordenados; obligatorio en clasificación.
- umbral e indice_positivo: decisión binaria; ignorados si no hay probabilidades o si es multiclase.
- preparacion: transformador ya ajustado, solo para la red.
- es_red: True llama a Keras y adapta sigmoid/softmax; False usa el estimador sklearn.
- Devuelve un diccionario con predicciones, probabilidades y puntuaciones. Estos dos últimos son None si no aplican.

No se ejecuta fit ni se calibra un clasificador. Si solo hay decision_function, la puntuación sirve para AUC binaria, no se presenta como probabilidad. La red sigmoid produce probabilidad del código 1; se respeta tu CLASE_POSITIVA aunque tenga código 0.

Ejemplo ejecutable: predecir_tabular(DummyRegressor(strategy="mean").fit([[0],[1]], [10,20]), [[2]], "regresion").
Salida esperada: predicciones=[15.0], probabilidades=None y puntuaciones=None. No modifica el modelo ni la entrada.

In [ ]:
def predecir_tabular(modelo, datos, problema, clases=None, umbral=0.5, indice_positivo=None, preparacion=None, es_red=False):
    """Obtiene salidas de un candidato ajustado, sin aprender transformaciones nuevas."""
    if es_red:
        if preparacion is None:
            raise ValueError("La red requiere su preparación ya ajustada.")
        datos_transformados = preparacion.transform(datos).astype(np.float32)
        salida_red = np.asarray(modelo.predict(datos_transformados, verbose=0))
        if problema == "regresion":
            return {"predicciones": salida_red.ravel(), "probabilidades": None, "puntuaciones": None}
        probabilidades_modelo = np.column_stack([1 - salida_red.ravel(), salida_red.ravel()]) if len(clases) == 2 else salida_red
    elif problema == "regresion":
        return {"predicciones": np.asarray(modelo.predict(datos)), "probabilidades": None, "puntuaciones": None}
    else:
        probabilidades_modelo = np.asarray(modelo.predict_proba(datos)) if hasattr(modelo, "predict_proba") else None

    if probabilidades_modelo is not None:
        etiquetas_modelo = np.where(probabilidades_modelo[:, indice_positivo] >= umbral, indice_positivo, 1 - indice_positivo) if len(clases) == 2 else np.asarray(clases)[np.argmax(probabilidades_modelo, axis=1)]
        return {"predicciones": etiquetas_modelo, "probabilidades": probabilidades_modelo, "puntuaciones": None}
    puntuacion_cruda = np.asarray(modelo.decision_function(datos)) if hasattr(modelo, "decision_function") and len(clases) == 2 else None
    puntuacion_positiva = (-puntuacion_cruda if indice_positivo == 0 else puntuacion_cruda) if puntuacion_cruda is not None else None
    return {"predicciones": np.asarray(modelo.predict(datos)), "probabilidades": None, "puntuaciones": puntuacion_positiva}

## 6. Modelos manuales
### 6.1 Clasificación: predecir una categoría
**Objetivo:** comparar formas de aprender una etiqueta. **Entrada:** X_train/y_train y validación. **Operación:** entrenar cada Pipeline independiente con train. **Resultado:** modelos ajustados y métricas train/validación. **Interpretación:** supera la línea base y revisa los errores que importan.

Base frecuente siempre anuncia la clase más común. Logística combina efectos y regulariza sus coeficientes. Un árbol aplica reglas sucesivas; Random Forest promedia muchos árboles. XGBoost y LightGBM suman correcciones mediante boosting. El árbol sin límite permite observar una complejidad mayor que el árbol podado.

| Hiperparámetro | Qué controla y efecto posible |
|---|---|
| C | 1.0 en logística; un número positivo. Menor C aplica mayor regularización y puede reducir variabilidad a costa de simplificar relaciones. |
| max_iter | 1500 iteraciones máximas de logística. Aumentarlo da más oportunidades de converger; no añade observaciones. Revisa avisos de convergencia. |
| max_depth | 5 en árbol podado, 10 en RF y 4 en XGBoost. Limita niveles; más profundidad permite interacciones complejas y aumenta riesgo de memorizar. None no limita. |
| min_samples_leaf | 5 en árbol y 3 en RF. Exige ejemplos por hoja; aumentarlo suaviza predicciones y puede perder patrones pequeños. |
| n_estimators | 150 árboles iniciales en RF/boosting. Más árboles suelen costar más tiempo; no siempre mejoran validación. |
| learning_rate | 0.05 en boosting, positivo. Reduce la contribución de cada corrección; suele requerir más árboles cuando baja. |
| num_leaves | 15 hojas máximas de LightGBM. Más hojas aumentan complejidad; compáralas con el tamaño de la base. |
| min_child_samples | 20 casos mínimos por hoja de LightGBM. Un valor mayor evita particiones muy pequeñas. |
| subsample | 0.8 en XGBoost: fracción de filas usada por árbol. Menor valor introduce variación y puede regularizar. |
| colsample_bytree | 0.8 en XGBoost: fracción de predictores por árbol; reducirla limita información de cada árbol. |
| random_state | Usa SEMILLA para reproducir partes aleatorias. |
| n_jobs | -1 utiliza los núcleos disponibles; 1 reduce paralelismo y puede facilitar reproducibilidad/consumo en Colab. |
| verbosity | -1 en LightGBM reduce mensajes informativos; no oculta los errores de nuestra validación. |

Los parámetros están en diccionarios junto al modelo. Modificarlos requiere reentrenar ese bloque y comparar de nuevo, antes de mirar prueba. Un hiperparámetro lo eliges tú; un coeficiente aprendido lo estima el entrenamiento.

In [ ]:
if TIPO_PROBLEMA == "clasificacion":
    inicio_manual_clasificacion = time.perf_counter()
    parametros_logistica = dict(C=1.0, max_iter=1500, random_state=SEMILLA)
    parametros_arbol_clasificacion = dict(max_depth=5, min_samples_leaf=5, random_state=SEMILLA)
    parametros_rf_clasificacion = dict(n_estimators=150, max_depth=10, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1)
    parametros_xgb_clasificacion = dict(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEMILLA, n_jobs=-1)
    parametros_lgb_clasificacion = dict(n_estimators=150, learning_rate=0.05, num_leaves=15, min_child_samples=20, random_state=SEMILLA, n_jobs=-1, verbosity=-1)
    estimadores_clasificacion = {
        "Base frecuente": DummyClassifier(strategy="most_frequent"),
        "Regresión logística": LogisticRegression(**parametros_logistica),
        "Árbol sin límite": DecisionTreeClassifier(random_state=SEMILLA),
        "Árbol podado": DecisionTreeClassifier(**parametros_arbol_clasificacion),
        "Random Forest": RandomForestClassifier(**parametros_rf_clasificacion),
        "XGBoost": XGBClassifier(**parametros_xgb_clasificacion),
        "LightGBM": LGBMClassifier(**parametros_lgb_clasificacion)}
    pipelines_clasificacion_sin_ajustar = {nombre: Pipeline([
        ("preparacion", clone(preprocesador_sin_ajustar)), ("modelo", estimador)])
        for nombre, estimador in estimadores_clasificacion.items()}
    modelos_clasificacion_ajustados = {nombre: clone(pipeline).fit(X_train, y_train)
        for nombre, pipeline in pipelines_clasificacion_sin_ajustar.items()}
    duracion_manual_clasificacion = time.perf_counter() - inicio_manual_clasificacion

In [ ]:
if TIPO_PROBLEMA == "clasificacion":
    filas_metricas_clasificacion = []
    for nombre_clasificador, clasificador_ajustado in modelos_clasificacion_ajustados.items():
        for particion_clasificacion, datos_clasificacion, reales_clasificacion in [
            ("Train", X_train, y_train), ("Validación", X_validacion, y_validacion)]:
            probabilidades_clasificacion = clasificador_ajustado.predict_proba(datos_clasificacion)
            etiquetas_clasificacion = np.where(probabilidades_clasificacion[:, indice_positivo] >= UMBRAL,
                indice_positivo, 1 - indice_positivo) if len(clases_originales) == 2 else clasificador_ajustado.predict(datos_clasificacion)
            filas_metricas_clasificacion.append({"Modelo": nombre_clasificador, "Partición": particion_clasificacion,
                **metricas_clasificacion(reales_clasificacion, etiquetas_clasificacion, probabilidades_clasificacion, clases_codificadas, indice_positivo)})
    tabla_metricas_clasificacion = pd.DataFrame(filas_metricas_clasificacion).set_index(["Modelo", "Partición"])
    display(tabla_metricas_clasificacion.round(4))
    brechas_clasificacion = tabla_metricas_clasificacion.xs("Train", level="Partición") - tabla_metricas_clasificacion.xs("Validación", level="Partición")
    display(brechas_clasificacion.add_prefix("Train menos validación: ").round(4))

### 6.2 Regresión: predecir una cantidad
**Objetivo:** estimar números como tiempo, ventas o precio. **Entrada:** predictores y target numérico. **Operación:** entrenar modelos con train y medir errores en validación. **Resultado:** tabla de métricas y modelos ajustados. **Interpretación:** prioriza una métrica en unidades comprensibles para la prueba.

La base predice el promedio de entrenamiento. La regresión lineal combina cantidades con coeficientes aprendidos, sin regularización en esta guía. Árbol, Random Forest, XGBoost y LightGBM capturan relaciones no lineales.

Los diccionarios exponen los mismos parámetros explicados en clasificación (profundidad, hojas, árboles, muestreo y tasa de aprendizaje), con valores iniciales equivalentes. No se usa C en la regresión lineal. Un error RMSE alto respecto a MAE sugiere revisar casos con errores grandes; no demuestra su causa.

Los residuos se calcularán después de tu elección: real menos predicción. Un residuo positivo indica subestimación. Cambiar hiperparámetros exige repetir entrenamiento, comparación y elección.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    inicio_manual_regresion = time.perf_counter()
    parametros_arbol_regresion = dict(max_depth=5, min_samples_leaf=5, random_state=SEMILLA)
    parametros_rf_regresion = dict(n_estimators=150, max_depth=10, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1)
    parametros_xgb_regresion = dict(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEMILLA, n_jobs=-1)
    parametros_lgb_regresion = dict(n_estimators=150, learning_rate=0.05, num_leaves=15, min_child_samples=20, random_state=SEMILLA, n_jobs=-1, verbosity=-1)
    estimadores_regresion = {
        "Base promedio": DummyRegressor(strategy="mean"),
        "Regresión lineal": LinearRegression(),
        "Árbol sin límite": DecisionTreeRegressor(random_state=SEMILLA),
        "Árbol podado": DecisionTreeRegressor(**parametros_arbol_regresion),
        "Random Forest": RandomForestRegressor(**parametros_rf_regresion),
        "XGBoost": XGBRegressor(**parametros_xgb_regresion),
        "LightGBM": LGBMRegressor(**parametros_lgb_regresion)}
    pipelines_regresion_sin_ajustar = {nombre: Pipeline([
        ("preparacion", clone(preprocesador_sin_ajustar)), ("modelo", estimador)])
        for nombre, estimador in estimadores_regresion.items()}
    modelos_regresion_ajustados = {nombre: clone(pipeline).fit(X_train, y_train)
        for nombre, pipeline in pipelines_regresion_sin_ajustar.items()}
    duracion_manual_regresion = time.perf_counter() - inicio_manual_regresion
    filas_metricas_regresion = [{"Modelo": nombre, "Partición": particion,
        **metricas_regresion(reales, modelo.predict(datos))}
        for nombre, modelo in modelos_regresion_ajustados.items()
        for particion, datos, reales in [("Train", X_train, y_train), ("Validación", X_validacion, y_validacion)]]
    tabla_metricas_regresion = pd.DataFrame(filas_metricas_regresion).set_index(["Modelo", "Partición"])
    display(tabla_metricas_regresion.round(4))
    brechas_regresion = tabla_metricas_regresion.xs("Validación", level="Partición") - tabla_metricas_regresion.xs("Train", level="Partición")
    display(brechas_regresion[["MAE", "RMSE"]].add_prefix("Validación menos train: "))

### 6.3 Opcional: explorar hiperparámetros con CV
**Objetivo:** comparar configuraciones dentro del entrenamiento. **Entrada:** un Pipeline sin ajustar y una grilla que tú defines. **Operación:** GridSearchCV prueba todas las combinaciones; RandomizedSearchCV toma una muestra. Cada fold vuelve a aprender preparación y modelo. **Resultado:** puntajes medios y su dispersión, sin sustituir tu elección. **Interpretación:** un promedio mejor con alta variabilidad puede ser inestable.

| Parámetro | Uso |
|---|---|
| EJECUTAR_RANDOM_SEARCH | False omite la búsqueda aleatoria; True la activa. |
| EJECUTAR_GRID_SEARCH | False omite el recorrido exhaustivo; True lo activa. |
| FOLDS_CV | 3 divide entrenamiento en tres partes: cada una se evalúa una vez usando las otras para aprender. Más folds aumentan trabajo y requieren ejemplos suficientes por clase. |
| ITERACIONES_RANDOM | 4 configuraciones muestreadas, hasta el número de combinaciones disponibles. Aumentarlo explora más y tarda más. |
| GRILLA_RF | Diccionario de listas, por ejemplo modelo__max_depth:[4,8]. El prefijo modelo__ indica el paso del Pipeline. El producto de longitudes determina cuántas combinaciones prueba GridSearch. |

f1_macro es la métrica inicial de clasificación; neg_mean_absolute_error es -MAE, de modo que un puntaje mayor significa menos error. refit=False conserva resultados sin elegir un modelo operativo por ti. Copia la configuración que decidas a los diccionarios y vuelve a comparar en validación. Cambiar la grilla solo exige repetir la búsqueda y las etapas que decidas adoptar.

In [ ]:
EJECUTAR_RANDOM_SEARCH = False
EJECUTAR_GRID_SEARCH = False
FOLDS_CV = 3
ITERACIONES_RANDOM = 4
GRILLA_RF = {"modelo__n_estimators": [80, 150], "modelo__max_depth": [4, 8], "modelo__min_samples_leaf": [3, 6]}

if TIPO_PROBLEMA in {"clasificacion", "regresion"} and (EJECUTAR_RANDOM_SEARCH or EJECUTAR_GRID_SEARCH):
    if TIPO_PROBLEMA == "clasificacion":
        if y_train.value_counts().min() < FOLDS_CV:
            raise ValueError("No hay suficientes filas por clase para FOLDS_CV.")
        pipeline_busqueda = pipelines_clasificacion_sin_ajustar["Random Forest"]
        cv_busqueda = StratifiedKFold(n_splits=FOLDS_CV, shuffle=True, random_state=SEMILLA)
        metrica_busqueda = "f1_macro"
    else:
        pipeline_busqueda = pipelines_regresion_sin_ajustar["Random Forest"]
        cv_busqueda = KFold(n_splits=FOLDS_CV, shuffle=True, random_state=SEMILLA)
        metrica_busqueda = "neg_mean_absolute_error"  # Mayor = menor MAE
    if EJECUTAR_RANDOM_SEARCH:
        busqueda_random_sin_ajustar = RandomizedSearchCV(clone(pipeline_busqueda), GRILLA_RF,
            n_iter=ITERACIONES_RANDOM, scoring=metrica_busqueda, cv=cv_busqueda, random_state=SEMILLA, n_jobs=-1, refit=False, error_score="raise")
        busqueda_random_ajustada = copy.deepcopy(busqueda_random_sin_ajustar).fit(X_train, y_train)
        resultados_random = pd.DataFrame(busqueda_random_ajustada.cv_results_)
        display(resultados_random[["params", "mean_test_score", "std_test_score", "rank_test_score"]])
    if EJECUTAR_GRID_SEARCH:
        busqueda_grid_sin_ajustar = GridSearchCV(clone(pipeline_busqueda), GRILLA_RF,
            scoring=metrica_busqueda, cv=cv_busqueda, n_jobs=-1, refit=False, error_score="raise")
        busqueda_grid_ajustada = copy.deepcopy(busqueda_grid_sin_ajustar).fit(X_train, y_train)
        resultados_grid = pd.DataFrame(busqueda_grid_ajustada.cv_results_)
        display(resultados_grid[["params", "mean_test_score", "std_test_score", "rank_test_score"]])

### 6.4 Series de tiempo: aprender del pasado para pronosticar
**Objetivo:** predecir periodos posteriores con una sola serie regular. **Entrada:** fecha, target y configuración temporal. **Operación:** construir calendario y memoria histórica. **Resultado:** tabla de predictores y funciones de pronóstico. **Interpretación:** una fila del futuro no puede utilizar la respuesta real de otra fila futura.

Un rezago es un valor anterior: lag_1 del martes usa el lunes. Una media móvil de ventana 7 promedia los siete periodos anteriores. Se aplica shift(1) para no incluir el target que se intenta predecir. El pronóstico recursivo usa la predicción de mañana como parte de la historia de pasado mañana; los errores pueden acumularse.

**construir_rezagos(historia, rezagos, ventanas)** recibe una Series numérica con DatetimeIndex regular y ordenado, listas de rezagos y ventanas. Devuelve otra tabla con anio/mes/dia/dia_semana, lag_k, media_k y objetivo.
Ejemplo: construir_rezagos(pd.Series([10,20,30], index=pd.date_range("2025-01-01", periods=3)), [1], [2]).iloc[-1].
Salida esperada: lag_1=20, media_2=15 y objetivo=30 en la tercera fila.

**pronostico_recursivo(modelo, historia, fechas, rezagos, ventanas)** recibe un regresor ajustado o None para persistencia, historia conocida antes del horizonte, fechas futuras consecutivas con frecuencia explícita y las mismas listas usadas al entrenar. Devuelve una Series de predicciones; no cambia historia.
Ejemplo: pronostico_recursivo(None, pd.Series([10,20,30], index=pd.date_range("2025-01-01", periods=3)), pd.date_range("2025-01-04", periods=2, freq="D"), [1], [2]).tolist().
Salida esperada: [30.0,30.0], porque persistencia repite el último valor.

No se rellenan huecos ni se recortan predicciones negativas automáticamente. Si el target no puede ser negativo, valora cómo modelarlo y medirlo; no se oculta el error mediante un recorte.

Con CALENDARIO_AMPLIADO=True, ambas funciones llaman a variables_calendario con los mismos feriados; no se mantienen dos fórmulas diferentes de calendario. La semana ISO no es la semana de calendario que siempre comienza el 1 de enero.

Para la base estacional, pronostico_recursivo(None, historia, fechas, REZAGOS, VENTANAS, periodo_estacional=7) repite el último ciclo de siete periodos sin consultar respuestas futuras. Por ejemplo, [10,20,30] con periodo 2 produce [20,30,20,30]. No aprende efectos nuevos de feriados.

### 6.5 Entrenamiento y validación temporal
**Objetivo:** medir qué ocurre al pronosticar una ventana completa. **Entrada:** historia de entrenamiento y fechas de validación. **Operación:** entrenar RF/XGBoost con rezagos históricos y pronosticar recursivamente. **Resultado:** métricas y curvas de reales/predicciones. **Interpretación:** compara contra persistencia, que repite el último valor.

Solo se omiten las primeras filas que todavía no tienen suficiente historia para construir memoria; se informa cuántas. Train se evalúa a un paso con rezagos reales, validación de manera recursiva: la brecha mezcla generalización y acumulación de error, no demuestra solo sobreajuste.

parametros_rf_temporal y parametros_xgb_temporal exponen n_estimators=150, max_depth=8/4, min_samples_leaf=3 para RF, learning_rate=0.05 para XGBoost, random_state y n_jobs. Tienen el significado de árboles/profundidad/regularización explicado en 6.1. Modificarlos exige repetir entrenamiento y backtesting.

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    historia_train = df_train_limpio.set_index(FECHA)[TARGET].copy()
    tabla_temporal_completa = construir_rezagos(historia_train, REZAGOS, VENTANAS, FERIADOS, CALENDARIO_AMPLIADO)
    tabla_temporal_entrenable = tabla_temporal_completa.dropna()
    print("Filas iniciales sin memoria suficiente:", len(tabla_temporal_completa) - len(tabla_temporal_entrenable))
    X_train_temporal = tabla_temporal_entrenable.drop(columns="objetivo")
    y_train_temporal = tabla_temporal_entrenable["objetivo"].copy()
    parametros_rf_temporal = dict(n_estimators=150, max_depth=8, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1)
    parametros_xgb_temporal = dict(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=SEMILLA, n_jobs=-1)
    modelos_temporales_sin_ajustar = {"Random Forest": RandomForestRegressor(**parametros_rf_temporal),
        "XGBoost": XGBRegressor(**parametros_xgb_temporal)}
    modelos_temporales_ajustados = {nombre: clone(modelo).fit(X_train_temporal, y_train_temporal)
        for nombre, modelo in modelos_temporales_sin_ajustar.items()}
    fechas_validacion_temporal = pd.date_range(df_validacion_limpia[FECHA].iloc[0], periods=len(df_validacion_limpia), freq=FRECUENCIA)
    bases_temporales = {"Persistencia": None, **({"Estacional": None} if PERIODO_ESTACIONAL else {})}
    predicciones_validacion_temporal = {nombre: pronostico_recursivo(modelo, historia_train, fechas_validacion_temporal,
        REZAGOS, VENTANAS, FERIADOS, CALENDARIO_AMPLIADO, PERIODO_ESTACIONAL if nombre == "Estacional" else None)
        for nombre, modelo in {**bases_temporales, **modelos_temporales_ajustados}.items()}
    filas_metricas_temporales = [
        {"Modelo": nombre, "Partición": "Validación recursiva", **metricas_regresion(df_validacion_limpia[TARGET], prediccion)}
        for nombre, prediccion in predicciones_validacion_temporal.items()]
    filas_metricas_train_temporal = [{"Modelo": nombre, "Partición": "Train un paso",
        **metricas_regresion(y_train_temporal, modelo.predict(X_train_temporal))}
        for nombre, modelo in modelos_temporales_ajustados.items()]
    tabla_metricas_temporales = pd.DataFrame(filas_metricas_train_temporal + filas_metricas_temporales).set_index(["Modelo", "Partición"])
    display(tabla_metricas_temporales)
    comparacion_temporal = pd.DataFrame({"Real": df_validacion_limpia[TARGET].to_numpy(),
        **{nombre: pred.to_numpy() for nombre, pred in predicciones_validacion_temporal.items()}}, index=fechas_validacion_temporal)
    comparacion_temporal.plot(figsize=(12, 4), title="Validación: horizonte completo sin targets futuros")
    plt.show()

### 6.6 Backtesting y elección temporal
**Objetivo:** comprobar si el resultado depende de un único momento histórico. **Entrada:** entrenamiento temporal. **Operación:** situarse en varios orígenes del pasado, aprender solo con historia previa y pronosticar HORIZONTE periodos en cada uno. **Resultado:** errores por ventana y promedios. **Interpretación:** revisa estabilidad y periodos difíciles antes de elegir.

VENTANAS_BACKTEST, PASO_BACKTEST y HORIZONTE se explican en configuración. Por ejemplo horizonte 14 y paso 7 generan ventanas solapadas. Los orígenes terminan dentro de entrenamiento; no se consumen valores de prueba.

MODELO_TEMPORAL_ELEGIDO empieza en Random Forest como ejemplo explícito, no por un ranking. Puedes escribir XGBoost o Persistencia. Si no alcanza historia, reduce exigencias conscientemente o aporta más datos. Cambiar la elección exige repetir evaluación y guardado temporal.

Si activas PERIODO_ESTACIONAL, compara también Estacional y puedes escribir ese nombre en MODELO_TEMPORAL_ELEGIDO. La memoria mínima debe alcanzar el ciclo elegido en todos los orígenes. Las bases Persistencia y Estacional no tienen importancias de árboles ni SHAP.

In [ ]:
MODELO_TEMPORAL_ELEGIDO = "Random Forest"  # También "XGBoost", "Persistencia" o "Estacional" si la activaste

if TIPO_PROBLEMA == "serie_tiempo":
    origenes_backtest = [len(historia_train) - HORIZONTE - PASO_BACKTEST * k for k in reversed(range(VENTANAS_BACKTEST))]
    if min(origenes_backtest) <= memoria_temporal + 5:
        raise ValueError("Reduce ventanas/horizonte/paso de backtesting o aporta más historia.")
    filas_backtest = []
    errores_backtest = []
    for numero_ventana, origen_ventana in enumerate(origenes_backtest, start=1):
        historia_ventana = historia_train.iloc[:origen_ventana].copy()
        tabla_ventana_completa = construir_rezagos(historia_ventana, REZAGOS, VENTANAS, FERIADOS, CALENDARIO_AMPLIADO)
        tabla_ventana_entrenable = tabla_ventana_completa.dropna()
        fechas_ventana = pd.date_range(historia_train.index[origen_ventana], periods=HORIZONTE, freq=FRECUENCIA)
        reales_ventana = historia_train.loc[fechas_ventana]
        for nombre_backtest, modelo_backtest in {**bases_temporales, **modelos_temporales_sin_ajustar}.items():
            modelo_ventana_ajustado = clone(modelo_backtest).fit(tabla_ventana_entrenable.drop(columns="objetivo"), tabla_ventana_entrenable["objetivo"]) if modelo_backtest is not None else None
            prediccion_ventana = pronostico_recursivo(modelo_ventana_ajustado, historia_ventana, fechas_ventana,
                REZAGOS, VENTANAS, FERIADOS, CALENDARIO_AMPLIADO, PERIODO_ESTACIONAL if nombre_backtest == "Estacional" else None)
            errores_backtest.append(errores_temporales(reales_ventana, prediccion_ventana).assign(Modelo=nombre_backtest, Ventana=numero_ventana))
            filas_backtest.append({"Ventana": numero_ventana, "Modelo": nombre_backtest, "Inicio": fechas_ventana[0],
                **metricas_regresion(reales_ventana, prediccion_ventana)})
    tabla_backtest = pd.DataFrame(filas_backtest)
    display(tabla_backtest)
    display(tabla_backtest.groupby("Modelo")[["MAE", "RMSE", "MAPE_pct"]].mean())
    tabla_backtest.pivot(index="Ventana", columns="Modelo", values="MAE").plot.bar(title="MAE por ventana de backtesting")
    plt.show()
    if MODELO_TEMPORAL_ELEGIDO not in {*bases_temporales, *modelos_temporales_ajustados}:
        raise ValueError("Elige un modelo temporal de la comparación.")

### 6.7 ¿Cuándo y a qué distancia se equivoca?
Los errores por día de semana ayudan a detectar patrones de calendario. El error por paso muestra cómo cambia el desempeño al alejarse del origen: no tiene por qué aumentar de forma monótona. Compara todos los modelos sobre las mismas ventanas.

El backtesting resume varios orígenes para cada paso. Con ventanas solapadas, una misma fecha puede aparecer en varios pronósticos; el conteo representa pronósticos, no días independientes. Un solo error de validación por paso no estima por sí solo su variabilidad. Usa estos resultados antes de elegir; no vuelvas a seleccionar usando prueba.

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    reales_validacion_temporal = pd.Series(df_validacion_limpia[TARGET].to_numpy(), index=fechas_validacion_temporal)
    errores_validacion_temporal = pd.concat([errores_temporales(reales_validacion_temporal, pred).assign(Modelo=nombre)
        for nombre, pred in predicciones_validacion_temporal.items()])
    diagnostico_backtest = pd.concat(errores_backtest)
    error_por_dia_temporal = errores_validacion_temporal.groupby(["Modelo", "dia_semana"]).agg(
        pronosticos=("error_absoluto", "size"), MAE=("error_absoluto", "mean"), sesgo_real_menos_predicho=("residuo", "mean"))
    error_por_paso_temporal = diagnostico_backtest.groupby(["Modelo", "paso"]).agg(
        pronosticos=("error_absoluto", "size"), MAE=("error_absoluto", "mean"), sesgo_real_menos_predicho=("residuo", "mean"))
    display(error_por_dia_temporal)
    display(error_por_paso_temporal)
    error_por_paso_temporal["MAE"].unstack("Modelo").plot(figsize=(12, 4), title="Backtesting: MAE por distancia desde el origen")
    plt.show()

## 7. Comparación ampliada: qué automatiza cada opción
**Objetivo:** explorar más familias y comparar contra los modelos manuales. **Entrada:** únicamente entrenamiento y validación. **Operación:** LazyPredict prueba modelos por defecto; FLAML busca familia e hiperparámetros; la red aprende una arquitectura que tú defines. **Resultado:** candidatos, tiempos e indicadores. **Interpretación:** tú decides qué fuente adoptar; ninguna limpieza se activa por estos resultados.

Las banderas están en la sección 1. Estos bloques se omiten para series porque su validación requiere el backtesting temporal, no una CV aleatoria. No se cargan las bases clínicas ni las imágenes de la referencia.

No confundas CV, validación reservada y prueba: FLAML optimiza CV dentro de train; la validación reservada compara su candidato con las demás fuentes; prueba solo mide la decisión final. Usar repetidamente validación también puede ajustar demasiado tus decisiones a ella.

### 7.1 LazyPredict: comparar muchos modelos por defecto
**Objetivo:** identificar familias que merecen revisión. **Entrada:** X_train/y_train y X_validacion/y_validacion. **Operación:** ajustar nuestra preparación solo con train y entregarle entradas numéricas a LazyPredict. **Resultado:** ranking original, modelos recuperados, tiempos y tabla común train/validación. **Interpretación:** el orden no es una orden de usar el primero.

MODELOS_LAZY="all" prueba los candidatos disponibles; también admite una lista de nombres, por ejemplo ["LogisticRegression","RandomForestClassifier"]. Una lista más corta controla tiempo. Nombres no encontrados detienen la ejecución para que los corrijas. Los modelos que fallan se muestran, sin sustituirlos.

LazyPredict incorpora su propio preprocesador (imputación/escalado sobre la matriz ya preparada). La preparación externa devuelve un DataFrame con nombres de columnas, porque el preprocesador recuperado necesita esos nombres. Se conserva en el Pipeline recuperado; las entradas sin nulos ya fueron verificadas o imputadas según tus reglas. Este paso adicional puede hacer que resultados difieran del recorrido manual.

F1/Precision/Recall del ranking de LazyPredict son ponderados; nuestra tabla común usa clase positiva en binaria y macro en multiclase. En modelos sin predict_proba el umbral no aplica; decision_function no es una probabilidad. Los clasificadores solo de etiquetas pueden participar, pero AUC puede no calcularse.

La función fit de LazyPredict llama X_test a su segundo argumento: aquí le pasamos **validación**, nunca X_prueba. Cambiar candidatos exige repetir este bloque, elección y etapas posteriores.

In [ ]:
MODELOS_LAZY = "all"

if TIPO_PROBLEMA in {"clasificacion", "regresion"} and EJECUTAR_LAZY:
    from lazypredict.Supervised import LazyClassifier, LazyRegressor
    from sklearn.utils import all_estimators
    catalogo_lazy = {**dict(all_estimators(type_filter="classifier" if TIPO_PROBLEMA == "clasificacion" else "regressor")),
        **({"XGBClassifier": XGBClassifier, "LGBMClassifier": LGBMClassifier} if TIPO_PROBLEMA == "clasificacion" else {"XGBRegressor": XGBRegressor, "LGBMRegressor": LGBMRegressor})}
    if MODELOS_LAZY != "all":
        if not isinstance(MODELOS_LAZY, list) or not MODELOS_LAZY:
            raise ValueError("MODELOS_LAZY debe ser all o una lista no vacía de nombres.")
        if set(MODELOS_LAZY) - set(catalogo_lazy):
            raise ValueError(f"Nombres Lazy no encontrados: {set(MODELOS_LAZY) - set(catalogo_lazy)}")
    candidatos_lazy = "all" if MODELOS_LAZY == "all" else [catalogo_lazy[n] for n in MODELOS_LAZY]
    preparacion_lazy_sin_ajustar = clone(preprocesador_sin_ajustar).set_output(transform="pandas")
    preparacion_lazy_ajustada = clone(preparacion_lazy_sin_ajustar).fit(X_train, y_train)
    X_train_lazy = pd.DataFrame(preparacion_lazy_ajustada.transform(X_train), index=X_train.index)
    X_validacion_lazy = pd.DataFrame(preparacion_lazy_ajustada.transform(X_validacion), index=X_validacion.index)
    lazy_sin_ajustar = LazyClassifier(classifiers=candidatos_lazy, verbose=1, ignore_warnings=False, random_state=SEMILLA) if TIPO_PROBLEMA == "clasificacion" else LazyRegressor(regressors=candidatos_lazy, verbose=1, ignore_warnings=False, random_state=SEMILLA)
    lazy_ajustado = copy.deepcopy(lazy_sin_ajustar)
    inicio_lazy = time.perf_counter()
    tabla_lazy_original, predicciones_lazy_originales = lazy_ajustado.fit(X_train_lazy, X_validacion_lazy, y_train, y_validacion)
    duracion_lazy = time.perf_counter() - inicio_lazy
    modelos_lazy_recuperados = lazy_ajustado.provide_models(X_train_lazy, X_validacion_lazy, y_train, y_validacion)
    modelos_lazy_ajustados = {nombre: Pipeline([("preparacion", copy.deepcopy(preparacion_lazy_ajustada)),
        ("modelo", copy.deepcopy(modelo))]) for nombre, modelo in modelos_lazy_recuperados.items()}
    print("Candidatos ajustados:", len(modelos_lazy_ajustados), "Tiempo total (s):", round(duracion_lazy, 2))
    display(tabla_lazy_original)
    if MODELOS_LAZY != "all":
        print("Candidatos sin resultado:", sorted(set(MODELOS_LAZY) - set(modelos_lazy_ajustados)))
    print("Detalles de fallos:", getattr(lazy_ajustado, "failed_models_", getattr(lazy_ajustado, "failed_models", "Consulta los mensajes emitidos por cada candidato.")))

### 7.2 FLAML con preparación dentro de cada partición
FLAML elige el algoritmo y sus hiperparámetros, como en la referencia. Para no aprender medianas/categorías de todo train antes de CV, necesitamos **un adaptador técnico**, no otra función que decida limpiezas.

**crear_learner_pipeline(clase_base, preparador)**
- clase_base: clase FLAML de una familia, por ejemplo RandomForestEstimator.
- preparador: nuestro ColumnTransformer sin ajustar.
- Retorna una clase registrable en AutoML que conserva el espacio de búsqueda de la familia y ajusta un Pipeline por fold.
- Ejemplo dentro del bloque activado: ClaseRF = crear_learner_pipeline(RandomForestEstimator, preprocesador_sin_ajustar); candidato = ClaseRF(task="classification", n_estimators=10); candidato.fit(X_train,y_train).
- Salida esperada: fit devuelve segundos; candidato.estimator es un Pipeline ajustado que recibe predictores crudos.

El adaptador define dos métodos:
- **ConPreparacion._preprocess(self, X)** recibe los predictores y los devuelve sin cambiarlos. Evita una codificación adicional de FLAML antes del Pipeline. Ejemplo: candidato._preprocess(X_train) devuelve X_train sin modificarlo.
- **ConPreparacion.fit(self, X_train, y_train, budget=None, free_mem_ratio=0, **kwargs)** recibe el fold de entrenamiento. Construye un Pipeline nuevo, lo ajusta, lo conserva en el atributo requerido por FLAML y devuelve tiempo transcurrido. budget/free_mem_ratio se aceptan por contrato; esta versión no interrumpe un fit a mitad. kwargs solo admite sample_weight; is_retrain es una señal interna sin efecto en el modelo.
  Ejemplo: candidato.fit(X_train,y_train) devuelve un float >=0 y deja listo candidato.estimator.predict(X_validacion).

El presupuesto global puede excederse por un entrenamiento que ya comenzó. No se modifica ningún DataFrame; la asignación interna al atributo del estimador es el estado normal de entrenamiento de una librería.

In [ ]:
if TIPO_PROBLEMA in {"clasificacion", "regresion"} and EJECUTAR_FLAML:
    from flaml import AutoML, tune
    from flaml.automl.model import LGBMEstimator, XGBoostSklearnEstimator, RandomForestEstimator, ExtraTreesEstimator

    def crear_learner_pipeline(clase_base, preparador):
        """Crea un learner FLAML que ajusta la preparación dentro de cada fold."""
        class ConPreparacion(clase_base):
            def _preprocess(self, X):
                """Conserva entradas crudas: la preparación pertenece al Pipeline."""
                return X

            def fit(self, X_train, y_train, budget=None, free_mem_ratio=0, **kwargs):
                """Ajusta un Pipeline nuevo con el fold recibido y devuelve segundos."""
                inicio_ajuste = time.perf_counter()
                argumentos_no_admitidos = set(kwargs) - {"sample_weight", "is_retrain"}
                if argumentos_no_admitidos:
                    raise ValueError(f"Argumentos FLAML no soportados: {argumentos_no_admitidos}")
                estimador_fold = self.estimator_class(**self.params)
                pipeline_fold_sin_ajustar = Pipeline([("preparacion", clone(preparador)), ("modelo", estimador_fold)])
                parametros_fit = {"modelo__sample_weight": kwargs["sample_weight"]} if kwargs.get("sample_weight") is not None else {}
                pipeline_fold_ajustado = clone(pipeline_fold_sin_ajustar).fit(X_train, y_train, **parametros_fit)
                self._model = pipeline_fold_ajustado
                return time.perf_counter() - inicio_ajuste
        return ConPreparacion

    clases_flaml_base = {"lgbm": LGBMEstimator, "xgboost": XGBoostSklearnEstimator,
        "rf": RandomForestEstimator, "extra_tree": ExtraTreesEstimator}
    clases_flaml_pipeline = {n: crear_learner_pipeline(c, preprocesador_sin_ajustar) for n, c in clases_flaml_base.items()}

### 7.3 Buscar, registrar y comparar
**Objetivo:** buscar configuraciones bajo un presupuesto definido por ti. **Entrada:** train crudo y el adaptador. **Operación:** CV estratificada en clasificación o KFold en regresión. **Resultado:** mejor configuración por familia, historial y candidatos reajustados solo con train. **Interpretación:** la validación reservada permitirá compararlos con las otras fuentes.

| Parámetro | Significado y efecto |
|---|---|
| TIEMPO_FLAML | 120 segundos iniciales de búsqueda. Más tiempo permite probar más; no garantiza mejor evaluación externa. Un fit ya iniciado puede excederlo. |
| FOLDS_FLAML | 5 particiones de CV dentro de train. Cada fold requiere ejemplos de todas las clases. Aumentarlo cuesta más. |
| ESTIMADORES_FLAML | ["lgbm","xgboost","rf","extra_tree"]. Quitar familias reduce la búsqueda. Deben pertenecer a este catálogo adaptado. |
| METRICA_FLAML | auto usa roc_auc en binaria, roc_auc_ovr en multiclase y r2 en regresión. También puedes elegir macro_f1, accuracy, mae o rmse cuando corresponda. |
| MAX_ITER_FLAML | None deja limitar por tiempo; un entero positivo limita también la cantidad de configuraciones. |
| MAX_ARBOLES_FLAML | 300 limita los árboles de cada candidato. Un valor mayor permite más complejidad y costo; el límite reduce el riesgo de que un solo ajuste consuma toda la prueba. Debe ser entero de al menos 4. |
| RUTA_LOG_FLAML | /content/flaml_busqueda.log registra intentos. Se inicia un historial nuevo al ejecutar; cambia nombre para conservar otra corrida. |

FLAML minimiza pérdida. Para AUC, F1, accuracy y R², métrica=1-pérdida. Para MAE/RMSE, pérdida ya es el error y menor es mejor. No se usa 1-pérdida indiscriminadamente. sampling y aumento automático de clases se desactivan; el objetivo es hacer visible el conjunto utilizado.

In [ ]:
TIEMPO_FLAML = 120
FOLDS_FLAML = 5
ESTIMADORES_FLAML = ["lgbm", "xgboost", "rf", "extra_tree"]
METRICA_FLAML = "auto"
MAX_ITER_FLAML = None
MAX_ARBOLES_FLAML = 300
RUTA_LOG_FLAML = "/content/flaml_busqueda.log"

if TIPO_PROBLEMA in {"clasificacion", "regresion"} and EJECUTAR_FLAML:
    if not ESTIMADORES_FLAML or set(ESTIMADORES_FLAML) - set(clases_flaml_pipeline):
        raise ValueError("Elige familias del catálogo lgbm, xgboost, rf, extra_tree.")
    if FOLDS_FLAML < 2 or len(y_train) < FOLDS_FLAML or TIEMPO_FLAML <= 0:
        raise ValueError("Revisa folds, cantidad de filas y presupuesto positivo.")
    if TIPO_PROBLEMA == "clasificacion" and y_train.value_counts().min() < FOLDS_FLAML:
        raise ValueError("No hay ejemplos suficientes por clase para FOLDS_FLAML.")
    tarea_flaml = "classification" if TIPO_PROBLEMA == "clasificacion" else "regression"
    metrica_flaml_elegida = ("roc_auc" if len(clases_originales) == 2 else "roc_auc_ovr") if TIPO_PROBLEMA == "clasificacion" and METRICA_FLAML == "auto" else "r2" if METRICA_FLAML == "auto" else METRICA_FLAML
    metricas_flaml_admitidas = {"roc_auc", "roc_auc_ovr", "macro_f1", "accuracy"} if TIPO_PROBLEMA == "clasificacion" else {"r2", "mae", "rmse"}
    if metrica_flaml_elegida not in metricas_flaml_admitidas:
        raise ValueError(f"Métrica no admitida para este problema: {metrica_flaml_elegida}")
    if metrica_flaml_elegida == "roc_auc" and len(clases_originales) != 2:
        raise ValueError("roc_auc corresponde a binaria; usa roc_auc_ovr para multiclase.")
    if not isinstance(MAX_ARBOLES_FLAML, int) or MAX_ARBOLES_FLAML < 4:
        raise ValueError("MAX_ARBOLES_FLAML debe ser un entero de al menos 4.")
    espacios_flaml = {n + "_pipeline": {"n_estimators": {"domain": tune.randint(4, MAX_ARBOLES_FLAML + 1),
        "init_value": 4, "low_cost_init_value": 4}} for n in ESTIMADORES_FLAML}
    cv_flaml = StratifiedKFold(n_splits=FOLDS_FLAML, shuffle=True, random_state=SEMILLA) if TIPO_PROBLEMA == "clasificacion" else KFold(n_splits=FOLDS_FLAML, shuffle=True, random_state=SEMILLA)
    automl_sin_ajustar = AutoML()
    for familia_registro, clase_registro in clases_flaml_pipeline.items():
        automl_sin_ajustar.add_learner(learner_name=familia_registro + "_pipeline", learner_class=clase_registro)
    automl_ajustado = copy.deepcopy(automl_sin_ajustar)
    Path(RUTA_LOG_FLAML).parent.mkdir(parents=True, exist_ok=True)
    inicio_flaml = time.perf_counter()
    automl_ajustado.fit(X_train=X_train, y_train=y_train, task=tarea_flaml,
        metric=metrica_flaml_elegida, time_budget=TIEMPO_FLAML, max_iter=MAX_ITER_FLAML,
        estimator_list=[n + "_pipeline" for n in ESTIMADORES_FLAML],
        eval_method="cv", split_type=cv_flaml, n_splits=FOLDS_FLAML,
        skip_transform=True, auto_augment=False, sample=False, ensemble=False, custom_hp=espacios_flaml,
        log_file_name=RUTA_LOG_FLAML, log_type="all", append_log=False,
        seed=SEMILLA, n_jobs=1, verbose=0, mlflow_logging=False)
    duracion_flaml = time.perf_counter() - inicio_flaml
    if not automl_ajustado.best_config:
        raise RuntimeError("FLAML no produjo un candidato. Revisa mensajes o aumenta el presupuesto.")
    modelos_flaml_ajustados = {}
    for nombre_familia, configuracion_familia in automl_ajustado.best_config_per_estimator.items():
        if configuracion_familia is not None:
            familia_original = nombre_familia.removesuffix("_pipeline")
            configuracion_modelo_familia = {k: v for k, v in configuracion_familia.items() if k != "FLAML_sample_size"}
            candidato_flaml_sin_ajustar = clases_flaml_pipeline[familia_original](task=tarea_flaml, **configuracion_modelo_familia)
            candidato_flaml_ajustado = copy.deepcopy(candidato_flaml_sin_ajustar)
            candidato_flaml_ajustado.fit(X_train, y_train)
            modelos_flaml_ajustados[familia_original] = candidato_flaml_ajustado.estimator
    mejor_familia_flaml = automl_ajustado.best_estimator.removesuffix("_pipeline")
    historial_flaml_crudo = pd.read_json(RUTA_LOG_FLAML, lines=True)
    historial_flaml = historial_flaml_crudo.dropna(subset=["learner"]).copy()
    metrica_es_error_flaml = metrica_flaml_elegida in {"mae", "rmse"}
    historial_flaml_metricas = historial_flaml.assign(metrica_cv=historial_flaml["validation_loss"] if metrica_es_error_flaml else 1 - historial_flaml["validation_loss"])
    resumen_flaml = historial_flaml_metricas.groupby("learner").agg(intentos=("metrica_cv", "size"),
        mejor=("metrica_cv", "min" if metrica_es_error_flaml else "max"), mediana=("metrica_cv", "median"))
    print("Mejor familia por CV:", mejor_familia_flaml, "Tiempo (s):", round(duracion_flaml, 2))
    print("Configuración elegida por FLAML:", automl_ajustado.best_config)
    display(resumen_flaml)
    display(historial_flaml_metricas[["learner", "metrica_cv", "trial_time", "wall_clock_time", "config"]])
    for familia_grafico, datos_grafico in historial_flaml_metricas.groupby("learner"):
        plt.scatter(datos_grafico["wall_clock_time"], datos_grafico["metrica_cv"], label=familia_grafico, s=20)
    curva_mejor_flaml = historial_flaml_metricas["metrica_cv"].cummin() if metrica_es_error_flaml else historial_flaml_metricas["metrica_cv"].cummax()
    plt.step(historial_flaml_metricas["wall_clock_time"], curva_mejor_flaml, where="post", color="black", label="Mejor hasta el momento")
    plt.xlabel("Segundos de búsqueda")
    plt.ylabel(metrica_flaml_elegida + " en CV")
    plt.legend(fontsize=8)
    plt.show()

### 7.4 Red neuronal tabular
**Objetivo:** comparar una red densa con árboles y modelos lineales usando la misma información. **Entrada:** train preparado solo con train y validación transformada con ese preparador. **Operación:** aprender pesos mediante lotes y detenerse según val_loss. **Resultado:** red entrenada, historial y tiempo. **Interpretación:** compara métricas y costo; una arquitectura más grande no garantiza superar árboles en tablas.

Cada neurona combina entradas y aplica una transformación; las capas permiten relaciones no lineales. Dropout desactiva temporalmente parte de las unidades mientras aprende, para regularizar. En predicción no se aplica ese descarte.

| Parámetro | Qué controla |
|---|---|
| NEURONAS_RED | [32,16] define dos capas ocultas. Agregar capas o unidades aumenta capacidad, costo y riesgo de memorizar. |
| DROPOUT_RED | 0.20 descarta 20% de unidades durante entrenamiento. Debe estar entre 0 y 1, sin incluir 1. |
| APRENDIZAJE_RED | 0.001 controla el paso de Adam al ajustar pesos. Un paso muy grande puede ser inestable; uno muy pequeño puede aprender lento. |
| EPOCAS_RED | 200 recorridos máximos por entrenamiento; early stopping puede terminar antes. |
| LOTE_RED | 64 observaciones por actualización. Cambiarlo afecta costo y trayectoria de aprendizaje. |
| PACIENCIA_RED | 15 épocas sin mejora de val_loss antes de detenerse y restaurar los mejores pesos. |
| EPOCAS_REENTRENAMIENTO | 50 épocas fijas para el modelo final entrenado con train+validación, ya sin usar prueba para detenerlo. Decide este número mirando el historial de validación, antes de evaluar prueba. |

Binaria usa una salida sigmoid y binary_crossentropy; multiclase usa softmax y sparse_categorical_crossentropy; regresión usa salida lineal y mse. La arquitectura está visible abajo. tf.data divide las tablas en lotes; limitar su pool a un hilo reduce consumo para sesiones pequeñas de Colab.

La curva loss corresponde al entrenamiento y val_loss a validación; una divergencia persistente merece revisar complejidad y preparación. No se etiqueta automáticamente como sobreajuste. Cambiar arquitectura exige repetir este bloque y la elección final.

In [ ]:
NEURONAS_RED = [32, 16]
DROPOUT_RED = 0.20
APRENDIZAJE_RED = 0.001
EPOCAS_RED = 200
LOTE_RED = 64
PACIENCIA_RED = 15
EPOCAS_REENTRENAMIENTO = 50

if TIPO_PROBLEMA in {"clasificacion", "regresion"} and EJECUTAR_RED:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEMILLA)
    if not NEURONAS_RED or any(n < 1 for n in NEURONAS_RED) or not 0 <= DROPOUT_RED < 1:
        raise ValueError("Revisa unidades y dropout de la red.")
    if min(APRENDIZAJE_RED, EPOCAS_RED, LOTE_RED, EPOCAS_REENTRENAMIENTO) <= 0 or PACIENCIA_RED < 0:
        raise ValueError("Revisa tasa, épocas, lote y paciencia.")
    preparacion_red_ajustada = clone(preprocesador_sin_ajustar).fit(X_train, y_train)
    X_train_red = preparacion_red_ajustada.transform(X_train).astype(np.float32)
    X_validacion_red = preparacion_red_ajustada.transform(X_validacion).astype(np.float32)
    unidades_salida_red = len(clases_originales) if TIPO_PROBLEMA == "clasificacion" and len(clases_originales) > 2 else 1
    activacion_salida_red = "softmax" if unidades_salida_red > 1 else "sigmoid" if TIPO_PROBLEMA == "clasificacion" else "linear"
    perdida_red = "sparse_categorical_crossentropy" if unidades_salida_red > 1 else "binary_crossentropy" if TIPO_PROBLEMA == "clasificacion" else "mse"
    capas_ocultas_red = [capa for n in NEURONAS_RED for capa in [
        tf.keras.layers.Dense(n, activation="relu"), tf.keras.layers.Dropout(DROPOUT_RED)]]
    red_sin_ajustar = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train_red.shape[1],)),
        *capas_ocultas_red,
        tf.keras.layers.Dense(unidades_salida_red, activation=activacion_salida_red)])
    red_ajustada = tf.keras.models.clone_model(red_sin_ajustar)
    red_ajustada.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=APRENDIZAJE_RED), loss=perdida_red)
    parada_red = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA_RED, restore_best_weights=True)
    opciones_dataset_red = tf.data.Options()
    opciones_dataset_red.threading.private_threadpool_size = 1
    dataset_train_red = tf.data.Dataset.from_tensor_slices((X_train_red, y_train.to_numpy())).batch(LOTE_RED).with_options(opciones_dataset_red)
    dataset_validacion_red = tf.data.Dataset.from_tensor_slices((X_validacion_red, y_validacion.to_numpy())).batch(LOTE_RED).with_options(opciones_dataset_red)
    inicio_red = time.perf_counter()
    historial_red = red_ajustada.fit(dataset_train_red, validation_data=dataset_validacion_red,
        epochs=EPOCAS_RED, callbacks=[parada_red], verbose=0)
    duracion_red = time.perf_counter() - inicio_red
    print("Épocas ejecutadas:", len(historial_red.history["loss"]), "Tiempo (s):", round(duracion_red, 2))
    red_ajustada.summary()
    pd.DataFrame(historial_red.history).plot(xlabel="Época (desde 0)", ylabel="Pérdida", title="Aprendizaje de la red")
    plt.show()

### 7.5 Comparación común antes de elegir
**Objetivo:** leer todas las fuentes con las mismas métricas. **Entrada:** candidatos ajustados, train y validación. **Operación:** predecir sin reentrenar y calcular indicadores homogéneos. **Resultado:** tabla por fuente, modelo y partición, brechas y tiempos de bloques. **Interpretación:** decide con validación y con el costo de los errores, no con el mayor puntaje de entrenamiento.

Train mide ajuste a ejemplos conocidos. Validación mide observaciones no usadas por ese entrenamiento, aunque algunas herramientas utilicen CV interna. Una brecha no identifica por sí sola la causa de errores.

Los tiempos se muestran por bloque: el manual incluye todos sus modelos; FLAML incluye búsqueda y se informa por separado del reajuste de candidatos; Lazy incluye su comparación; la red incluye su entrenamiento. No son tiempos por fila ni una competencia de velocidad idéntica entre algoritmos.

In [ ]:
if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
    modelos_manuales_comparacion = modelos_clasificacion_ajustados if TIPO_PROBLEMA == "clasificacion" else modelos_regresion_ajustados
    candidatos_comparacion = {
        **{("manual", n): (m, None, False) for n, m in modelos_manuales_comparacion.items()},
        **({("lazypredict", n): (m, None, False) for n, m in modelos_lazy_ajustados.items()} if EJECUTAR_LAZY else {}),
        **({("flaml", n): (m, None, False) for n, m in modelos_flaml_ajustados.items()} if EJECUTAR_FLAML else {}),
        **({("red_neuronal", "Red tabular"): (red_ajustada, preparacion_red_ajustada, True)} if EJECUTAR_RED else {})
    }
    filas_comparacion_ampliada = []
    for (fuente_comparacion, nombre_comparacion), (modelo_comparacion, preparacion_comparacion, es_red_comparacion) in candidatos_comparacion.items():
        for nombre_particion, datos_particion, reales_particion in [("Train", X_train, y_train), ("Validación", X_validacion, y_validacion)]:
            salida_comparacion = predecir_tabular(modelo_comparacion, datos_particion, TIPO_PROBLEMA,
                clases_codificadas if TIPO_PROBLEMA == "clasificacion" else None,
                UMBRAL, indice_positivo if TIPO_PROBLEMA == "clasificacion" else None,
                preparacion_comparacion, es_red_comparacion)
            metricas_comparacion = metricas_clasificacion(reales_particion, salida_comparacion["predicciones"],
                salida_comparacion["probabilidades"], clases_codificadas, indice_positivo, salida_comparacion["puntuaciones"]) if TIPO_PROBLEMA == "clasificacion" else metricas_regresion(reales_particion, salida_comparacion["predicciones"])
            filas_comparacion_ampliada.append({"Fuente": fuente_comparacion, "Modelo": nombre_comparacion,
                "Partición": nombre_particion, **metricas_comparacion})
    tabla_comparacion_ampliada = pd.DataFrame(filas_comparacion_ampliada).set_index(["Fuente", "Modelo", "Partición"])
    display(tabla_comparacion_ampliada.round(4))
    brechas_comparacion = tabla_comparacion_ampliada.xs("Train", level="Partición") - tabla_comparacion_ampliada.xs("Validación", level="Partición")
    display(brechas_comparacion.add_prefix("Train menos validación: ").round(4))
    tiempos_bloques = {"Manual (todos)": duracion_manual_clasificacion if TIPO_PROBLEMA == "clasificacion" else duracion_manual_regresion,
        **({"LazyPredict": duracion_lazy} if EJECUTAR_LAZY else {}),
        **({"FLAML búsqueda": duracion_flaml} if EJECUTAR_FLAML else {}),
        **({"Red neuronal": duracion_red} if EJECUTAR_RED else {})}
    display(pd.Series(tiempos_bloques, name="Segundos del bloque").to_frame())

## 8. Elección explícita y evaluación
### 8.1 Selecciona una fuente y un candidato
**Objetivo:** decidir qué modelo llevarás a prueba y a la base nueva. **Entrada:** tabla de validación. **Operación:** leer tu configuración, sin ordenar ni adoptar automáticamente el primero. **Resultado:** modelo_tabular_elegido y sus salidas de validación. **Interpretación:** revisa sus errores y, en binaria, el umbral.

| Parámetro | Cómo usarlo |
|---|---|
| FUENTE_MODELO_FINAL | manual (inicio), lazypredict, flaml o red_neuronal. La fuente debe haberse ejecutado. |
| MODELO_CLASIFICACION_ELEGIDO | Nombre manual exacto, inicialmente Random Forest. Solo clasificación con fuente manual. |
| MODELO_REGRESION_ELEGIDO | Nombre manual exacto, inicialmente Random Forest. Solo regresión con fuente manual. |
| MODELO_LAZY_ELEGIDO | Nombre del ranking, inicialmente AdaBoostClassifier en clasificación y RandomForestRegressor en regresión. Cámbialo por uno realmente recuperado. |
| FAMILIA_FLAML_ELEGIDA | mejor_cv usa el ganador de CV, como la opción A de la referencia. También puedes escribir lgbm, xgboost, rf o extra_tree si se evaluó. |
| UMBRALES_A_COMPARAR | Lista de cortes binarios para comparar sobre validación. No cambia UMBRAL ni el modelo por sí sola. |
| N_DECILES | 10 agrupa puntuaciones de mayor a menor en diez grupos. Si hay pocas filas se informa y se omite la tabla. |
| COLUMNAS_SEGMENTOS | [] no agrupa errores; ["canal"] resume error absoluto por canal en regresión. |

Elegir FLAML autoriza usar su ganador de CV dentro del presupuesto, pero no reemplaza la fuente manual a menos que lo escribas aquí. Si cambias la elección, repite diagnósticos, evaluación final y guardado; evita elegir de nuevo después de mirar prueba.

In [ ]:
FUENTE_MODELO_FINAL = "manual"
MODELO_CLASIFICACION_ELEGIDO = "Random Forest"
MODELO_REGRESION_ELEGIDO = "Random Forest"
MODELO_LAZY_ELEGIDO = "AdaBoostClassifier" if TIPO_PROBLEMA == "clasificacion" else "RandomForestRegressor"
FAMILIA_FLAML_ELEGIDA = "mejor_cv"
UMBRALES_A_COMPARAR = [0.20, 0.35, 0.50, 0.65, 0.80]
N_DECILES = 10
COLUMNAS_SEGMENTOS = []

if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
    if FUENTE_MODELO_FINAL not in {"manual", "lazypredict", "flaml", "red_neuronal"}:
        raise ValueError("Fuente final no reconocida.")
    if FUENTE_MODELO_FINAL == "manual":
        nombre_modelo_elegido = MODELO_CLASIFICACION_ELEGIDO if TIPO_PROBLEMA == "clasificacion" else MODELO_REGRESION_ELEGIDO
    elif FUENTE_MODELO_FINAL == "lazypredict":
        nombre_modelo_elegido = MODELO_LAZY_ELEGIDO
    elif FUENTE_MODELO_FINAL == "flaml":
        if not EJECUTAR_FLAML:
            raise ValueError("Activa y ejecuta FLAML antes de elegirlo.")
        nombre_modelo_elegido = mejor_familia_flaml if FAMILIA_FLAML_ELEGIDA == "mejor_cv" else FAMILIA_FLAML_ELEGIDA
    else:
        nombre_modelo_elegido = "Red tabular"
    clave_candidato_elegido = (FUENTE_MODELO_FINAL, nombre_modelo_elegido)
    if clave_candidato_elegido not in candidatos_comparacion:
        raise ValueError("El candidato no fue ejecutado. Revisa fuente, bandera y nombre de la tabla.")
    modelo_tabular_elegido, preparacion_elegida, es_red_elegida = candidatos_comparacion[clave_candidato_elegido]
    salida_validacion_elegida = predecir_tabular(modelo_tabular_elegido, X_validacion, TIPO_PROBLEMA,
        clases_codificadas if TIPO_PROBLEMA == "clasificacion" else None, UMBRAL,
        indice_positivo if TIPO_PROBLEMA == "clasificacion" else None, preparacion_elegida, es_red_elegida)
    print("Elección:", clave_candidato_elegido)

### 8.2 Leer errores, umbral y deciles en clasificación
**Entrada:** salidas de validación del candidato que acabas de elegir. **Operación:** mostrar matriz y métricas; en binaria con probabilidades, comparar umbrales y agrupar por riesgo. **Resultado:** decisiones y proporciones por grupo. **Interpretación:** bajar el umbral puede aumentar recall y falsas alarmas; elige según el costo del error.

El decil 1 reúne puntuaciones más altas. Lift=2 indica que ese grupo tiene el doble de tasa del evento que toda validación; no implica efecto causal de una campaña. Captura acumulada indica cuántos eventos reales quedan cubiertos al considerar los primeros grupos. Los empates se ordenan establemente por fila: los grupos no son cortes únicos de probabilidad.

Para regresión, las figuras siguientes muestran real/predicho, residuo y distribución del error; revisa sesgos y casos extremos. COLUMNAS_SEGMENTOS permite ver en qué grupos se concentra el error, sin retirar esos casos.

In [ ]:
if TIPO_PROBLEMA == "clasificacion":
    predicciones_validacion_elegido = salida_validacion_elegida["predicciones"]
    probabilidades_validacion_elegido = salida_validacion_elegida["probabilidades"]
    ConfusionMatrixDisplay.from_predictions(y_validacion, predicciones_validacion_elegido,
        labels=clases_codificadas, display_labels=clases_originales)
    plt.title("Matriz de confusión · validación")
    plt.show()
    print(classification_report(y_validacion, predicciones_validacion_elegido,
        labels=clases_codificadas, target_names=[str(c) for c in clases_originales], zero_division=0))
    if len(clases_originales) == 2 and probabilidades_validacion_elegido is not None:
        filas_umbrales = [{"Umbral": u, **metricas_clasificacion(y_validacion,
            np.where(probabilidades_validacion_elegido[:, indice_positivo] >= u, indice_positivo, 1 - indice_positivo),
            probabilidades_validacion_elegido, clases_codificadas, indice_positivo)} for u in UMBRALES_A_COMPARAR]
        display(pd.DataFrame(filas_umbrales).round(4))
        fpr_validacion, tpr_validacion, cortes_roc = roc_curve(y_validacion == indice_positivo, probabilidades_validacion_elegido[:, indice_positivo])
        plt.plot(fpr_validacion, tpr_validacion)
        plt.plot([0,1], [0,1], "--", color="gray")
        plt.xlabel("Falsos positivos")
        plt.ylabel("Recall")
        plt.show()
        base_deciles = pd.DataFrame({"probabilidad": probabilidades_validacion_elegido[:, indice_positivo],
            "evento": (y_validacion.to_numpy() == indice_positivo).astype(int)})
        if N_DECILES < 2:
            raise ValueError("N_DECILES debe ser al menos 2.")
        if len(base_deciles) >= N_DECILES and base_deciles["evento"].sum() > 0:
            base_deciles_ordenada = base_deciles.sort_values("probabilidad", ascending=False, kind="stable")
            base_deciles_grupos = base_deciles_ordenada.assign(decil=np.floor(np.arange(len(base_deciles_ordenada)) * N_DECILES / len(base_deciles_ordenada)).astype(int) + 1)
            resumen_deciles = base_deciles_grupos.groupby("decil").agg(casos=("evento","size"), eventos=("evento","sum"), tasa=("evento","mean"))
            tabla_deciles = resumen_deciles.assign(lift=resumen_deciles["tasa"]/base_deciles["evento"].mean(),
                captura_acumulada_pct=resumen_deciles["eventos"].cumsum()/base_deciles["evento"].sum()*100)
            display(tabla_deciles)
        else:
            print("No hay suficientes filas/eventos para deciles.")
    elif probabilidades_validacion_elegido is None:
        print("Este candidato no produce probabilidades: el umbral, barrido y deciles no aplican. No se fabrican probabilidades.")

In [ ]:
if TIPO_PROBLEMA == "regresion":
    predicciones_validacion_regresion = salida_validacion_elegida["predicciones"]
    diagnostico_regresion = df_validacion_limpia.assign(ml_prediccion=predicciones_validacion_regresion,
        ml_residuo=y_validacion.to_numpy() - predicciones_validacion_regresion,
        ml_error_absoluto=np.abs(y_validacion.to_numpy() - predicciones_validacion_regresion))
    fig_errores, ejes_errores = plt.subplots(1,3,figsize=(15,4))
    ejes_errores[0].scatter(y_validacion, predicciones_validacion_regresion, alpha=0.5)
    limites_reales = [y_validacion.min(), y_validacion.max()]
    ejes_errores[0].plot(limites_reales, limites_reales, "--", color="gray")
    ejes_errores[0].set(xlabel="Real", ylabel="Predicción")
    ejes_errores[1].scatter(diagnostico_regresion["ml_prediccion"], diagnostico_regresion["ml_residuo"], alpha=0.5)
    ejes_errores[1].axhline(0,color="gray",linestyle="--")
    ejes_errores[1].set(xlabel="Predicción",ylabel="Real menos predicción")
    sns.histplot(diagnostico_regresion["ml_residuo"],ax=ejes_errores[2])
    plt.tight_layout()
    plt.show()
    display(diagnostico_regresion.sort_values("ml_error_absoluto",ascending=False).head(10))
    for segmento_error in COLUMNAS_SEGMENTOS:
        display(diagnostico_regresion.groupby(segmento_error,dropna=False)["ml_error_absoluto"].agg(["count","mean","median"]))

### 8.3 Evaluación final: usar prueba después de elegir
**Objetivo:** medir la decisión final sobre filas reservadas. **Entrada:** modelo/configuración elegidos y train+validación. **Operación:** crear y entrenar un modelo nuevo con esos dos conjuntos, luego predecir prueba. **Resultado:** métricas finales; los modelos comparados se conservan. **Interpretación:** no uses esta tabla para volver a elegir.

En FLAML se congela la configuración elegida y se reentrena su Pipeline; no se vuelve a ejecutar la búsqueda sobre prueba. En LazyPredict se conserva también su preprocesador interno. La red aprende durante EPOCAS_REENTRENAMIENTO fijas, sin early stopping sobre prueba.

El codificador puede aprender nuevas categorías de los predictores de validación al reentrenar. La red se reconstruye con el nuevo número de columnas para evitar desajustes. Las clases del target ya debían existir en entrenamiento.

En series el modelo conoce hasta final de validación y pronostica toda prueba recursivamente. Repetir esta evaluación después de cambiar hiperparámetros deja de ser una evaluación final independiente de tus decisiones.

In [ ]:
if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
    X_desarrollo_final = pd.concat([X_train, X_validacion], axis=0)
    y_desarrollo_final = pd.concat([y_train, y_validacion], axis=0)
    if es_red_elegida:
        preprocesador_final_ajustado = clone(preprocesador_sin_ajustar).fit(X_desarrollo_final, y_desarrollo_final)
        X_desarrollo_red = preprocesador_final_ajustado.transform(X_desarrollo_final).astype(np.float32)
        capas_red_final = [capa for n in NEURONAS_RED for capa in [
            tf.keras.layers.Dense(n, activation="relu"), tf.keras.layers.Dropout(DROPOUT_RED)]]
        red_final_sin_ajustar = tf.keras.Sequential([tf.keras.layers.Input(shape=(X_desarrollo_red.shape[1],)),
            *capas_red_final, tf.keras.layers.Dense(unidades_salida_red, activation=activacion_salida_red)])
        red_final_ajustada = tf.keras.models.clone_model(red_final_sin_ajustar)
        red_final_ajustada.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=APRENDIZAJE_RED), loss=perdida_red)
        dataset_desarrollo_red = tf.data.Dataset.from_tensor_slices((X_desarrollo_red, y_desarrollo_final.to_numpy())).batch(LOTE_RED).with_options(opciones_dataset_red)
        historial_red_final = red_final_ajustada.fit(dataset_desarrollo_red, epochs=EPOCAS_REENTRENAMIENTO, verbose=0)
        modelo_tabular_final = red_final_ajustada
    else:
        pipeline_final_sin_ajustar = clone(modelo_tabular_elegido)
        pipeline_final_ajustado = clone(pipeline_final_sin_ajustar).fit(X_desarrollo_final, y_desarrollo_final)
        modelo_tabular_final = pipeline_final_ajustado
        preprocesador_final_ajustado = pipeline_final_ajustado.named_steps["preparacion"]
    salida_prueba_final = predecir_tabular(modelo_tabular_final, X_prueba, TIPO_PROBLEMA,
        clases_codificadas if TIPO_PROBLEMA == "clasificacion" else None, UMBRAL,
        indice_positivo if TIPO_PROBLEMA == "clasificacion" else None,
        preprocesador_final_ajustado if es_red_elegida else None, es_red_elegida)
    if TIPO_PROBLEMA == "clasificacion":
        y_prueba = pd.Series(codificador_target_ajustado.transform(df_prueba_limpia[TARGET]), index=X_prueba.index)
        metricas_prueba_final = metricas_clasificacion(y_prueba, salida_prueba_final["predicciones"],
            salida_prueba_final["probabilidades"], clases_codificadas, indice_positivo, salida_prueba_final["puntuaciones"])
        ConfusionMatrixDisplay.from_predictions(y_prueba, salida_prueba_final["predicciones"], labels=clases_codificadas, display_labels=clases_originales)
        plt.title("Prueba final")
        plt.show()
    else:
        metricas_prueba_final = metricas_regresion(df_prueba_limpia[TARGET], salida_prueba_final["predicciones"])

else:
    df_desarrollo_temporal = pd.concat([df_train_limpio, df_validacion_limpia], axis=0)
    historia_desarrollo_temporal = df_desarrollo_temporal.set_index(FECHA)[TARGET].copy()
    tabla_desarrollo_temporal_completa = construir_rezagos(historia_desarrollo_temporal, REZAGOS, VENTANAS, FERIADOS, CALENDARIO_AMPLIADO)
    tabla_desarrollo_temporal_entrenable = tabla_desarrollo_temporal_completa.dropna()
    modelo_temporal_final_ajustado = clone(modelos_temporales_sin_ajustar[MODELO_TEMPORAL_ELEGIDO]).fit(
        tabla_desarrollo_temporal_entrenable.drop(columns="objetivo"), tabla_desarrollo_temporal_entrenable["objetivo"]) if MODELO_TEMPORAL_ELEGIDO in modelos_temporales_sin_ajustar else None
    fechas_prueba_temporal = pd.date_range(df_prueba_limpia[FECHA].iloc[0], periods=len(df_prueba_limpia), freq=FRECUENCIA)
    predicciones_prueba_temporal = pronostico_recursivo(modelo_temporal_final_ajustado, historia_desarrollo_temporal, fechas_prueba_temporal, REZAGOS, VENTANAS, FERIADOS, CALENDARIO_AMPLIADO,
        PERIODO_ESTACIONAL if MODELO_TEMPORAL_ELEGIDO == "Estacional" else None)
    metricas_prueba_final = metricas_regresion(df_prueba_limpia[TARGET], predicciones_prueba_temporal)
    errores_prueba_temporal = errores_temporales(pd.Series(df_prueba_limpia[TARGET].to_numpy(), index=fechas_prueba_temporal), predicciones_prueba_temporal)
    display(errores_prueba_temporal.groupby("dia_semana").agg(pronosticos=("error_absoluto", "size"), MAE=("error_absoluto", "mean")))
    diagnostico_prueba_temporal = pd.DataFrame({"Real": df_prueba_limpia[TARGET].to_numpy(), "Predicción": predicciones_prueba_temporal.to_numpy()}, index=fechas_prueba_temporal)
    diagnostico_prueba_temporal.plot(figsize=(12, 4), title="Prueba recursiva final")
    plt.show()
display(pd.Series(metricas_prueba_final, name="Prueba final").to_frame())

## 9. Interpretación del modelo
### 9.1 Importancia de variables y reglas
**Objetivo:** identificar qué predictores utiliza el candidato, sin atribuir causalidad. **Entrada:** modelo comparado (ajustado sin validación) y validación. **Operación:** permutar una columna y medir cuánto empeora el puntaje. **Resultado:** importancia media y dispersión. **Interpretación:** una importancia cercana a cero puede deberse a redundancia con otras variables.

EJECUTAR_IMPORTANCIA=False omite el cálculo. True repite cinco permutaciones por predictor; puede tardar con muchas columnas. TOP_IMPORTANCIAS=15 limita filas mostradas. MOSTRAR_ARBOL=False evita dibujar reglas; True las muestra hasta tres niveles cuando el candidato es un árbol individual compatible.

Para clasificación se usa F1 macro y para regresión -MAE; una caída mayor indica más dependencia de esa variable. Para la red se calcula la misma permutación con la función de predicción común. En series se muestra importancia interna del árbol, que usa otra escala. Persistencia no tiene ese atributo.

Cambiar estos parámetros solo exige repetir interpretación; no reentrena ni elige el modelo.

In [ ]:
EJECUTAR_IMPORTANCIA = False
MOSTRAR_ARBOL = False
TOP_IMPORTANCIAS = 15

if EJECUTAR_IMPORTANCIA:
    if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
        if not es_red_elegida:
            resultado_permutacion = permutation_importance(modelo_tabular_elegido, X_validacion, y_validacion,
                scoring="f1_macro" if TIPO_PROBLEMA == "clasificacion" else "neg_mean_absolute_error", n_repeats=5, random_state=SEMILLA, n_jobs=1)
            tabla_importancias = pd.DataFrame({"variable": predictores_declarados,
                "importancia": resultado_permutacion.importances_mean,
                "desviacion": resultado_permutacion.importances_std}).sort_values("importancia", ascending=False)
            estimador_para_reglas = modelo_tabular_elegido.named_steps["modelo"]
            if MOSTRAR_ARBOL and isinstance(estimador_para_reglas, (DecisionTreeClassifier, DecisionTreeRegressor)):
                plt.figure(figsize=(18,7))
                plot_tree(estimador_para_reglas, max_depth=3,
                    feature_names=modelo_tabular_elegido.named_steps["preparacion"].get_feature_names_out(), filled=True)
                plt.show()
        else:
            puntaje_base_red = f1_score(y_validacion, salida_validacion_elegida["predicciones"], average="macro", zero_division=0) if TIPO_PROBLEMA == "clasificacion" else -mean_absolute_error(y_validacion, salida_validacion_elegida["predicciones"])
            filas_importancia_red = []
            for columna_permutada in predictores_declarados:
                for repeticion_permutacion in range(5):
                    generador_permutacion = np.random.default_rng(SEMILLA + repeticion_permutacion)
                    X_validacion_permutada = X_validacion.assign(**{columna_permutada: generador_permutacion.permutation(X_validacion[columna_permutada].to_numpy())})
                    salida_red_permutada = predecir_tabular(modelo_tabular_elegido, X_validacion_permutada, TIPO_PROBLEMA,
                        clases_codificadas if TIPO_PROBLEMA == "clasificacion" else None, UMBRAL,
                        indice_positivo if TIPO_PROBLEMA == "clasificacion" else None, preparacion_elegida, True)
                    puntaje_red_permutada = f1_score(y_validacion, salida_red_permutada["predicciones"], average="macro", zero_division=0) if TIPO_PROBLEMA == "clasificacion" else -mean_absolute_error(y_validacion, salida_red_permutada["predicciones"])
                    filas_importancia_red.append({"variable": columna_permutada, "caida": puntaje_base_red - puntaje_red_permutada})
            tabla_importancias = pd.DataFrame(filas_importancia_red).groupby("variable")["caida"].agg(importancia="mean", desviacion="std").reset_index().sort_values("importancia", ascending=False)
    elif MODELO_TEMPORAL_ELEGIDO in modelos_temporales_ajustados:
        tabla_importancias = pd.DataFrame({"variable": X_train_temporal.columns,
            "importancia": modelos_temporales_ajustados[MODELO_TEMPORAL_ELEGIDO].feature_importances_}).sort_values("importancia", ascending=False)
    else:
        tabla_importancias = pd.DataFrame(columns=["variable","importancia"])
        print("Las líneas base temporales no tienen importancias de árboles.")
    display(tabla_importancias.head(TOP_IMPORTANCIAS))
    if not tabla_importancias.empty:
        tabla_importancias.head(TOP_IMPORTANCIAS).set_index("variable")["importancia"].sort_values().plot.barh()
        plt.show()

### 9.2 SHAP: contribuciones a una predicción
**Objetivo:** describir cómo las variables contribuyen a la salida del modelo respecto a una referencia. **Entrada:** candidato comparado y un pequeño fondo de entrenamiento. **Operación:** aproximar contribuciones por permutación. **Resultado:** barras globales y explicación individual. **Interpretación:** un aporte positivo aumenta la salida explicada; no demuestra causalidad ni es un coeficiente estable.

EJECUTAR_SHAP=False lo omite. FILAS_FONDO_SHAP=30 define ejemplos de referencia; aumentar cantidad aumenta costo. FILAS_EXPLICAR_SHAP=5 limita casos explicados. CLASE_SHAP indica la etiqueta original cuya probabilidad se explica; solo clasificación con probabilidades.

El waterfall parte del valor base, suma contribuciones y llega a la predicción. En clasificación se explican probabilidades; en regresión, unidades del target. En la red binaria el código adapta la probabilidad del código 1 a la etiqueta elegida. Las columnas categóricas aparecen como indicadores one-hot.

Los clasificadores sin probabilidades se omiten explícitamente; no se interpretan sus códigos de clase como probabilidades. Para series se explica un paso con rezagos conocidos, no toda la cadena recursiva. Repetir SHAP no entrena el modelo ni cambia la elección.

**salida_shap_modelo(matriz, modelo, columnas, problema, es_red=False)** es un adaptador de salida, no una limpieza:
- matriz: valores numéricos ya transformados que entrega SHAP.
- modelo: estimador o Pipeline interno ya ajustado.
- columnas: nombres de esas variables transformadas; se reconstruye un DataFrame si el Pipeline lo requiere.
- problema: clasificacion devuelve probabilidades; regresion devuelve cantidades.
- es_red: True utiliza Keras con entradas float32.
- Retorna un vector/matriz de salidas, sin ajustar ni modificar el modelo.

Ejemplo ejecutable dentro del bloque activado: salida_shap_modelo(np.array([[2]]), DummyRegressor(strategy="mean").fit([[0],[1]],[10,20]), ["x"], "regresion").
Salida esperada: array([15.]). Este adaptador evita que SHAP pierda los nombres de columnas que necesita el preprocesador interno de LazyPredict.


In [ ]:
EJECUTAR_SHAP = False
FILAS_FONDO_SHAP = 30
FILAS_EXPLICAR_SHAP = 5
CLASE_SHAP = CLASE_POSITIVA

if EJECUTAR_SHAP:
    import shap
    from functools import partial

    def salida_shap_modelo(matriz, modelo, columnas, problema, es_red=False):
        """Adapta la matriz de SHAP a los nombres que requiere un Pipeline recuperado."""
        datos_modelo = pd.DataFrame(matriz, columns=columnas) if isinstance(modelo, Pipeline) else np.asarray(matriz)
        if es_red:
            return modelo.predict(np.asarray(matriz, dtype=np.float32), verbose=0)
        return modelo.predict_proba(datos_modelo) if problema == "clasificacion" else modelo.predict(datos_modelo)

    if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
        preparador_shap = preparacion_elegida if es_red_elegida else modelo_tabular_elegido.named_steps["preparacion"]
        estimador_shap = modelo_tabular_elegido if es_red_elegida else modelo_tabular_elegido.named_steps["modelo"]
        if TIPO_PROBLEMA == "clasificacion" and not es_red_elegida and not hasattr(estimador_shap, "predict_proba"):
            print("SHAP de probabilidad omitido: el candidato no produce probabilidades.")
        else:
            fondo_shap = preparador_shap.transform(X_train.head(FILAS_FONDO_SHAP))
            datos_shap = preparador_shap.transform(X_validacion.head(FILAS_EXPLICAR_SHAP))
            nombres_shap = preparador_shap.get_feature_names_out()
            funcion_shap = partial(salida_shap_modelo, modelo=estimador_shap, columnas=nombres_shap, problema=TIPO_PROBLEMA, es_red=es_red_elegida)
            explicador_shap = shap.Explainer(funcion_shap, fondo_shap, algorithm="permutation", feature_names=nombres_shap, seed=SEMILLA)
            valores_shap = explicador_shap(datos_shap, max_evals=2*len(nombres_shap)+1)
            if TIPO_PROBLEMA == "clasificacion":
                indice_clase_shap = int(codificador_target_ajustado.transform([CLASE_SHAP])[0])
                if es_red_elegida and len(clases_originales) == 2:
                    valores_shap_sigmoid = valores_shap[:,:,0] if valores_shap.values.ndim == 3 else valores_shap
                    valores_shap_clase = shap.Explanation(values=-valores_shap_sigmoid.values,
                        base_values=1-valores_shap_sigmoid.base_values, data=valores_shap_sigmoid.data,
                        feature_names=valores_shap_sigmoid.feature_names) if indice_clase_shap == 0 else valores_shap_sigmoid
                else:
                    valores_shap_clase = valores_shap[:,:,indice_clase_shap]
                shap.plots.bar(valores_shap_clase)
                shap.plots.waterfall(valores_shap_clase[0])
            else:
                valores_shap_regresion = valores_shap[:,:,0] if es_red_elegida and valores_shap.values.ndim == 3 else valores_shap
                shap.plots.bar(valores_shap_regresion)
                shap.plots.waterfall(valores_shap_regresion[0])
    elif MODELO_TEMPORAL_ELEGIDO in modelos_temporales_ajustados:
        modelo_shap_temporal = modelos_temporales_ajustados[MODELO_TEMPORAL_ELEGIDO]
        explicador_temporal_shap = shap.Explainer(modelo_shap_temporal.predict, X_train_temporal.head(FILAS_FONDO_SHAP), algorithm="permutation", seed=SEMILLA)
        valores_temporales_shap = explicador_temporal_shap(X_train_temporal.tail(FILAS_EXPLICAR_SHAP), max_evals=2*X_train_temporal.shape[1]+1)
        shap.plots.bar(valores_temporales_shap)
        shap.plots.waterfall(valores_temporales_shap[0])
    else:
        print("SHAP omitido para las líneas base temporales.")

## 10. Guardar y aplicar a una base nueva
### 10.1 Conservar la decisión y sus transformaciones
**Objetivo:** reutilizar el modelo sin volver a aprender. **Entrada:** modelo final ajustado y configuración. **Operación:** guardar preprocesador, esquema, reglas, clases y modelo. **Resultado:** archivo joblib y JSON; la red añade un archivo .keras. **Interpretación:** conserva todos los archivos de una misma ejecución y sus versiones.

DIRECTORIO_SALIDA empieza en /content/salidas_guia_pc2. La carpeta se crea dentro de Colab; su contenido se pierde al terminar la sesión si no lo descargas o copias a Drive. Cambiar la ruta no reentrena.

REENTRENAR_SERIE_CON_TODA_HISTORIA=True autoriza otro modelo temporal operativo usando toda la historia después de reportar prueba. No modifica la evaluación ya mostrada. False mantiene el origen al final de validación, por lo que las fechas nuevas deben comenzar allí. Solo series.

El paquete tabular conserva el Pipeline final de train+validación; no incorpora prueba. Para FLAML se guarda el Pipeline sklearn elegido, no una búsqueda ni una clase dinámica. Para la red se guarda la arquitectura/pesos en .keras y el preprocesador en joblib. No cargues joblib de un origen que no conozcas.

In [ ]:
DIRECTORIO_SALIDA = "/content/salidas_guia_pc2"
REENTRENAR_SERIE_CON_TODA_HISTORIA = True

directorio_salida = Path(DIRECTORIO_SALIDA)
directorio_salida.mkdir(parents=True, exist_ok=True)
versiones_modelado = {**versiones_entorno,
    **({"ydata-profiling": version("ydata-profiling")} if EJECUTAR_PROFILE_REPORT else {}),
    **({"lazypredict": version("lazypredict")} if EJECUTAR_LAZY and TIPO_PROBLEMA != "serie_tiempo" else {}),
    **({"flaml": version("flaml")} if EJECUTAR_FLAML and TIPO_PROBLEMA != "serie_tiempo" else {}),
    **({"tensorflow": tf.__version__} if EJECUTAR_RED and TIPO_PROBLEMA != "serie_tiempo" else {})}
configuracion_modelo = {
    "tipo_problema": TIPO_PROBLEMA, "target": TARGET, "numericas": COLUMNAS_NUMERICAS,
    "categoricas": COLUMNAS_CATEGORICAS, "predictores": predictores_declarados,
    "identificadores": COLUMNAS_ID, "exclusiones": COLUMNAS_EXCLUIR,
    "reglas_numericas": REGLAS_NUMERICAS, "reglas_textos": REGLAS_TEXTOS,
    "reglas_fechas": REGLAS_FECHAS, "reglas_reemplazos": REGLAS_REEMPLAZOS,
    "imputar_numericas": IMPUTAR_NUMERICAS, "imputar_categoricas": IMPUTAR_CATEGORICAS,
    "estrategia_numerica": ESTRATEGIA_NUMERICA, "estrategia_categorica": ESTRATEGIA_CATEGORICA,
    "escalar": ESCALAR, "categoria_desconocida": CATEGORIA_DESCONOCIDA,
    "umbral": UMBRAL, "clase_positiva": CLASE_POSITIVA,
    "fecha": FECHA, "formato_fecha": FORMATO_FECHA_SERIE, "frecuencia": FRECUENCIA,
    "rezagos": REZAGOS, "ventanas": VENTANAS, "horizonte": HORIZONTE,
    "calendario_ampliado": CALENDARIO_AMPLIADO, "feriados": list(FERIADOS),
    "periodo_estacional": PERIODO_ESTACIONAL, "agregacion_mensual": AGREGACION_MENSUAL,
    "periodos_validacion_serie": PERIODOS_VALIDACION_SERIE, "periodos_prueba_serie": PERIODOS_PRUEBA_SERIE,
    "semilla": SEMILLA, "fraccion_validacion": FRACCION_VALIDACION, "fraccion_prueba": FRACCION_PRUEBA,
    "versiones": versiones_modelado,
    "tipos_columnas": TIPOS_COLUMNAS, "formatos_fecha": FORMATOS_FECHA,
    "fuente": FUENTE_MODELO_FINAL if TIPO_PROBLEMA != "serie_tiempo" else "temporal"}

if TIPO_PROBLEMA == "serie_tiempo":
    df_historia_operativa = pd.concat([df_train_limpio, df_validacion_limpia, df_prueba_limpia], axis=0) if REENTRENAR_SERIE_CON_TODA_HISTORIA else df_desarrollo_temporal.copy()
    historia_operativa = df_historia_operativa.set_index(FECHA)[TARGET].copy()
    tabla_operativa_completa = construir_rezagos(historia_operativa, REZAGOS, VENTANAS, FERIADOS, CALENDARIO_AMPLIADO)
    tabla_operativa_entrenable = tabla_operativa_completa.dropna()
    modelo_temporal_operativo = clone(modelos_temporales_sin_ajustar[MODELO_TEMPORAL_ELEGIDO]).fit(
        tabla_operativa_entrenable.drop(columns="objetivo"), tabla_operativa_entrenable["objetivo"]) if MODELO_TEMPORAL_ELEGIDO in modelos_temporales_sin_ajustar else None
    paquete_modelo = {"configuracion": configuracion_modelo, "modelo": modelo_temporal_operativo,
        "historia": historia_operativa, "nombre_modelo": MODELO_TEMPORAL_ELEGIDO,
        "esquema": {"fecha": str(df_historia_operativa[FECHA].dtype), "target": str(historia_operativa.dtype)}}
else:
    ruta_red_guardada = directorio_salida / "red_final.keras"
    if es_red_elegida:
        modelo_tabular_final.save(ruta_red_guardada)
    esquema_categorias_numericas = {c: (all(isinstance(v, (int,float,np.number,bool)) for v in X_desarrollo_final[c].dropna().unique())
        if X_desarrollo_final[c].notna().any() else None) for c in COLUMNAS_CATEGORICAS}
    paquete_modelo = {"configuracion": configuracion_modelo,
        "modelo": None if es_red_elegida else modelo_tabular_final,
        "preparacion": preprocesador_final_ajustado,
        "es_red": es_red_elegida, "archivo_red": ruta_red_guardada.name if es_red_elegida else None,
        "codificador_target": codificador_target_ajustado if TIPO_PROBLEMA == "clasificacion" else None,
        "nombre_modelo": nombre_modelo_elegido, "esquema": X_desarrollo_final.dtypes.astype(str).to_dict(),
        "categorias_numericas": esquema_categorias_numericas}
ruta_modelo_guardado = directorio_salida / "modelo_y_configuracion.joblib"
ruta_configuracion_guardada = directorio_salida / "configuracion.json"
joblib.dump(paquete_modelo, ruta_modelo_guardado)
ruta_configuracion_guardada.write_text(json.dumps({"configuracion": configuracion_modelo,
    "esquema": paquete_modelo["esquema"], "modelo": paquete_modelo["nombre_modelo"]},
    ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Modelo:", ruta_modelo_guardado)
print("Configuración:", ruta_configuracion_guardada)
if TIPO_PROBLEMA != "serie_tiempo" and es_red_elegida:
    print("Red Keras: descarga también", ruta_red_guardada)

### 10.2 Leer una segunda base
**Objetivo:** predecir observaciones nuevas con la decisión guardada. **Entrada:** otro CSV/Excel; no necesita target. **Operación:** recargar el paquete y leer el archivo. **Resultado:** df_nueva_original. **Interpretación:** conserva nombres y unidades de predictores, identificadores y orden.

| Parámetro | Uso |
|---|---|
| SUBIR_BASE_NUEVA | False usa la ruta; True abre selector de Colab. |
| RUTA_BASE_NUEVA | /content/mi_base_nueva.csv indica el segundo archivo. |
| FORMATO_BASE_NUEVA | csv o excel, según el archivo nuevo. |
| SEPARADOR_NUEVA | Delimitador CSV: "," o ";" según el archivo. |
| CODIFICACION_NUEVA | utf-8 inicialmente; ajusta si aparecen errores de letras. Solo CSV. |
| DECIMAL_NUEVA | "." inicialmente; "," si las cantidades usan coma decimal. |
| HOJA_NUEVA | 0 lee primera hoja; admite nombre de hoja en Excel. |
| EVALUAR_BASE_NUEVA_CON_TARGET | False solo predice. True compara con el target real, que debe existir y no tener nulos. |
| USAR_ARCHIVO_FUTURO | Solo series. False crea los siguientes HORIZONTE periodos; True lee tu archivo futuro con columna fecha. |

La red necesita también red_final.keras junto al joblib. La inferencia vuelve a importar TensorFlow si corresponde, pero no aprende nada. Las columnas añadidas empiezan con ml_; el archivo nuevo no debe usar ese prefijo para evitar colisiones. Cambiar la base nueva solo exige repetir desde esta sección.

In [ ]:
SUBIR_BASE_NUEVA = False
RUTA_BASE_NUEVA = "/content/mi_base_nueva.csv"
FORMATO_BASE_NUEVA = "csv"
SEPARADOR_NUEVA = ","
CODIFICACION_NUEVA = "utf-8"
DECIMAL_NUEVA = "."
HOJA_NUEVA = 0
EVALUAR_BASE_NUEVA_CON_TARGET = False
USAR_ARCHIVO_FUTURO = False

# Solo series. FECHA_FIN_PRONOSTICO se aplica al generar fechas sin archivo.
FECHA_FIN_PRONOSTICO = None  # Ejemplo: "2026-01-31"; None conserva HORIZONTE
FECHA_INICIO_REPORTE = None  # Ejemplo: "2025-10-20"; None desde el primer paso
FECHA_FIN_REPORTE = None  # Ejemplo: "2025-11-03"; None hasta el último paso

In [ ]:
paquete_modelo_cargado = joblib.load(ruta_modelo_guardado)
configuracion_cargada = paquete_modelo_cargado["configuracion"]
if paquete_modelo_cargado.get("es_red", False):
    import tensorflow as tf
    modelo_cargado = tf.keras.models.load_model(ruta_modelo_guardado.parent / paquete_modelo_cargado["archivo_red"])
else:
    modelo_cargado = paquete_modelo_cargado["modelo"]
preparacion_cargada = paquete_modelo_cargado.get("preparacion")
tipo_cargado = configuracion_cargada["tipo_problema"]
target_cargado = configuracion_cargada["target"]

if tipo_cargado == "serie_tiempo" and not USAR_ARCHIVO_FUTURO:
    inicio_futuro = paquete_modelo_cargado["historia"].index[-1] + pd.tseries.frequencies.to_offset(configuracion_cargada["frecuencia"])
    fin_futuro = pd.Timestamp(FECHA_FIN_PRONOSTICO) if FECHA_FIN_PRONOSTICO is not None else None
    if fin_futuro is not None and (pd.isna(fin_futuro) or fin_futuro < inicio_futuro):
        raise ValueError("FECHA_FIN_PRONOSTICO debe ser posterior al origen guardado.")
    fechas_futuras_configuradas = pd.date_range(start=inicio_futuro, end=fin_futuro,
        periods=configuracion_cargada["horizonte"] if fin_futuro is None else None, freq=configuracion_cargada["frecuencia"])
    if fin_futuro is not None and (len(fechas_futuras_configuradas) == 0 or fechas_futuras_configuradas[-1] != fin_futuro):
        raise ValueError("FECHA_FIN_PRONOSTICO debe coincidir con un periodo de la frecuencia guardada.")
    df_nueva_original = pd.DataFrame({configuracion_cargada["fecha"]: fechas_futuras_configuradas})
else:
    if tipo_cargado == "serie_tiempo" and FECHA_FIN_PRONOSTICO is not None:
        raise ValueError("Con USAR_ARCHIVO_FUTURO=True, el archivo define el horizonte; deja FECHA_FIN_PRONOSTICO=None.")
    if SUBIR_BASE_NUEVA:
        from google.colab import files
        archivos_nuevos_subidos = files.upload()
        if len(archivos_nuevos_subidos) != 1:
            raise ValueError("Sube exactamente un archivo nuevo.")
        ruta_nueva_lectura = Path(next(iter(archivos_nuevos_subidos)))
    else:
        ruta_nueva_lectura = Path(RUTA_BASE_NUEVA)
    if not ruta_nueva_lectura.is_file():
        raise FileNotFoundError(f"No existe {ruta_nueva_lectura}. Sube tu segundo archivo.")
    if FORMATO_BASE_NUEVA == "csv":
        df_nueva_original = pd.read_csv(ruta_nueva_lectura, sep=SEPARADOR_NUEVA, encoding=CODIFICACION_NUEVA, decimal=DECIMAL_NUEVA)
    elif FORMATO_BASE_NUEVA == "excel":
        df_nueva_original = pd.read_excel(ruta_nueva_lectura, sheet_name=HOJA_NUEVA, decimal=DECIMAL_NUEVA)
    else:
        raise ValueError("FORMATO_BASE_NUEVA debe ser csv o excel.")

if df_nueva_original.empty or not df_nueva_original.columns.is_unique:
    raise ValueError("La base nueva está vacía o tiene columnas repetidas.")
if any(str(c).startswith("ml_") for c in df_nueva_original.columns):
    raise ValueError("Renombra las columnas originales con prefijo ml_ para evitar sobrescribirlas.")
display(df_nueva_original.head())

### 10.3 Comprobar que la base nueva significa lo mismo
**Objetivo:** evitar predicciones sobre columnas equivocadas. **Entrada:** base nueva y esquema guardado. **Operación:** revisar faltantes, aplicar las mismas reglas de tipos/textos y comprobar nulos/categorías. **Resultado:** X_nueva o fechas futuras validadas. **Interpretación:** corrige el archivo ante errores; no se inventan predictores.

Columnas adicionales se informan y no se modelan. El target no es necesario para predecir y sus reglas se omiten si falta. No se eliminan filas nuevas: se conserva su orden. Las medianas y categorías provienen del preprocesador guardado, nunca del segundo archivo.

Cambiar entre entero y decimal compatible se informa y se admite; texto donde se esperaba una cantidad exige una conversión declarada previamente. Las categorías desconocidas obedecen tu política ignore/error. Un ID con ceros iniciales debe haberse leído como texto desde origen para no perderlos.

En series las fechas deben ser consecutivas y comenzar justo después del último periodo conocido por el modelo operativo. No se reordena ni agrega automáticamente el archivo futuro.

In [ ]:
columnas_requeridas_nuevas = [configuracion_cargada["fecha"]] if tipo_cargado == "serie_tiempo" else configuracion_cargada["predictores"]
columnas_faltantes_nuevas = sorted(set(columnas_requeridas_nuevas) - set(df_nueva_original.columns))
columnas_adicionales_nuevas = sorted(set(df_nueva_original.columns) - set(columnas_requeridas_nuevas) - {target_cargado} - set(configuracion_cargada["identificadores"]))
print("Predictores/fecha faltantes:", columnas_faltantes_nuevas)
print("Columnas adicionales que no se modelan:", columnas_adicionales_nuevas)
if columnas_faltantes_nuevas:
    raise ValueError("Faltan columnas requeridas. Corrige el archivo; no se crearán artificialmente.")

reglas_presentes_numericas = [c for c in configuracion_cargada["reglas_numericas"] if c in df_nueva_original]
reglas_presentes_textos = [c for c in configuracion_cargada["reglas_textos"] if c in df_nueva_original]
reglas_presentes_fechas = {c: f for c, f in configuracion_cargada["reglas_fechas"].items() if c in df_nueva_original}
reglas_presentes_reemplazos = {c: r for c, r in configuracion_cargada["reglas_reemplazos"].items() if c in df_nueva_original}
columnas_reglas_guardadas = set(configuracion_cargada["reglas_numericas"]) | set(configuracion_cargada["reglas_textos"]) | set(configuracion_cargada["reglas_fechas"]) | set(configuracion_cargada["reglas_reemplazos"])
print("Reglas omitidas por columnas no presentes:", sorted(columnas_reglas_guardadas - set(df_nueva_original.columns)))
df_nueva_reemplazos = aplicar_reglas(df_nueva_original, reglas_presentes_numericas, reglas_presentes_textos, reglas_presentes_fechas, reglas_presentes_reemplazos)
tipos_presentes_nuevos = {c: t for c, t in configuracion_cargada["tipos_columnas"].items() if c in df_nueva_reemplazos}
formatos_presentes_nuevos = {c: f for c, f in configuracion_cargada["formatos_fecha"].items() if c in df_nueva_reemplazos}
df_nueva_reglas = cambiar_tipos(df_nueva_reemplazos, tipos_presentes_nuevos, formatos_presentes_nuevos)

if tipo_cargado != "serie_tiempo":
    X_nueva = df_nueva_reglas.loc[:, configuracion_cargada["predictores"]].copy()
    comparacion_esquema_nuevo = pd.DataFrame({"entrenamiento": paquete_modelo_cargado["esquema"], "nuevo": X_nueva.dtypes.astype(str).to_dict()})
    display(comparacion_esquema_nuevo)
    for columna_nueva_numerica in configuracion_cargada["numericas"]:
        if not pd.api.types.is_numeric_dtype(X_nueva[columna_nueva_numerica]):
            raise TypeError(f"{columna_nueva_numerica} debe ser numérica. Revisa reglas y unidades.")
    for columna_nueva_categorica in configuracion_cargada["categoricas"]:
        tipos_categoria_nueva = {"texto" if isinstance(v, str) else "numero" if isinstance(v, (int, float, np.number, bool)) else "otro"
            for v in X_nueva[columna_nueva_categorica].dropna().unique()}
        tipo_categoria_guardado = paquete_modelo_cargado["esquema"][columna_nueva_categorica]
        categoria_guardada_numerica = paquete_modelo_cargado["categorias_numericas"].get(columna_nueva_categorica)
        if "otro" in tipos_categoria_nueva or len(tipos_categoria_nueva) > 1:
            raise TypeError(f"{columna_nueva_categorica} mezcla tipos incompatibles.")
        if tipos_categoria_nueva and categoria_guardada_numerica is not None and categoria_guardada_numerica != (tipos_categoria_nueva == {"numero"}):
            raise TypeError(f"{columna_nueva_categorica} cambió entre códigos numéricos y texto.")
    if np.isinf(X_nueva[configuracion_cargada["numericas"]].to_numpy(dtype=float, na_value=np.nan)).any():
        raise ValueError("La base nueva tiene infinitos.")
    if not configuracion_cargada["imputar_numericas"] and X_nueva[configuracion_cargada["numericas"]].isna().any().any():
        raise ValueError("Hay nulos numéricos y el modelo guardado no tiene imputación.")
    if not configuracion_cargada["imputar_categoricas"] and X_nueva[configuracion_cargada["categoricas"]].isna().any().any():
        raise ValueError("Hay nulos categóricos y el modelo guardado no tiene imputación.")
    if configuracion_cargada["categoricas"]:
        codificador_categorias_guardado = preparacion_cargada.named_transformers_["categoricas"].named_steps["codificacion"]
        categorias_desconocidas_nuevas = {c: [v for v in X_nueva[c].dropna().unique() if v not in conocidas]
            for c, conocidas in zip(configuracion_cargada["categoricas"], codificador_categorias_guardado.categories_)}
        print("Categorías desconocidas:", categorias_desconocidas_nuevas)
        if configuracion_cargada["categoria_desconocida"] == "error" and any(categorias_desconocidas_nuevas.values()):
            raise ValueError("Hay categorías desconocidas y configuraste detener la predicción.")
else:
    df_nueva_fechas = df_nueva_reglas.assign(**{configuracion_cargada["fecha"]:
        pd.to_datetime(df_nueva_reglas[configuracion_cargada["fecha"]], format=configuracion_cargada["formato_fecha"], errors="raise")})
    fechas_archivo_nuevo = pd.DatetimeIndex(df_nueva_fechas[configuracion_cargada["fecha"]])
    fechas_futuras_esperadas = pd.date_range(
        paquete_modelo_cargado["historia"].index[-1] + pd.tseries.frequencies.to_offset(configuracion_cargada["frecuencia"]),
        periods=len(df_nueva_fechas), freq=configuracion_cargada["frecuencia"])
    if not fechas_archivo_nuevo.equals(fechas_futuras_esperadas):
        raise ValueError("Las fechas nuevas deben ser consecutivas, ordenadas y comenzar después del origen guardado.")

### 10.4 Predecir y exportar
**Objetivo:** producir decisiones o cantidades para cada fila nueva. **Entrada:** esquema comprobado y modelo recargado. **Operación:** transformar con lo aprendido y predecir; no llamar a fit. **Resultado:** resultado_base_nueva y CSV. **Interpretación:** ml_prediccion es la respuesta estimada, no la real.

ml_fila_origen identifica la posición original desde cero. En clasificación con probabilidades se añade una columna por código de clase y se muestra su correspondencia con la etiqueta original. En binaria se usa el umbral guardado. Si el modelo solo produce etiquetas, se exportan etiquetas y se informa que no existen probabilidades.

Si activas evaluación con target se usan las mismas métricas; no se reentrena ni se calibra sobre esa base. Una clase nueva en el target impide evaluar con el codificador anterior. En series no se consulta el target futuro al pronosticar, aunque el archivo lo contenga.

In [ ]:
if tipo_cargado in {"clasificacion", "regresion"}:
    codificador_target_cargado = paquete_modelo_cargado.get("codificador_target")
    clases_cargadas = codificador_target_cargado.classes_ if tipo_cargado == "clasificacion" else None
    indice_positivo_cargado = int(codificador_target_cargado.transform([configuracion_cargada["clase_positiva"]])[0]) if tipo_cargado == "clasificacion" and len(clases_cargadas) == 2 else None
    salida_base_nueva = predecir_tabular(modelo_cargado, X_nueva, tipo_cargado,
        np.arange(len(clases_cargadas)) if tipo_cargado == "clasificacion" else None,
        configuracion_cargada["umbral"], indice_positivo_cargado,
        preparacion_cargada if paquete_modelo_cargado.get("es_red",False) else None,
        paquete_modelo_cargado.get("es_red",False))
    if tipo_cargado == "clasificacion":
        codigos_prediccion_nueva = salida_base_nueva["predicciones"].astype(int)
        etiquetas_prediccion_nueva = codificador_target_cargado.inverse_transform(codigos_prediccion_nueva)
        probabilidades_nuevas = salida_base_nueva["probabilidades"]
        columnas_probabilidades_nuevas = {f"ml_prob_clase_{i}": probabilidades_nuevas[:,i] for i in range(len(clases_cargadas))} if probabilidades_nuevas is not None else {}
        resultado_base_nueva = df_nueva_original.assign(ml_fila_origen=np.arange(len(df_nueva_original)),
            ml_prediccion=etiquetas_prediccion_nueva, **columnas_probabilidades_nuevas)
        if columnas_probabilidades_nuevas:
            display(pd.DataFrame({"Columna": list(columnas_probabilidades_nuevas), "Etiqueta": clases_cargadas}))
        else:
            print("El modelo no ofrece probabilidades. Se exportan únicamente sus etiquetas.")
    else:
        resultado_base_nueva = df_nueva_original.assign(ml_fila_origen=np.arange(len(df_nueva_original)),
            ml_prediccion=salida_base_nueva["predicciones"])
else:
    predicciones_futuras_nuevas = pronostico_recursivo(modelo_cargado, paquete_modelo_cargado["historia"],
        fechas_futuras_esperadas, configuracion_cargada["rezagos"], configuracion_cargada["ventanas"],
        configuracion_cargada.get("feriados", []), configuracion_cargada.get("calendario_ampliado", False),
        configuracion_cargada.get("periodo_estacional") if paquete_modelo_cargado["nombre_modelo"] == "Estacional" else None)
    resultado_base_nueva = df_nueva_original.assign(ml_fila_origen=np.arange(len(df_nueva_original)),
        ml_prediccion=predicciones_futuras_nuevas.to_numpy())
    resultado_base_nueva.plot(x=configuracion_cargada["fecha"],y="ml_prediccion",figsize=(12,4),title="Pronóstico futuro")
    plt.show()
display(resultado_base_nueva.head(20))

In [ ]:
if EVALUAR_BASE_NUEVA_CON_TARGET:
    if target_cargado not in df_nueva_reglas or df_nueva_reglas[target_cargado].isna().any():
        raise ValueError("Para evaluar necesitas target real no nulo en todas las filas nuevas.")
    if tipo_cargado == "clasificacion":
        clases_target_nuevo_desconocidas = set(df_nueva_reglas[target_cargado]) - set(clases_cargadas)
        if clases_target_nuevo_desconocidas:
            raise ValueError(f"El target nuevo contiene clases no entrenadas: {clases_target_nuevo_desconocidas}")
        reales_nuevos_codificados = codificador_target_cargado.transform(df_nueva_reglas[target_cargado])
        metricas_base_nueva = metricas_clasificacion(reales_nuevos_codificados, codigos_prediccion_nueva,
            probabilidades_nuevas, np.arange(len(clases_cargadas)), indice_positivo_cargado, salida_base_nueva["puntuaciones"])
    else:
        if not pd.api.types.is_numeric_dtype(df_nueva_reglas[target_cargado]) or not np.isfinite(df_nueva_reglas[target_cargado].to_numpy(dtype=float, na_value=np.nan)).all():
            raise ValueError("El target real nuevo debe ser numérico y finito.")
        metricas_base_nueva = metricas_regresion(df_nueva_reglas[target_cargado], resultado_base_nueva["ml_prediccion"])
    display(pd.Series(metricas_base_nueva, name="Base nueva").to_frame())

ruta_predicciones_exportadas = directorio_salida / "predicciones_base_nueva.csv"
resultado_base_nueva.to_csv(ruta_predicciones_exportadas, index=False, encoding="utf-8-sig")
print("Predicciones exportadas:", ruta_predicciones_exportadas)
# En Colab, opcionalmente:
# from google.colab import files
# files.download(str(ruta_predicciones_exportadas))
# files.download(str(ruta_modelo_guardado))

### 10.5 Intervalo solicitado y resumen mensual
`FECHA_FIN_PRONOSTICO` amplía el horizonte sin archivo futuro y debe coincidir con la frecuencia. Con archivo, sus fechas definen el horizonte completo. `FECHA_INICIO_REPORTE` y `FECHA_FIN_REPORTE` seleccionan un intervalo dentro de ese horizonte; se pronostican **todos** los periodos intermedios antes de filtrar, aunque el reporte empiece después del origen.

El CSV original conserva todo el horizonte; `pronostico_intervalo.csv` contiene el intervalo solicitado. Con `AGREGACION_MENSUAL="sum"` o `"mean"`, `pronostico_mensual.csv` resume ese intervalo. Un mes parcial conserva únicamente su importe/promedio observado, sin extrapolarlo. La comparación interanual solo aparece si ambos meses están completos y el histórico comparable es distinto de cero.

La tabla histórica comparable se construye con la historia guardada junto al modelo. Las predicciones se generan con el modelo operativo, reentrenado con toda la historia si elegiste esa opción después de medir prueba. No son intervalos de confianza; los errores recursivos pueden acumularse y una suma mensual no los elimina.

In [ ]:
if tipo_cargado == "serie_tiempo":
    inicio_reporte = pd.Timestamp(FECHA_INICIO_REPORTE) if FECHA_INICIO_REPORTE is not None else fechas_futuras_esperadas[0]
    fin_reporte = pd.Timestamp(FECHA_FIN_REPORTE) if FECHA_FIN_REPORTE is not None else fechas_futuras_esperadas[-1]
    if pd.isna(inicio_reporte) or pd.isna(fin_reporte) or inicio_reporte > fin_reporte:
        raise ValueError("El intervalo de reporte debe tener fechas válidas y ordenadas.")
    if inicio_reporte < fechas_futuras_esperadas[0] or fin_reporte > fechas_futuras_esperadas[-1]:
        raise ValueError("El intervalo solicitado debe estar dentro del horizonte pronosticado.")
    fechas_resultado = pd.DatetimeIndex(df_nueva_fechas[configuracion_cargada["fecha"]])
    resultado_intervalo = resultado_base_nueva.loc[(fechas_resultado >= inicio_reporte) & (fechas_resultado <= fin_reporte)].copy()
    if resultado_intervalo.empty:
        raise ValueError("El intervalo solicitado no contiene periodos de la frecuencia guardada.")
    resultado_intervalo.to_csv(directorio_salida / "pronostico_intervalo.csv", index=False, encoding="utf-8-sig")
    display(resultado_intervalo)
    agregacion_reporte = configuracion_cargada.get("agregacion_mensual")
    if agregacion_reporte is not None:
        fechas_intervalo = fechas_resultado[(fechas_resultado >= inicio_reporte) & (fechas_resultado <= fin_reporte)]
        serie_intervalo = pd.Series(resultado_intervalo["ml_prediccion"].to_numpy(), index=fechas_intervalo)
        resumen_intervalo_mensual = resumir_mensual(serie_intervalo, configuracion_cargada["frecuencia"], agregacion_reporte)
        resumen_historia_mensual = resumir_mensual(paquete_modelo_cargado["historia"], configuracion_cargada["frecuencia"], agregacion_reporte)
        valores_comparables = [resumen_historia_mensual.loc[mes - 12, "valor"]
            if completo and mes - 12 in resumen_historia_mensual.index and resumen_historia_mensual.loc[mes - 12, "mes_completo"]
            else np.nan for mes, completo in resumen_intervalo_mensual["mes_completo"].items()]
        resumen_con_historico = resumen_intervalo_mensual.assign(real_anio_anterior=valores_comparables)
        resumen_mensual_pronostico = resumen_con_historico.assign(variacion_interanual_pct=(
            resumen_con_historico["valor"].div(resumen_con_historico["real_anio_anterior"].replace(0, np.nan)) - 1).mul(100))
        resumen_mensual_pronostico.to_csv(directorio_salida / "pronostico_mensual.csv", encoding="utf-8-sig")
        display(resumen_mensual_pronostico)
        resumen_mensual_pronostico["valor"].plot.bar(title=f"Pronóstico mensual ({agregacion_reporte}); revisar meses parciales", figsize=(12, 4))
        plt.show()

## Anexo opcional · PyCaret en un entorno separado
**Objetivo:** reproducir el anexo de comparación de la referencia sin reiniciar el recorrido principal. **Entrada:** train y validación, nunca prueba. **Operación:** ejecutar un intérprete compatible que ya hayas preparado. **Resultado:** ranking y predicciones de validación exportados al directorio de salida. **Interpretación:** es una comparación adicional; no cambia FUENTE_MODELO_FINAL.

PyCaret 3.3.2 requiere una combinación distinta de dependencias y Python 3.8–3.11. No lo instales sobre las librerías del recorrido principal. Esta celda no instala conda ni reinicia Colab. Si no dispones de otro intérprete compatible, deja EJECUTAR_PYCARET=False y usa las comparaciones anteriores.

| Parámetro | Uso |
|---|---|
| EJECUTAR_PYCARET | False omite el anexo. True comprueba un intérprete externo y ejecuta el código separado. Solo tabular. |
| PYTHON_PYCARET | /content/pycaret311/bin/python es un ejemplo de entorno Python 3.11 previamente creado, con pycaret==3.3.2 instalado. Debes proporcionar la ruta real. |
| FOLDS_PYCARET | 3 folds dentro de train; más exige tiempo y ejemplos suficientes por clase. |
| MODELOS_PYCARET | ["lr","dt","rf","lightgbm"] restringe familias comparadas. Son identificadores propios de PyCaret, no nombres del ranking Lazy. |

El anexo declara predictores y desactiva eliminación de atípicos, selección automática de variables, eliminación por correlación y balanceo automático. Emplea las estrategias de imputación elegidas si son compatibles; constant numérica/categórica se comunica como valor constante. PyCaret muestra su propio ganador de CV, que se evalúa con validación, no prueba. El modelo no sustituye el seleccionado en la guía.

El código auxiliar se escribe únicamente cuando ejecutas este anexo en /content, como salida de Colab; no forma parte de los archivos del proyecto.

In [ ]:
EJECUTAR_PYCARET = False
PYTHON_PYCARET = "/content/pycaret311/bin/python"
FOLDS_PYCARET = 3
MODELOS_PYCARET = ["lr","dt","rf","lightgbm"]

if EJECUTAR_PYCARET:
    if TIPO_PROBLEMA not in {"clasificacion","regresion"}:
        raise ValueError("El anexo PyCaret es tabular.")
    if not Path(PYTHON_PYCARET).is_file():
        raise FileNotFoundError("Prepara un entorno separado Python 3.11 con pycaret==3.3.2 y corrige PYTHON_PYCARET. No se reiniciará Colab.")
    subprocess.check_call([PYTHON_PYCARET,"-c","import sys,pycaret; assert (3,8)<=sys.version_info[:2]<=(3,11), 'PyCaret requiere Python 3.8–3.11'"])
    if FOLDS_PYCARET < 2 or (TIPO_PROBLEMA == "clasificacion" and y_train.value_counts().min() < FOLDS_PYCARET):
        raise ValueError("Revisa FOLDS_PYCARET y ejemplos por clase.")
    if ESTRATEGIA_NUMERICA not in {"mean","median","constant"} or ESTRATEGIA_CATEGORICA not in {"most_frequent","constant"}:
        raise ValueError("El anexo admite imputación numérica mean/median/constant y categórica most_frequent/constant.")
    ruta_train_pycaret = directorio_salida / "pycaret_train.csv"
    ruta_val_pycaret = directorio_salida / "pycaret_validacion.csv"
    X_train.assign(**{TARGET:y_train.to_numpy()}).to_csv(ruta_train_pycaret,index=False)
    X_validacion.assign(**{TARGET:y_validacion.to_numpy()}).to_csv(ruta_val_pycaret,index=False)
    parametros_pycaret = {
        "train":str(ruta_train_pycaret),"validacion":str(ruta_val_pycaret),
        "salida":str(directorio_salida),"target":TARGET,"problema":TIPO_PROBLEMA,
        "numericas":COLUMNAS_NUMERICAS,"categoricas":COLUMNAS_CATEGORICAS,
        "semilla":SEMILLA,"folds":FOLDS_PYCARET,"modelos":MODELOS_PYCARET,"normalizar":ESCALAR,
        "imputacion_num":VALOR_NUMERICO_CONSTANTE if ESTRATEGIA_NUMERICA=="constant" else ESTRATEGIA_NUMERICA,
        "imputacion_cat":VALOR_CATEGORICO_CONSTANTE if ESTRATEGIA_CATEGORICA=="constant" else "mode"}
    ruta_parametros_pycaret = directorio_salida / "pycaret_parametros.json"
    ruta_parametros_pycaret.write_text(json.dumps(parametros_pycaret,ensure_ascii=False),encoding="utf-8")
    codigo_pycaret = """
import sys,json,importlib
from pathlib import Path
import pandas as pd
config=json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
modulo=importlib.import_module("pycaret.classification" if config["problema"]=="clasificacion" else "pycaret.regression")
datos_train=pd.read_csv(config["train"])
datos_validacion=pd.read_csv(config["validacion"])
opciones_balanceo={"fix_imbalance":False} if config["problema"]=="clasificacion" else {}
sesion=modulo.setup(data=datos_train,target=config["target"],test_data=datos_validacion,
    session_id=config["semilla"],fold=config["folds"],numeric_features=config["numericas"],
    categorical_features=config["categoricas"],normalize=config["normalizar"],
    numeric_imputation=config["imputacion_num"],categorical_imputation=config["imputacion_cat"],
    remove_outliers=False,remove_multicollinearity=False,feature_selection=False,**opciones_balanceo,
    html=False,verbose=False)
ganador_cv=modulo.compare_models(include=config["modelos"],sort="F1" if config["problema"]=="clasificacion" else "MAE",verbose=False)
ranking=modulo.pull()
ranking.to_csv(Path(config["salida"])/"pycaret_ranking.csv")
predicciones_validacion=modulo.predict_model(ganador_cv,data=datos_validacion,verbose=False)
predicciones_validacion.to_csv(Path(config["salida"])/"pycaret_predicciones_validacion.csv",index=False)
print(ranking.to_string())
"""
    ruta_script_pycaret = directorio_salida / "pycaret_anexo.py"
    ruta_script_pycaret.write_text(codigo_pycaret,encoding="utf-8")
    subprocess.check_call([PYTHON_PYCARET,str(ruta_script_pycaret),str(ruta_parametros_pycaret)])

## Cierre · Registra tus conclusiones
Para responder una prueba, escribe: unidad de observación, target y disponibilidad de predictores; qué limpiezas ejecutaste y por qué; qué mostró el EDA; qué modelo/hiperparámetros elegiste; métrica de validación y prueba; errores importantes y resultado en la base nueva.

No existe un algoritmo ganador universal. Un modelo que supera una línea base en esta base puede cambiar de desempeño con otras fechas, poblaciones, nulos o categorías. Revisa esos cambios con nuevos datos. Una importancia o un SHAP no demuestra que modificar una variable cause el resultado.

**Archivos de Colab:** el CSV de predicciones, configuración, joblib, HTML de ProfileReport y .keras si elegiste red son salidas de ejecución. Descárgalos antes de cerrar la sesión. El único archivo de código que necesitas cargar para esta guía es este notebook.

### Referencias
S3/S4: orden de comprensión → calidad/limpieza → EDA → preparación; clasificación y explicación.
S5: compra, métricas, umbral y deciles.
Caso RapiMenu: regresión, línea base y errores por segmento.
XGBoost/LightGBM: boosting y predicciones nuevas.
Caso de series: memoria histórica, pronóstico recursivo y backtesting.
AutoML_muchos_modelos_vf_.ipynb: LazyPredict, búsqueda FLAML, redes tabulares y anexo PyCaret.

Documentación técnica:
[FLAML](https://microsoft.github.io/FLAML/docs/Use-Cases/Task-Oriented-AutoML/),
[LazyPredict](https://lazypredict.readthedocs.io/en/latest/usage.html),
[ProfileReport](https://docs.profiling.ydata.ai/latest/getting-started/quickstart/),
[Keras](https://keras.io/api/models/model_saving_apis/).

### Verificación de esta entrega
El notebook se entrega sin datos ni resultados precalculados. La celda de texto siguiente registra las comprobaciones efectivamente realizadas; no afirma una ejecución dentro de una sesión real de Colab si solo pudo verificarse localmente.

**Práctica dirigida de regresión y series:** PD_Solucion_Regresion_Series.ipynb aporta calendario, gráficos temporales y reportes diarios/mensuales. Se adapta sin copiar resultados numéricos ni fechas del caso. Indica frecuencia, origen, horizonte, línea base, cobertura mensual y limitaciones de tu pronóstico.

### Comprobaciones de esta actualización
Se ejecutó `tests/verificar_series_pc2.py` con Python 3.12, NumPy y pandas disponibles en el entorno local. Los casos sintéticos comprobaron:

- Feriados históricos y futuros, semana ISO al cambiar de año y fin de semana.
- Las mismas columnas y cálculos de predictores en entrenamiento y recursión; el target actual no entra en la media móvil.
- Conservación de series de entrada, persistencia, ciclo estacional y rechazo de fechas futuras no consecutivas.
- Particiones cronológicas por fracciones y por cantidades explícitas; rechazo de huecos, fechas repetidas/desordenadas, horizontes inválidos e historia insuficiente.
- Generación de fechas diarias, semanales y mensuales; horizonte hasta una fecha y filtro del intervalo después de calcular los pasos intermedios.
- Agregación mensual de cantidades y niveles, año bisiesto, cobertura según frecuencia y meses parciales.
- Comparación interanual solo con meses completos y protección frente a denominadores históricos iguales a cero.
- Exportación temporal de detalle por intervalo y resumen mensual.

También se comprobaron sintaxis de las celdas Python, IDs únicos, funciones requeridas con docstrings, ausencia de asignaciones autorreferentes y de inplace=True. El notebook se entrega sin resultados precalculados.

**Límites:** los gráficos se omitieron y el cargador del paquete se simuló en la prueba ligera. No se ejecutó entrenamiento de Random Forest/XGBoost, guardado/recarga real con joblib, clasificación ni regresión en esta actualización: las dependencias completas no estaban disponibles en el entorno restringido y las solicitudes para ejecutar con los paquetes existentes o descargarlos fueron rechazadas. Tampoco se ejecutó Google Colab ni la validación de esquema con nbformat.

`tests/verificar_guia_pc2.py` queda actualizado con los casos de integración de los tres recorridos, incluido guardado/recarga real, ausencia de reajuste durante inferencia y las nuevas opciones temporales. Ejecuta ese verificador en un entorno con las dependencias de instalación y nbformat. Sus pruebas usan archivos sintéticos temporales y no modifican el notebook.
